# Unsupervised Learning for HR+ Breast Cancer RNA Sequencing
This notebook performs a comprehensive analysis of single-cell RNA sequencing data to predict immunotherapy response in high-risk HR+/HER2- breast cancer patients. It includes data loading, quality control, TCR sequence integration, unsupervised clustering, and multi-modal machine learning classification.


In [1]:
# Memory monitoring utility
import psutil
import os

def print_memory_usage():
    """Print current memory usage"""
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    mem_mb = mem_info.rss / 1024 / 1024
    print(f"Current memory usage: {mem_mb:.2f} MB")
    
    # System memory info
    vm = psutil.virtual_memory()
    print(f"System memory: {vm.percent}% used ({vm.used / 1024**3:.2f} GB / {vm.total / 1024**3:.2f} GB)")
    return mem_mb

# Check initial memory
print("=== Initial Memory Check ===")
initial_mem = print_memory_usage()
print("\nTip: Run this cell periodically to monitor memory usage")

=== Initial Memory Check ===
Current memory usage: 101.90 MB
System memory: 3.7% used (0.72 GB / 31.35 GB)

Tip: Run this cell periodically to monitor memory usage


In [ ]:
# Set these flags to True to SKIP the corresponding sections

SKIP_UNSUPERVISED_LEARNING = False
SKIP_XGBOOST_LOPO_CV = False
SKIP_TRADITIONAL_ML = False
SKIP_TO_DEEP_LEARNING = False

if SKIP_TO_DEEP_LEARNING:
    SKIP_UNSUPERVISED_LEARNING = True
    SKIP_XGBOOST_LOPO_CV = True
    SKIP_TRADITIONAL_ML = True
    print("FAST MODE: Skipping unsupervised learning and traditional ML, going straight to deep learning!")
else:
    print("FULL MODE: Running all analysis sections")

FULL MODE: Running all analysis sections


In [3]:
%pip install anndata scanpy scikit-learn umap-learn --quiet
%pip install biopython --quiet
%pip install scikit-learn --quiet
%pip install umap-learn --quiet
%pip install hdbscan --quiet
%pip install plotly --quiet
%pip install xgboost --quiet
%pip install tensorflow --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 100.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Ensure non-interactive Matplotlib backend to avoid font import issues
import matplotlib
try:
    matplotlib.use('Agg')
except Exception as e:
    print("Could not set Agg backend:", e)

# Set memory optimization flags
import os
os.environ['PYTHONHASHSEED'] = '0'
os.environ['OMP_NUM_THREADS'] = '4'  # Limit parallel threads to save memory

# --- Idempotent monkeypatch CountVectorizer.fit_transform to handle empty vocabulary errors ---
try:
    from sklearn.feature_extraction.text import CountVectorizer
    import scipy.sparse as _sps

    # Only patch once; store original on the class to avoid double-wrapping
    if not hasattr(CountVectorizer, '_orig_fit_transform'):
        CountVectorizer._orig_fit_transform = CountVectorizer.fit_transform

        def _safe_cv_fit(self, raw_docs, *args, **kwargs):
            try:
                return CountVectorizer._orig_fit_transform(self, raw_docs, *args, **kwargs)
            except ValueError as e:
                # Handle sklearn's "empty vocabulary" error by returning an all-zero matrix
                if 'empty vocabulary' in str(e).lower():
                    n = len(raw_docs) if raw_docs is not None else 0
                    return _sps.csr_matrix((n, 1))
                raise

        CountVectorizer.fit_transform = _safe_cv_fit
except Exception as e:
    # If sklearn/scipy are not available at import time, skip patching and log reason
    print("CountVectorizer monkeypatch skipped:", e)

# --- Idempotent monkeypatch TruncatedSVD.fit_transform for 1-feature inputs ---
try:
    from sklearn.decomposition import TruncatedSVD
    import numpy as _np

    if not hasattr(TruncatedSVD, '_orig_fit_transform_safe'):
        TruncatedSVD._orig_fit_transform_safe = TruncatedSVD.fit_transform

        def _safe_svd_fit_transform(self, X, *args, **kwargs):
            n_features = 0
            try:
                n_features = int(X.shape[1])
            except Exception:
                n_features = 0

            if n_features < 2:
                # TruncatedSVD requires at least 2 features; bypass and return dense fallback
                dense = X.toarray() if hasattr(X, 'toarray') else _np.asarray(X)
                if dense.ndim == 1:
                    dense = dense.reshape(-1, 1)
                return dense.astype(_np.float32, copy=False)

            return TruncatedSVD._orig_fit_transform_safe(self, X, *args, **kwargs)

        TruncatedSVD.fit_transform = _safe_svd_fit_transform
except Exception as e:
    print("TruncatedSVD monkeypatch skipped:", e)

In [5]:
# --- Initial Setup & Imports ---
import sys
import subprocess

# Install critical dependencies if missing
try:
    import Bio
except ImportError:
    print("Installing biopython...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "biopython"])

import pandas as pd
import requests
import os
import tarfile
import glob
from io import BytesIO
from collections import Counter
import warnings

# BioPython Imports
try:
    from Bio.Seq import Seq
    from Bio.SeqUtils import ProtParam
except ImportError:
    # If install just happened, might need re-import logic or kernel restart, 
    # but usually works in same session after import
    pass

# Suppress warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# --- Environment Detection ---
IS_KAGGLE = os.path.exists('/kaggle/input') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
print(f"Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    # Ensure standard directories exist
    os.makedirs('/kaggle/working/Data', exist_ok=True)
    os.makedirs('/kaggle/working/Output', exist_ok=True)


Running on Kaggle: True


## Data Loading and Preparation
We analyze a single-cell dataset recently published by Sun et al. (2025) (GEO accession GSE300475). The data originates from the DFCI 16-466 clinical trial (NCT02999477), a randomized phase II study evaluating neoadjuvant nab-paclitaxel in combination with pembrolizumab for high-risk, early-stage HR+/HER2- breast cancer. The specific cohort analyzed consists of longitudinal peripheral blood mononuclear cell (PBMC) samples from patients in the chemotherapy-first arm.

Patients were classified into binary response categories based on Residual Cancer Burden (RCB) index assessed at surgery:
*   **Responders:** Patients achieving Pathologic Complete Response (pCR, RCB-0) or minimal residual disease (RCB-I).
*   **Non-Responders:** Patients with moderate (RCB-II) or extensive (RCB-III) residual disease.

The following code handles the downloading and extraction of the raw data files.

In [6]:
files_to_fetch = [
    {
        "name": "GSE300475_RAW.tar",
        "size": "565.5 Mb",
        "download_url": "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE300475&format=file",
        "type": "TAR (of CSV, MTX, TSV)"
    },
    {
        "name": "GSE300475_feature_ref.xlsx",
        "size": "5.4 Kb",
        "download_url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE300nnn/GSE300475/suppl/GSE300475%5Ffeature%5Fref.xlsx",
        "type": "XLSX"
    }
]

In [7]:
# Set download directory based on environment
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "README.md").exists() or (candidate / "Code").exists():
            return candidate
    return cwd

if IS_KAGGLE:
    # On Kaggle, use /kaggle/working which is writable
    download_dir = Path("/kaggle/working/Data")
else:
    # Local (VS Code / Windows): use project-root/Data
    project_root = _find_project_root()
    download_dir = project_root / "Data"

download_dir = Path(download_dir)
download_dir.mkdir(parents=True, exist_ok=True)
print(f"Downloads will be saved in: {download_dir.resolve()}\n")

def download_file(url, filename, destination_folder):
    """
    Downloads a file from a given URL to a specified destination folder.
    """
    filepath = os.path.join(destination_folder, filename)
    print(f"Attempting to download {filename} from {url}...")
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()

        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        print(f"Successfully downloaded {filename} to {filepath}")
        return filepath
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {filename}: {e}")
        return None

Downloads will be saved in: /kaggle/working/Data



In [8]:
for file_info in files_to_fetch:
    filename = file_info["name"]
    url = file_info["download_url"]
    file_type = file_info["type"]

    downloaded_filepath = download_file(url, filename, download_dir)

    # If the file is a TAR archive, extract it and list the contents
    if downloaded_filepath and filename.endswith(".tar"):
        print(f"Extracting {filename}...\n")
        try:
            with tarfile.open(downloaded_filepath, "r") as tar:
                # List contents
                members = tar.getnames()
                print(f"Files contained in {filename}:")
                for member in members:
                    print(f" - {member}")

                # Extract to a subdirectory within download_dir
                extract_path = os.path.join(download_dir, filename.replace(".tar", ""))
                os.makedirs(extract_path, exist_ok=True)
                tar.extractall(path=extract_path, filter='data')
                print(f"\nExtracted to: {extract_path}")
        except tarfile.TarError as e:
            print(f"Error extracting {filename}: {e}")

        print("-" * 50 + "\n")

Attempting to download GSE300475_RAW.tar from https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE300475&format=file...
Successfully downloaded GSE300475_RAW.tar to /kaggle/working/Data/GSE300475_RAW.tar
Extracting GSE300475_RAW.tar...

Files contained in GSE300475_RAW.tar:
 - GSM9061665_S1_barcodes.tsv.gz
 - GSM9061665_S1_features.tsv.gz
 - GSM9061665_S1_matrix.mtx.gz
 - GSM9061666_S2_barcodes.tsv.gz
 - GSM9061666_S2_features.tsv.gz
 - GSM9061666_S2_matrix.mtx.gz
 - GSM9061667_S3_barcodes.tsv.gz
 - GSM9061667_S3_features.tsv.gz
 - GSM9061667_S3_matrix.mtx.gz
 - GSM9061668_S4_barcodes.tsv.gz
 - GSM9061668_S4_features.tsv.gz
 - GSM9061668_S4_matrix.mtx.gz
 - GSM9061669_S5_barcodes.tsv.gz
 - GSM9061669_S5_features.tsv.gz
 - GSM9061669_S5_matrix.mtx.gz
 - GSM9061670_S6_barcodes.tsv.gz
 - GSM9061670_S6_features.tsv.gz
 - GSM9061670_S6_matrix.mtx.gz
 - GSM9061671_S7_barcodes.tsv.gz
 - GSM9061671_S7_features.tsv.gz
 - GSM9061671_S7_matrix.mtx.gz
 - GSM9061672_S8_barcodes.tsv.gz
 - GSM9061672_S

In [9]:
import gzip
import shutil
from pathlib import Path
import pandas as pd
import os

# NOTE: We SKIP explicit decompression to avoid consuming disk space/memory.
# Scanpy's read_10x_mtx and other tools can read .gz files directly.

def preview_file(file_path):
    """
    Display the first few lines of a file (supports .gz automatically)
    """
    if file_path is None: return
    
    print(f"\n--- Preview of {os.path.basename(file_path)} ---")
    try:
        # Handle gzip if extension matches
        opener = gzip.open if str(file_path).endswith('.gz') else open
        
        if str(file_path).endswith(".tsv") or str(file_path).endswith(".csv") or str(file_path).endswith(".tsv.gz") or str(file_path).endswith(".csv.gz"):
            # Use pandas with nrows 
            sep = '\t' if 'tsv' in str(file_path) else ','
            comp = 'gzip' if str(file_path).endswith('.gz') else None
            try:
                # Try reading with header inference
                df = pd.read_csv(file_path, sep=sep, nrows=5, compression=comp)
                print(df)
            except:
                print("Could not read as CSV/TSV")
        elif 'matrix.mtx' in str(file_path):
            # Read as text stream
            with opener(file_path, 'rt') as f: # 'rt' for text mode
                print("First 10 lines (header and data):")
                for _ in range(10):
                    line = f.readline()
                    if not line: break
                    print(line.strip())
        else:
            print(f"File type {file_path} preview not customized.")
    except Exception as e:
        print(f"Could not preview {file_path}: {e}")

# Define extract_dir based on download_dir from previous cell
extract_dir = os.path.join(download_dir, "GSE300475_RAW")
raw_data_dir = Path(extract_dir) # Explicitly define this for downstream cells
print(f"Raw data directory set to: {raw_data_dir}")

gz_files = []
for root, _, files in os.walk(extract_dir):
    for file in files:
        if file.endswith(".gz"):
            gz_files.append((os.path.join(root, file), root))

print(f"Found {len(gz_files)} .gz files. Ready for processing (Decompression skipped).")

# Just preview a few to ensure they are readable
for path, _ in gz_files[:3]:
    preview_file(path)

Raw data directory set to: /kaggle/working/Data/GSE300475_RAW
Found 43 .gz files. Ready for processing (Decompression skipped).

--- Preview of GSM9061665_S1_features.tsv.gz ---
   ENSG00000243485 MIR1302-2HG  Gene Expression
0  ENSG00000237613     FAM138A  Gene Expression
1  ENSG00000186092       OR4F5  Gene Expression
2  ENSG00000238009  AL627309.1  Gene Expression
3  ENSG00000239945  AL627309.3  Gene Expression
4  ENSG00000239906  AL627309.2  Gene Expression

--- Preview of GSM9061665_S1_barcodes.tsv.gz ---
   AAACCTGAGAAGGGTA-1
0  AAACCTGAGACTGTAA-1
1  AAACCTGAGCAGCGTA-1
2  AAACCTGAGCCAACAG-1
3  AAACCTGAGCGTGAAC-1
4  AAACCTGAGCTACCTA-1

--- Preview of GSM9061690_S4_all_contig_annotations.csv.gz ---
              barcode  is_cell                    contig_id  high_confidence  \
0  AAACCTGAGGCTATCT-1     True  AAACCTGAGGCTATCT-1_contig_1             True   
1  AAACCTGAGGGTCTCC-1     True  AAACCTGAGGGTCTCC-1_contig_1             True   
2  AAACCTGAGGGTCTCC-1     True  AAACCTGAGGGTCTCC

In [10]:
%pip install scanpy --quiet

Note: you may need to restart the kernel to use updated packages.


In [11]:
import glob

# Find all "all_contig_annotations.csv" files in the extracted directory and sum their lengths (number of rows)

all_contig_files = glob.glob(os.path.join(extract_dir, "*_all_contig_annotations.csv"))
total_rows = 0

for file in all_contig_files:
    try:
        df = pd.read_csv(file)
        num_rows = len(df)
        print(f"{os.path.basename(file)}: {num_rows} rows")
        total_rows += num_rows
    except Exception as e:
        print(f"Could not read {file}: {e}")

print(f"\nTotal rows in all contig annotation files: {total_rows}")


Total rows in all contig annotation files: 0


## 3. Metadata Construction and Sample Mapping

We manually define the metadata mapping for the 11 samples included in this analysis. This ensures accurate association of sample IDs with patient identifiers, timepoints (Baseline, Post-Treatment, Recurrence), and clinical response status (Responder vs. Non-Responder), as detailed in the study's supplementary materials.


In [12]:
%pip install scanpy pandas numpy --quiet
# Import required libraries for single-cell RNA-seq analysis and data handling
import scanpy as sc  # Main library for single-cell analysis, provides AnnData structure and many tools
import pandas as pd  # For tabular data manipulation and metadata handling
import numpy as np   # For numerical operations and array handling
import os            # For operating system interactions (file paths, etc.)
from pathlib import Path  # For robust and readable file path management

# Print versions to ensure reproducibility and compatibility
print(f"Scanpy version: {sc.__version__}")
print(f"Pandas version: {pd.__version__}")

from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

Note: you may need to restart the kernel to use updated packages.
Scanpy version: 1.12
Pandas version: 2.3.3
All libraries imported successfully!


In [13]:
import gzip
import shutil
from pathlib import Path
import pandas as pd
import os
from joblib import Parallel, delayed

def preview_file(file_path):
    """
    Display the first few lines of a decompressed file without loading the whole file into memory.
    """
    if file_path is None: return
    
    print(f"\n--- Preview of {os.path.basename(file_path)} ---")
    try:
        if file_path.endswith(".tsv") or file_path.endswith(".csv"):
            # Use pandas with nrows to avoid loading full file
            sep = '\t' if file_path.endswith(".tsv") else ','
            df = pd.read_csv(file_path, sep=sep, nrows=5) 
            print(df)
        elif file_path.endswith(".mtx"):
            # Read as text stream to avoid loading massive matrix into memory
            with open(file_path, 'r') as f:
                print("First 10 lines (header and data):")
                for _ in range(10):
                    line = f.readline()
                    if not line: break
                    print(line.strip())
        elif str(file_path).endswith(".gz"):
             print(f"File is compressed ({file_path}). Scanpy will handle decompression automatically.")
        else:
            print("Unsupported file type for preview.")
    except Exception as e:
        print(f"Could not preview {file_path}: {e}")

# Ensure download_dir exists (fallback protection)
if 'download_dir' not in globals():
     # Fallback logic if variable not in scope
     if 'IS_KAGGLE' in globals() and IS_KAGGLE:
         download_dir = "/kaggle/working/Data"
     else:
         def _find_project_root():
             cwd = Path.cwd().resolve()
             for candidate in [cwd, *cwd.parents]:
                 if (candidate / "README.md").exists() or (candidate / "Code").exists():
                     return candidate
             return cwd
         download_dir = str(_find_project_root() / "Data")

# Normalize download_dir to string for os.path usage
if isinstance(download_dir, Path):
    download_dir = str(download_dir)

extract_dir = os.path.join(download_dir, "GSE300475_RAW")

# --- PATH CORRECTION LOGIC ---
# If extract_dir is empty or missing, but files are in download_dir, use download_dir
if not os.path.exists(extract_dir) or not any(f.endswith('.gz') for f in os.listdir(extract_dir) if os.path.isfile(os.path.join(extract_dir, f))):
    if os.path.exists(download_dir) and any(f.endswith('.gz') for f in os.listdir(download_dir) if os.path.isfile(os.path.join(download_dir, f))):
         print(f"Detecting files in {download_dir} directly. Adjusting path.")
         extract_dir = download_dir

# --- MEMORY OPTIMIZATION ---
# We SKIP explicit decompression here because Scanpy's read_10x_mtx can read .gz files directly.
# Decompressing large sparse matrices to dense text files on disk is unnecessary and wastes storage/IO.
print("Skipping explicit decompression to save disk space and IO.")
print("Scanpy handles .gz files directly during loading.")

# Just preview one GZ file to show it exists
gz_files = []
for root, _, files in os.walk(extract_dir):
    for file in files:
        if file.endswith(".gz"):
            gz_files.append(os.path.join(root, file))

if gz_files:
    print(f"Found {len(gz_files)} compressed files ready for loading.")
    print(f"Example: {gz_files[0]}")

Skipping explicit decompression to save disk space and IO.
Scanpy handles .gz files directly during loading.
Found 43 compressed files ready for loading.
Example: /kaggle/working/Data/GSE300475_RAW/GSM9061665_S1_features.tsv.gz


In [14]:
%%time
# --- Setup data paths ---
import os
from pathlib import Path
import tarfile
import requests

# Use existing IS_KAGGLE flag or detect
if 'IS_KAGGLE' not in globals():
    IS_KAGGLE = os.path.exists('/kaggle/input') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None

def _find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "README.md").exists() or (candidate / "Code").exists():
            return candidate
    return cwd

def _has_matrix_files(path: Path) -> bool:
    return path.exists() and any(path.rglob("*matrix.mtx*"))

# Determine candidate base directories
candidate_dirs = []
if IS_KAGGLE:
    candidate_dirs = [Path('/kaggle/working/Data'), Path('/Data'), Path('/kaggle/input')]
else:
    project_root = _find_project_root()
    if 'download_dir' in globals() and download_dir:
        candidate_dirs.append(Path(download_dir))
    candidate_dirs += [project_root / 'Data', project_root / 'data', project_root]

raw_data_dir = None
for base in candidate_dirs:
    if base is None:
        continue
    base = Path(base)
    if base.name == 'GSE300475_RAW' and _has_matrix_files(base):
        raw_data_dir = base
        break
    if _has_matrix_files(base / 'GSE300475_RAW'):
        raw_data_dir = base / 'GSE300475_RAW'
        break
    if _has_matrix_files(base):
        raw_data_dir = base
        break

# Fallback: search under project root (local only)
if raw_data_dir is None and not IS_KAGGLE:
    project_root = _find_project_root()
    for match in project_root.rglob('GSE300475_RAW'):
        if _has_matrix_files(match):
            raw_data_dir = match
            break

# Auto-download if still missing
if raw_data_dir is None:
    print("Raw data not found locally. Attempting download...")
    if IS_KAGGLE:
        download_dir = Path('/kaggle/working/Data')
    else:
        project_root = _find_project_root()
        if 'download_dir' in globals() and download_dir:
            download_dir = Path(download_dir)
        else:
            download_dir = project_root / 'Data'
    download_dir.mkdir(parents=True, exist_ok=True)
    tar_path = download_dir / 'GSE300475_RAW.tar'
    if not tar_path.exists():
        url = 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE300475&format=file'
        try:
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                with open(tar_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            print(f"Downloaded {tar_path}")
        except Exception as e:
            raise RuntimeError(f"Download failed: {e}")
    extract_path = download_dir / 'GSE300475_RAW'
    if not extract_path.exists():
        print(f"Extracting {tar_path} to {extract_path}...")
        try:
            with tarfile.open(tar_path, 'r') as tar:
                try:
                    tar.extractall(path=extract_path, filter='data')
                except TypeError:
                    tar.extractall(path=extract_path)
        except Exception as e:
            raise RuntimeError(f"Extraction failed: {e}")
    raw_data_dir = extract_path

print(f"Data directory set to: {raw_data_dir}")

# --- Manually create the metadata mapping ---
# This list contains information about each sample, including GEO IDs, patient IDs, timepoints, and treatment response.
# Note: S8 (GSM9061672) has GEX files but no corresponding TCR file.
metadata_list = [
    # Patient 1 (Responder)
    {'S_Number': 'S1',  'GEX_Sample_ID': 'GSM9061665', 'TCR_Sample_ID': 'GSM9061687', 'Patient_ID': 'PT1',  'Timepoint': 'Baseline',     'Response': 'Responder',     'In_Data': 'Yes'},
    {'S_Number': 'S2',  'GEX_Sample_ID': 'GSM9061666', 'TCR_Sample_ID': 'GSM9061688', 'Patient_ID': 'PT1',  'Timepoint': 'Post-Tx',      'Response': 'Responder',     'In_Data': 'Yes'},
    {'S_Number': 'S3',  'GEX_Sample_ID': 'GSM9061667', 'TCR_Sample_ID': 'GSM9061689', 'Patient_ID': 'PT1',  'Timepoint': 'Recurrence',   'Response': 'Responder',     'In_Data': 'Yes'},
    
    # Patient 2 (Responder)
    {'S_Number': 'S4',  'GEX_Sample_ID': 'GSM9061668', 'TCR_Sample_ID': 'GSM9061690', 'Patient_ID': 'PT2',  'Timepoint': 'Baseline',     'Response': 'Responder',     'In_Data': 'Yes'},
    {'S_Number': 'S5',  'GEX_Sample_ID': 'GSM9061669', 'TCR_Sample_ID': 'GSM9061691', 'Patient_ID': 'PT2',  'Timepoint': 'Post-Tx',      'Response': 'Responder',     'In_Data': 'Yes'},
    
    # Patient 3 (Non-Responder)
    {'S_Number': 'S6',  'GEX_Sample_ID': 'GSM9061670', 'TCR_Sample_ID': 'GSM9061692', 'Patient_ID': 'PT3',  'Timepoint': 'Baseline',     'Response': 'Non-Responder', 'In_Data': 'Yes'},
    {'S_Number': 'S7',  'GEX_Sample_ID': 'GSM9061671', 'TCR_Sample_ID': 'GSM9061693', 'Patient_ID': 'PT3',  'Timepoint': 'Post-Tx',      'Response': 'Non-Responder', 'In_Data': 'Yes'},
    {'S_Number': 'S8',  'GEX_Sample_ID': 'GSM9061672', 'TCR_Sample_ID': None,         'Patient_ID': 'PT3',  'Timepoint': 'Recurrence',   'Response': 'Non-Responder', 'In_Data': 'GEX only'},
    
    # Patient 4 (Non-Responder)
    {'S_Number': 'S9',  'GEX_Sample_ID': 'GSM9061673', 'TCR_Sample_ID': 'GSM9061694', 'Patient_ID': 'PT4',  'Timepoint': 'Baseline',     'Response': 'Non-Responder', 'In_Data': 'Yes'},
    {'S_Number': 'S10', 'GEX_Sample_ID': 'GSM9061674', 'TCR_Sample_ID': 'GSM9061695', 'Patient_ID': 'PT4',  'Timepoint': 'Post-Tx',      'Response': 'Non-Responder', 'In_Data': 'Yes'},
    {'S_Number': 'S11', 'GEX_Sample_ID': 'GSM9061675', 'TCR_Sample_ID': 'GSM9061696', 'Patient_ID': 'PT4',  'Timepoint': 'Recurrence',   'Response': 'Non-Responder', 'In_Data': 'Yes'},
]

# Create pandas DataFrame for easy access
metadata_df = pd.DataFrame(metadata_list)
print("Metadata table now matches the requested specification:")
display(metadata_df)

# --- Programmatic sanity-check for file presence ---
# This loop checks if the expected files exist for each sample and updates the 'In_Data' column accordingly.
for idx, row in metadata_df.iterrows():
    s = row['S_Number']
    g = row['GEX_Sample_ID']
    t = row['TCR_Sample_ID']
    # Check for gene expression matrix file (compressed or uncompressed)
    # Check .mtx, .mtx.gz, and also potential file name variations or if they are in subfolders
    # We look in raw_data_dir found above.
    g_exists = (raw_data_dir / f"{g}_{s}_matrix.mtx.gz").exists() or (raw_data_dir / f"{g}_{s}_matrix.mtx").exists()
    
    # Also check if just the GSM id is present in some filename if strict match fails (fallback)
    if not g_exists:
         # Try simpler wildcard search
         g_exists = len(list(raw_data_dir.glob(f"*{g}*matrix*"))) > 0

    t_exists = False
    # Check for TCR annotation file if TCR sample ID is present
    if pd.notna(t) and t is not None:
        t_exists = (raw_data_dir / f"{t}_{s}_all_contig_annotations.csv.gz").exists() or (raw_data_dir / f"{t}_{s}_all_contig_annotations.csv").exists()
        if not t_exists:
             t_exists = len(list(raw_data_dir.glob(f"*{t}*all_contig_annotations*"))) > 0
             
    # Update 'In_Data' column based on file presence
    if g_exists and t_exists:
        metadata_df.at[idx, 'In_Data'] = 'Yes'
    elif g_exists and not t_exists:
        metadata_df.at[idx, 'In_Data'] = 'GEX only'
    else:
        metadata_df.at[idx, 'In_Data'] = 'No'

print("\nPost-check In_Data column (based on files found in raw_data_dir):")
display(metadata_df)

Data directory set to: /kaggle/working/Data/GSE300475_RAW
Metadata table now matches the requested specification:


,S_Number,GEX_Sample_ID,TCR_Sample_ID,Patient_ID,Timepoint,Response,In_Data
0,S1,GSM9061665,GSM9061687,PT1,Baseline,Responder,Yes
1,S2,GSM9061666,GSM9061688,PT1,Post-Tx,Responder,Yes
2,S3,GSM9061667,GSM9061689,PT1,Recurrence,Responder,Yes
3,S4,GSM9061668,GSM9061690,PT2,Baseline,Responder,Yes
4,S5,GSM9061669,GSM9061691,PT2,Post-Tx,Responder,Yes
5,S6,GSM9061670,GSM9061692,PT3,Baseline,Non-Responder,Yes
6,S7,GSM9061671,GSM9061693,PT3,Post-Tx,Non-Responder,Yes
7,S8,GSM9061672,None,PT3,Recurrence,Non-Responder,GEX only
8,S9,GSM9061673,GSM9061694,PT4,Baseline,Non-Responder,Yes
9,S10,GSM9061674,GSM9061695,PT4,Post-Tx,Non-Responder,Yes



Post-check In_Data column (based on files found in raw_data_dir):


,S_Number,GEX_Sample_ID,TCR_Sample_ID,Patient_ID,Timepoint,Response,In_Data
0,S1,GSM9061665,GSM9061687,PT1,Baseline,Responder,Yes
1,S2,GSM9061666,GSM9061688,PT1,Post-Tx,Responder,Yes
2,S3,GSM9061667,GSM9061689,PT1,Recurrence,Responder,Yes
3,S4,GSM9061668,GSM9061690,PT2,Baseline,Responder,Yes
4,S5,GSM9061669,GSM9061691,PT2,Post-Tx,Responder,Yes
5,S6,GSM9061670,GSM9061692,PT3,Baseline,Non-Responder,Yes
6,S7,GSM9061671,GSM9061693,PT3,Post-Tx,Non-Responder,Yes
7,S8,GSM9061672,None,PT3,Recurrence,Non-Responder,GEX only
8,S9,GSM9061673,GSM9061694,PT4,Baseline,Non-Responder,Yes
9,S10,GSM9061674,GSM9061695,PT4,Post-Tx,Non-Responder,Yes


CPU times: user 27.1 ms, sys: 1.65 ms, total: 28.8 ms
Wall time: 28.3 ms


In [15]:
%%time
# --- DISK-BASED MAP-REDUCE STRATEGY TO SOLVE OOM ---
# Strategy: 
# 1. Map: Process each sample -> QC -> Save to temp .h5ad on disk
# 2. Reduce: Concatenate on disk (preferred) or in small batches
# This keeps RAM usage low during processing and avoids the iterative reallocation spike.

import gc
import shutil
import scanpy as sc
import anndata as ad
import pandas as pd
import scipy.sparse as sp
import numpy as np
import os
from pathlib import Path

gc.enable()

# Validate prerequisites
if 'metadata_df' not in globals():
    raise NameError("metadata_df is not defined. Please run the metadata creation cell first.")
if 'raw_data_dir' not in globals():
    raise NameError("raw_data_dir is not defined. Please run the data path setup cell first.")

# Setup temp directory for chunks
temp_chunk_dir = Path("temp_adata_chunks")
if temp_chunk_dir.exists():
    shutil.rmtree(temp_chunk_dir)
temp_chunk_dir.mkdir(exist_ok=True)

chunk_files = []
chunk_keys = []
tcr_data_list = []  # TCR data is small enough to keep in memory

print("Starting Map Phase (Processing & Saving Chunks)...")

# --- MAP PHASE: Process & Save ---
for index, row in metadata_df.iterrows():
    gex_sample_id = row['GEX_Sample_ID']
    s_number = row['S_Number']
    
    # Construct sample-level prefix
    sample_prefix = f"{gex_sample_id}_{s_number}"

    # Use robust file finding logic from previous cells
    matrix_file = None
    for ext in ['matrix.mtx.gz', 'matrix.mtx']:
        candidate = raw_data_dir / f"{sample_prefix}_{ext}"
        if candidate.exists():
            matrix_file = candidate
            break
            
    if not matrix_file:
         # Fallback search
        for ext in ['matrix.mtx.gz', 'matrix.mtx']:
            possible_files = list(raw_data_dir.glob(f"*{gex_sample_id}*{ext}"))
            if possible_files:
                matrix_file = possible_files[0]
                break
    
    if not matrix_file:
        print(f"Skipping {sample_prefix}: Matrix file not found.")
        continue

    sample_data_path = matrix_file.parent
    matrix_prefix = matrix_file.name.replace('matrix.mtx', '').replace('.gz', '')

    print(f"Processing {index+1}/{len(metadata_df)}: {sample_prefix}")
    
    try:
        # Load GEX
        adata_sample = sc.read_10x_mtx(
            sample_data_path, 
            var_names='gene_symbols',
            prefix=matrix_prefix,
            cache=True
        )
        
        # Ensure sparse float32 IMMEDIATELY
        if not hasattr(adata_sample.X, 'toarray'):
            adata_sample.X = sp.csr_matrix(adata_sample.X, dtype=np.float32)
        else:
            adata_sample.X = sp.csr_matrix(adata_sample.X, dtype=np.float32)
            
        # Add metadata
        adata_sample.obs['sample_id'] = gex_sample_id 
        adata_sample.obs['patient_id'] = row['Patient_ID']
        adata_sample.obs['timepoint'] = row['Timepoint']
        adata_sample.obs['response'] = row['Response']
        
        # QC Filtering (Crucial reduction)
        sc.pp.filter_cells(adata_sample, min_genes=200)
        sc.pp.filter_genes(adata_sample, min_cells=3)
        
        # Ensure unique var names before saving
        adata_sample.var_names_make_unique()
        
        # Save chunk
        chunk_path = temp_chunk_dir / f"chunk_{index}_{gex_sample_id}.h5ad"
        adata_sample.write_h5ad(chunk_path, compression='gzip')
        chunk_files.append(chunk_path)
        chunk_keys.append(sample_prefix)
        
        print(f"  Saved chunk: {adata_sample.n_obs} cells. Memory cleared.")
        
        # Cleanup
        del adata_sample
        gc.collect()

    except Exception as e:
        print(f"  Error processing {sample_prefix}: {e}")
        continue
        
    # TCR Loading (Keep separate list)
    tcr_sample_id = row['TCR_Sample_ID']
    if pd.notna(tcr_sample_id):
        tcr_file = raw_data_dir / f"{tcr_sample_id}_{s_number}_all_contig_annotations.csv.gz"
        if not tcr_file.exists():
             tcr_file = raw_data_dir / f"{tcr_sample_id}_{s_number}_all_contig_annotations.csv"
             
        if tcr_file.exists():
            try:
                # Load essential columns only
                cols = ['barcode', 'is_cell', 'contig_id', 'high_confidence', 'length', 
                        'chain', 'v_gene', 'd_gene', 'j_gene', 'c_gene', 'full_length', 
                        'productive', 'cdr3']
                        
                # Check which columns actually exist in the file first to strictly avoid errors?
                # Faster to just try/except or load all if cols obscure
                # Let's try loading header first? No, pandas handling is fine.
                tcr_df = pd.read_csv(tcr_file, usecols=lambda c: c in cols)
                tcr_df['sample_id'] = gex_sample_id
                tcr_data_list.append(tcr_df)
            except:
                pass


# --- REDUCE PHASE: Disk-safe Concatenation ---
print(f"\nStarting Reduce Phase (Merging {len(chunk_files)} chunks)...")

if not chunk_files:
    raise ValueError("No chunks were saved! Check data paths.")

merged_path = temp_chunk_dir / "merged.h5ad"
adata = None

# Prefer on-disk concatenation if available (anndata>=0.9)
try:
    if hasattr(ad, "experimental") and hasattr(ad.experimental, "concat_on_disk"):
        print("Using on-disk concatenation (anndata.experimental.concat_on_disk)...")
        # Use 'batch' as the label instead of 'sample_id' to preserve the original sample_id column
        ad.experimental.concat_on_disk(
            [str(p) for p in chunk_files],
            str(merged_path),
            join='inner',  # intersection avoids union blow-up
            merge='same',
            label='batch',  # Changed from 'sample_id' to preserve original sample_id
            keys=chunk_keys,
            index_unique='-'
        )
        adata = sc.read_h5ad(merged_path)
    else:
        raise AttributeError("concat_on_disk not available in this anndata version")
except Exception as e:
    print(f"Falling back to batch in-memory concat: {e}")
    batch_size = 4
    adata = None
    for i in range(0, len(chunk_files), batch_size):
        batch_files = chunk_files[i:i+batch_size]
        batch_adatas = [sc.read_h5ad(f) for f in batch_files]
        batch = ad.concat(batch_adatas, join='inner', merge='same', index_unique='-')
        del batch_adatas
        gc.collect()
        if adata is None:
            adata = batch
        else:
            adata = ad.concat([adata, batch], join='inner', merge='same', index_unique='-')
        del batch
        gc.collect()

# Final sparse enforcement
if not sp.issparse(adata.X):
    adata.X = sp.csr_matrix(adata.X, dtype=np.float32)

print(f"Final Merged Data: {adata.n_obs} cells x {adata.n_vars} genes")

# Merge TCR data
if tcr_data_list:
    full_tcr_df = pd.concat(tcr_data_list, ignore_index=True)
    print(f"Full TCR Data: {len(full_tcr_df)} rows")
    del tcr_data_list
else:
    print("No TCR data loaded.")

# Cleanup chunks
shutil.rmtree(temp_chunk_dir)
print("Temp chunks cleaned up.")

# Dummy adata_list for compatibility
adata_list = []

Starting Map Phase (Processing & Saving Chunks)...
Processing 1/11: GSM9061665_S1
  Saved chunk: 8804 cells. Memory cleared.
Processing 2/11: GSM9061666_S2
  Saved chunk: 9037 cells. Memory cleared.
Processing 3/11: GSM9061667_S3
  Saved chunk: 7343 cells. Memory cleared.
Processing 4/11: GSM9061668_S4
  Saved chunk: 8608 cells. Memory cleared.
Processing 5/11: GSM9061669_S5
  Saved chunk: 2887 cells. Memory cleared.
Processing 6/11: GSM9061670_S6
  Saved chunk: 10353 cells. Memory cleared.
Processing 7/11: GSM9061671_S7
  Saved chunk: 9186 cells. Memory cleared.
Processing 8/11: GSM9061672_S8
  Saved chunk: 12665 cells. Memory cleared.
Processing 9/11: GSM9061673_S9
  Saved chunk: 11216 cells. Memory cleared.
Processing 10/11: GSM9061674_S10
  Saved chunk: 9582 cells. Memory cleared.
Processing 11/11: GSM9061675_S11
  Saved chunk: 9286 cells. Memory cleared.

Starting Reduce Phase (Merging 11 chunks)...
Using on-disk concatenation (anndata.experimental.concat_on_disk)...
Final Merged 

## 3. Integrate TCR Data and Perform QC

Next, we'll merge the TCR information into the `.obs` of our main `AnnData` object. We will keep only the cells that have corresponding TCR data and filter based on the `high_confidence` flag.

In [16]:
# 3. Raw Processing Branch (Only runs if needed)
# Auto-define should_process_raw if missing to avoid NameError
if 'should_process_raw' not in globals():
    _loaded_h5ad = bool(globals().get('loaded_h5ad', False))
    _adata_missing = ('adata' not in globals()) or (adata is None)
    _metadata_ready = 'metadata_df' in globals()
    should_process_raw = _metadata_ready and (not _loaded_h5ad) and _adata_missing
    print(f"should_process_raw not set; defaulting to {should_process_raw} (loaded_h5ad={_loaded_h5ad}, adata_missing={_adata_missing})")

if should_process_raw:
    print("Starting raw data processing from metadata...")

    # Ensure raw_data_dir is defined
    if 'raw_data_dir' not in globals():
        base_dir = Path('/kaggle/working/Data') if (globals().get('IS_KAGGLE', False)) else Path('../Data')
        raw_data_dir = base_dir / 'GSE300475_RAW'
        print(f"raw_data_dir undefined. Defaulting to: {raw_data_dir}")
    
    # --- Initialize lists ---
    adata_list = []  
    tcr_data_list = []  

    # --- Iterate through each sample ---
    for index, row in metadata_df.iterrows():
        gex_sample_id = row['GEX_Sample_ID']
        tcr_sample_id = row['TCR_Sample_ID']
        s_number = row['S_Number']
        patient_id = row['Patient_ID']
        timepoint = row['Timepoint']
        response = row['Response']
        
        print(f"Processing sample {index+1}/{len(metadata_df)}: {gex_sample_id} ({s_number})...")
        
        # --- Robust File Finding (Fixing 'GEX data not found') ---
        # Pattern: *GSM123*matrix.mtx* matches both .mtx and .mtx.gz
        try:
            found_gex_files = list(raw_data_dir.rglob(f"*{gex_sample_id}*matrix.mtx*"))
        except Exception as e:
            print(f"Error searching {raw_data_dir}: {e}")
            found_gex_files = []
        
        if not found_gex_files:
            print(f"  Warning: GEX matrix file for {gex_sample_id} not found in {raw_data_dir}. Skipping.")
            try:
                 print("  Debug: Listing first 5 files in raw_data_dir to help diagnose:")
                 for i, p in enumerate(raw_data_dir.rglob('*')):
                     if i >= 5: break
                     print(f"    {p.name}")
            except: pass
            continue

should_process_raw not set; defaulting to False (loaded_h5ad=False, adata_missing=False)


## 4. TCR Integration and Data Saving

We integrate the TCR sequencing data into the AnnData object by retrieving the corresponding TCR contig annotations (TRA/TRB chains) and merging them with the gene expression data based on cell barcodes. We also perform quality control to retain only cells with high-confidence TCR data.

*Note: The code below also includes commented-out logic for saving the intermediate processed data to disk.*


In [17]:
%%time
# --- Integrate TCR data into AnnData.obs and perform quality control ---

# Validate that adata exists
if 'adata' not in globals():
    raise NameError("adata is not defined. Please run the data loading cell first.")
if adata is None:
    raise ValueError("adata is None. Data loading may have failed.")

# Check if TCR data exists and is not empty
if 'full_tcr_df' in globals() and isinstance(full_tcr_df, pd.DataFrame) and not full_tcr_df.empty:
    print(f"Integrating TCR data into AnnData (TCR contigs: {len(full_tcr_df)}, cells: {adata.n_obs})...")
    
    try:
        # --- TCR Data Aggregation ---
        # The previous join failed because one cell (barcode) can have multiple TCR contigs (e.g., TRA and TRB chains),
        # creating a one-to-many join that increases the number of rows.
        # The fix is to aggregate the TCR data to one row per cell *before* merging.

        # 1. Filter for high-confidence, productive TRA/TRB chains.
        # Only keep TCR contigs that are both high-confidence and productive, and are either TRA or TRB chains.
        if 'high_confidence' not in full_tcr_df.columns or 'productive' not in full_tcr_df.columns or 'chain' not in full_tcr_df.columns:
            print("WARNING: TCR dataframe missing required columns (high_confidence, productive, chain). Skipping TCR integration.")
            tcr_to_agg = pd.DataFrame()
        else:
            tcr_to_agg = full_tcr_df[
                (full_tcr_df['high_confidence'] == True) &
                (full_tcr_df['productive'] == True) &
                (full_tcr_df['chain'].isin(['TRA', 'TRB']))
            ].copy()

        if not tcr_to_agg.empty:
            # 2. Pivot the data to create one row per barcode, with columns for TRA and TRB data.
            # This step ensures each cell (barcode) has its TRA and TRB info in separate columns.
            tcr_aggregated = tcr_to_agg.pivot_table(
                index=['sample_id', 'barcode'],
                columns='chain',
                values=['v_gene', 'j_gene', 'cdr3'],
                aggfunc='first'  # 'first' is safe as we expect at most one productive TRA/TRB per cell
            )

            # 3. Flatten the multi-level column index (e.g., from ('v_gene', 'TRA') to 'v_gene_TRA')
            tcr_aggregated.columns = ['_'.join(col).strip() for col in tcr_aggregated.columns.values]
            tcr_aggregated.reset_index(inplace=True)

            # --- DEBUG: Print sample formats to diagnose any mismatches ---
            print(f"  DEBUG: adata.obs sample_id examples: {adata.obs['sample_id'].unique()[:3].tolist()}")
            print(f"  DEBUG: TCR sample_id examples: {tcr_aggregated['sample_id'].unique()[:3].tolist()}")
            
            # 4. Prepare adata.obs for the merge by creating a matching barcode column.
            # The index in adata.obs is like 'AGCCATGCAGCTGTTA-1-0' (barcode-concat_suffix).
            # The barcode in TCR data is like 'AGCCATGCAGCTGTTA-1'.
            adata.obs['barcode_for_merge'] = adata.obs.index.str.rsplit('-', n=1).str[0]
            
            # Handle case where sample_id might have been modified by concat (fallback fix)
            # Extract just the GSM ID if sample_id contains underscores (e.g., "GSM9061665_S1" -> "GSM9061665")
            if adata.obs['sample_id'].astype(str).str.contains('_').any():
                print("  INFO: sample_id contains underscores, extracting GSM ID portion for merge...")
                adata.obs['sample_id_for_merge'] = adata.obs['sample_id'].astype(str).str.split('_').str[0]
            else:
                adata.obs['sample_id_for_merge'] = adata.obs['sample_id']
            
            print(f"  DEBUG: adata barcode examples: {adata.obs['barcode_for_merge'].head(3).tolist()}")
            print(f"  DEBUG: TCR barcode examples: {tcr_aggregated['barcode'].head(3).tolist()}")

            # 5. Perform a left merge. This keeps all cells from adata and adds TCR info where available.
            # The number of rows will not change because tcr_aggregated has unique barcodes per sample.
            original_obs = adata.obs.copy()
            merged_obs = original_obs.merge(
                tcr_aggregated,
                left_on=['sample_id_for_merge', 'barcode_for_merge'],
                right_on=['sample_id', 'barcode'],
                how='left',
                suffixes=('', '_tcr')
            )
            
            # 6. Restore the original index to the merged dataframe.
            merged_obs.index = original_obs.index
            adata.obs = merged_obs
            
            # Clean up redundant columns from merge
            cols_to_drop = [c for c in ['sample_id_tcr', 'sample_id_for_merge'] if c in adata.obs.columns]
            if cols_to_drop:
                adata.obs.drop(columns=cols_to_drop, inplace=True)

            # Check how many cells got TCR info
            tcr_col = 'v_gene_TRA' if 'v_gene_TRA' in adata.obs.columns else None
            if tcr_col:
                cells_with_tcr = (~adata.obs[tcr_col].isna()).sum()
                print(f"Successfully merged TCR data. Cells with TCR info: {cells_with_tcr} / {adata.n_obs}")
                
                if cells_with_tcr == 0:
                    print("WARNING: No cells matched TCR data! Check barcode/sample_id formats.")
                    print("  Skipping TCR filtering to preserve data.")
                else:
                    # --- Filter for cells that have TCR information after the merge ---
                    # Only keep cells with non-null v_gene_TRA (i.e., cells with high-confidence TCR data)
                    initial_cells = adata.n_obs
                    adata = adata[~adata.obs[tcr_col].isna()].copy()
                    print(f"Filtered from {initial_cells} to {adata.n_obs} cells based on having high-confidence TCR data.")
            else:
                print("WARNING: TCR merge did not produce expected columns. Skipping TCR filtering.")
        else:
            print("WARNING: No high-confidence productive TRA/TRB chains found in TCR data. Skipping TCR filtering.")
            
    except Exception as e:
        import traceback
        print(f"ERROR during TCR integration: {e}")
        traceback.print_exc()
        print("Proceeding without TCR integration...")
else:
    print("No TCR data available or full_tcr_df is empty. Proceeding without TCR integration...")

# --- Basic QC and filtering ---
try:
    print(f"\nPerforming QC filtering (starting with {adata.n_obs} cells, {adata.n_vars} genes)...")
    
    # Filter out cells with fewer than 200 genes detected
    sc.pp.filter_cells(adata, min_genes=200)
    print(f"  After min_genes filter: {adata.n_obs} cells")
    
    # Filter out genes detected in fewer than 3 cells
    sc.pp.filter_genes(adata, min_cells=3)
    print(f"  After min_cells filter: {adata.n_vars} genes")

    # Annotate mitochondrial genes for QC metrics
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    # Calculate QC metrics (e.g., percent mitochondrial genes)
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

    print("\nPost-QC AnnData object:")
    print(adata)
    print("\nSample metadata preview:")
    display(adata.obs.head())
    
except Exception as e:
    print(f"ERROR during QC filtering: {e}")
    raise

Integrating TCR data into AnnData (TCR contigs: 162133, cells: 98967)...
  DEBUG: adata.obs sample_id examples: ['GSM9061665', 'GSM9061666', 'GSM9061667']
  DEBUG: TCR sample_id examples: ['GSM9061665', 'GSM9061666', 'GSM9061667']
  DEBUG: adata barcode examples: ['AAACCTGAGAAGGGTA-1', 'AAACCTGAGACTGTAA-1', 'AAACCTGAGCAGCGTA-1']
  DEBUG: TCR barcode examples: ['AAACCTGAGACTGTAA-1', 'AAACCTGAGCGTGAAC-1', 'AAACCTGAGCTACCTA-1']
Successfully merged TCR data. Cells with TCR info: 38413 / 98967
Filtered from 98967 to 38413 cells based on having high-confidence TCR data.

Performing QC filtering (starting with 38413 cells, 14819 genes)...
  After min_genes filter: 38413 cells
  After min_cells filter: 14816 genes

Post-QC AnnData object:
AnnData object with n_obs × n_vars = 38413 × 14816
    obs: 'sample_id', 'patient_id', 'timepoint', 'response', 'n_genes', 'batch', 'barcode_for_merge', 'barcode', 'cdr3_TRA', 'cdr3_TRB', 'j_gene_TRA', 'j_gene_TRB', 'v_gene_TRA', 'v_gene_TRB', 'n_genes_by_cou

,sample_id,patient_id,timepoint,response,n_genes,batch,barcode_for_merge,barcode,cdr3_TRA,cdr3_TRB,j_gene_TRA,j_gene_TRB,v_gene_TRA,v_gene_TRB,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt
AAACCTGAGACTGTAA-1-GSM9061665_S1,GSM9061665,PT1,Baseline,Responder,1379,GSM9061665_S1,AAACCTGAGACTGTAA-1,AAACCTGAGACTGTAA-1,CAVEARNYKLTF,CASGTGLNTEAFF,TRAJ53,TRBJ1-1,TRAV36/DV7,TRBV3-1,1379,4637.0,157.0,3.385810
AAACCTGAGCGTGAAC-1-GSM9061665_S1,GSM9061665,PT1,Baseline,Responder,1275,GSM9061665_S1,AAACCTGAGCGTGAAC-1,AAACCTGAGCGTGAAC-1,CAASAVGNEKLTF,CAWSALLGTVNGYTF,TRAJ48,TRBJ1-2,TRAV29/DV5,TRBV30,1275,4843.0,247.0,5.100144
AAACCTGAGCTACCTA-1-GSM9061665_S1,GSM9061665,PT1,Baseline,Responder,886,GSM9061665_S1,AAACCTGAGCTACCTA-1,AAACCTGAGCTACCTA-1,CALSEAWGNARLMF,CASRSREETYEQYF,TRAJ31,TRBJ2-7,TRAV19,TRBV2,886,3076.0,280.0,9.102731
AAACCTGAGCTGTTCA-1-GSM9061665_S1,GSM9061665,PT1,Baseline,Responder,1628,GSM9061665_S1,AAACCTGAGCTGTTCA-1,AAACCTGAGCTGTTCA-1,CALLGLKGEGSARQLTF,CASSLPPWRANTEAFF,TRAJ22,TRBJ1-1,TRAV9-2,TRBV11-2,1628,4914.0,288.0,5.860806
AAACCTGAGGCATTGG-1-GSM9061665_S1,GSM9061665,PT1,Baseline,Responder,1313,GSM9061665_S1,AAACCTGAGGCATTGG-1,AAACCTGAGGCATTGG-1,CAVTGFSDGQKLLF,CASSLTGEVWDEQFF,TRAJ16,TRBJ2-1,TRAV8-6,TRBV5-1,1313,4947.0,198.0,4.002426


CPU times: user 3.74 s, sys: 924 ms, total: 4.67 s
Wall time: 4.67 s


In [19]:
# --- Show basic statistics about the dataset ---

# Validate that adata exists
if 'adata' not in globals():
    raise NameError("adata is not defined. Please run the data loading and QC cells first.")
if adata is None:
    raise ValueError("adata is None. Data loading may have failed.")

print("=== Dataset Statistics ===")
print(f"Total cells: {adata.n_obs}")
print(f"Total genes: {adata.n_vars}")

if 'sample_id' in adata.obs.columns:
    print(f"\nSamples: {adata.obs['sample_id'].nunique()}")
    print(adata.obs['sample_id'].value_counts())

if 'patient_id' in adata.obs.columns:
    print(f"\nPatients: {adata.obs['patient_id'].nunique()}")
    print(adata.obs['patient_id'].value_counts())

if 'response' in adata.obs.columns:
    print(f"\nResponse distribution:")
    print(adata.obs['response'].value_counts())

if 'timepoint' in adata.obs.columns:
    print(f"\nTimepoint distribution:")
    print(adata.obs['timepoint'].value_counts())

=== Dataset Statistics ===
Total cells: 38413
Total genes: 14816

Samples: 10
sample_id
GSM9061670    5310
GSM9061674    5070
GSM9061673    5045
GSM9061671    4838
GSM9061665    4008
GSM9061666    3855
GSM9061675    3774
GSM9061667    3127
GSM9061668    2471
GSM9061669     915
Name: count, dtype: int64

Patients: 4
patient_id
PT4    13889
PT1    10990
PT3    10148
PT2     3386
Name: count, dtype: int64

Response distribution:
response
Non-Responder    24037
Responder        14376
Name: count, dtype: int64

Timepoint distribution:
timepoint
Baseline      16834
Post-Tx       14678
Recurrence     6901
Name: count, dtype: int64


## 5. Install Additional Libraries for Advanced ML and Visualization

Install and import libraries such as XGBoost, TensorFlow/Keras, scipy, and additional visualization tools for comprehensive ML analysis.

In [20]:
%%time
# --- Install required packages for genetic sequence encoding and ML ---
%pip install biopython --quiet
%pip install scikit-learn --quiet
%pip install umap-learn --quiet
%pip install hdbscan --quiet
%pip install plotly --quiet
%pip install xgboost --quiet
%pip install tensorflow --quiet

from Bio.Seq import Seq
from Bio.SeqUtils import ProtParam
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

# Import scipy for hierarchical clustering
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from scipy.stats import mannwhitneyu

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
import umap
import hdbscan
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Additional libraries installed!")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


2026-03-09 17:01:19.737045: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773075679.934254      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773075679.984721      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773075680.438316      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773075680.438364      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773075680.438369      25 computation_placer.cc:177] computation placer alr

Additional libraries installed!
CPU times: user 38.1 s, sys: 4.36 s, total: 42.5 s
Wall time: 1min 20s


In [21]:
# --- GPU acceleration helper (minimal, safe) ---
# Detect GPUs for TensorFlow, enable memory growth and mixed precision if available.
# Detect XGBoost GPU support and cuML availability.
# Provide a function _apply_gpu_patches() that will patch `models_eval` and `param_grids` in-place when they exist.

TF_GPU_AVAILABLE = False
MIXED_PRECISION_AVAILABLE = False
XGBOOST_GPU_AVAILABLE = False
CUML_AVAILABLE = False

try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    TF_GPU_AVAILABLE = len(gpus) > 0
    if TF_GPU_AVAILABLE:
        print("TensorFlow GPUs detected:", gpus)
        try:
            for g in gpus:
                tf.config.experimental.set_memory_growth(g, True)
            print("Set memory growth for TensorFlow GPUs.")
        except Exception as e:
            print("Could not set memory growth:", e)
        # Try enabling mixed precision for faster FP16 compute on modern GPUs
        try:
            from tensorflow.keras import mixed_precision
            mixed_precision.set_global_policy('mixed_float16')
            MIXED_PRECISION_AVAILABLE = True
            print("Enabled mixed precision (mixed_float16).")
        except Exception as e:
            print("Mixed precision policy not enabled:", e)
    else:
        print("No TensorFlow GPU detected.")
except Exception as e:
    print("TensorFlow import failed or no GPUs:", e)

# XGBoost GPU detection - supports both old (gpu_hist) and new (device='cuda') APIs
XGBOOST_GPU_METHOD = None  # Will be 'device' for XGBoost 2.0+, 'tree_method' for older versions
try:
    import xgboost as xgb
    xgb_version = tuple(int(x) for x in xgb.__version__.split('.')[:2])
    print(f"XGBoost version: {xgb.__version__}")
    
    # XGBoost 2.0+ uses device='cuda', older uses tree_method='gpu_hist'
    if xgb_version >= (2, 0):
        try:
            # Test new API
            _ = xgb.XGBClassifier(device='cuda', n_estimators=1)
            XGBOOST_GPU_AVAILABLE = True
            XGBOOST_GPU_METHOD = 'device'
            print("XGBoost GPU support detected (device='cuda' API).")
        except Exception as e:
            print(f"XGBoost 2.0+ GPU not available: {e}")
    else:
        try:
            # Test old API
            _ = xgb.XGBClassifier(tree_method='gpu_hist', predictor='gpu_predictor', n_estimators=1)
            XGBOOST_GPU_AVAILABLE = True
            XGBOOST_GPU_METHOD = 'tree_method'
            print("XGBoost GPU support detected (tree_method='gpu_hist' API).")
        except Exception as e:
            print(f"XGBoost GPU not available: {e}")
except Exception as e:
    print("XGBoost not importable:", e)

# cuML detection
try:
    import cuml
    CUML_AVAILABLE = True
    print("cuML is available.")
except Exception:
    CUML_AVAILABLE = False

# Utility: robust getter for adata.obsm with mask and padding
def _get_obsm_or_zeros(adata, key, mask=None, n_cols=0):
    """
    Return adata.obsm[key][mask] if present, otherwise zeros(shape=(n_rows, n_cols)).
    Ensures output is a dense numpy array with n_cols columns (pads with zeros if needed).
    """
    import numpy as _np
    # Determine number of rows requested
    if mask is not None:
        try:
            n_rows = int(mask.sum()) if hasattr(mask, 'sum') else int(sum(1 for v in mask if v))
        except Exception:
            n_rows = int(sum(1 for v in mask if v))
    else:
        n_rows = getattr(adata, 'n_obs', adata.shape[0]) if 'adata' in globals() else 0

    if key in getattr(adata, 'obsm', {}):
        arr = adata.obsm[key]
        try:
            if hasattr(arr, 'toarray'):
                arr = arr.toarray()
            arr = _np.asarray(arr)
        except Exception:
            return _np.zeros((n_rows, n_cols))
        # Apply mask if provided
        if mask is not None:
            try:
                arr = arr[mask]
            except Exception:
                arr = _np.array(arr)[mask]
        # Pad or trim columns to n_cols if requested
        if n_cols:
            if arr.shape[1] < n_cols:
                pad = _np.zeros((arr.shape[0], n_cols - arr.shape[1]))
                arr = _np.hstack([arr, pad])
            elif arr.shape[1] > n_cols:
                arr = arr[:, :n_cols]
        return arr
    else:
        return _np.zeros((n_rows, n_cols))

# Define sensible default param_grids early so LOPO can see them (will be overridden later if redefined)
param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10, 100],
        'penalty': ['l2'],
        'solver': ['liblinear']
    },
    'Decision Tree': {
        'max_depth': [5, 10, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5, 10]
    },
    'XGBoost': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0]
    }
}
print("Default param_grids defined early (can be overridden later).")

# Patching helper (improved with signature filtering and XGBoost 2.0+ support)
def _apply_gpu_patches():
    import inspect
    try:
        # Check if models_eval exists before trying to access it
        if 'models_eval' not in globals():
            return  # Nothing to patch yet
            
        models_eval_ref = globals()['models_eval']
        
        # Patch XGBoost model to use GPU params when available and supported
        if 'XGBoost' in models_eval_ref and XGBOOST_GPU_AVAILABLE:
            try:
                import xgboost as xgb_mod
                m = models_eval_ref['XGBoost']
                params = m.get_params() if hasattr(m, 'get_params') else {}
                # Determine class to instantiate (prefer wrapper if provided)
                XGBClass = globals().get('XGBClassifierSK', getattr(xgb_mod, 'XGBClassifier', None))
                if XGBClass is None:
                    raise ImportError('xgboost.XGBClassifier not found')
                # Build filtered params list based on constructor signature
                sig = inspect.signature(XGBClass.__init__)
                accepts_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())
                allowed = set(sig.parameters.keys())
                filtered_params = {}
                for k, v in params.items():
                    if accepts_kwargs or k in allowed:
                        filtered_params[k] = v
                
                # Add GPU params based on XGBoost version (2.0+ uses device, older uses tree_method)
                xgb_gpu_method = globals().get('XGBOOST_GPU_METHOD', 'tree_method')
                if xgb_gpu_method == 'device':
                    # XGBoost 2.0+ API
                    if accepts_kwargs or 'device' in allowed:
                        filtered_params['device'] = 'cuda'
                    # Remove old-style params if present
                    filtered_params.pop('tree_method', None)
                    filtered_params.pop('predictor', None)
                else:
                    # Old XGBoost API
                    if accepts_kwargs or 'tree_method' in allowed:
                        filtered_params['tree_method'] = 'gpu_hist'
                    if accepts_kwargs or 'predictor' in allowed:
                        filtered_params['predictor'] = 'gpu_predictor'
                
                # Remove unsupported keys
                filtered_params.pop('gpu_id', None)
                try:
                    models_eval_ref['XGBoost'] = XGBClass(**filtered_params)
                    print(f"Patched models_eval['XGBoost'] to use GPU (method={xgb_gpu_method}).")
                except TypeError as e:
                    # Fallback: try removing GPU-specific params and re-instantiate
                    for k in ['tree_method', 'predictor', 'device']:
                        filtered_params.pop(k, None)
                    fallback_params = {k: v for k, v in filtered_params.items() if accepts_kwargs or k in allowed}
                    models_eval_ref['XGBoost'] = XGBClass(**fallback_params)
                    print("Patched models_eval['XGBoost'] without GPU params due to TypeError:", e)
            except Exception as e:
                print("Failed to patch models_eval['XGBoost']:", e)
            # Patch Random Forest to use n_jobs=-1 when possible
            if 'Random Forest' in models_eval_ref:
                try:
                    from sklearn.ensemble import RandomForestClassifier
                    m = models_eval_ref['Random Forest']
                    params = m.get_params() if hasattr(m, 'get_params') else {}
                    params.setdefault('n_jobs', -1)
                    RFC = RandomForestClassifier
                    sig_rfc = inspect.signature(RFC.__init__)
                    accepts_kwargs_rfc = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig_rfc.parameters.values())
                    allowed_rfc = set(sig_rfc.parameters.keys())
                    filtered_rfc_params = {k: v for k, v in params.items() if accepts_kwargs_rfc or k in allowed_rfc}
                    models_eval_ref['Random Forest'] = RandomForestClassifier(**filtered_rfc_params)
                    print("Patched models_eval['Random Forest'] to use n_jobs=-1.")
                except Exception as e:
                    print("Failed to patch models_eval['Random Forest']:", e)
    except Exception as e:
        print("Error patching models_eval:", e)

    # Patch param_grids for XGBoost if available
    try:
        if 'param_grids' in globals() and XGBOOST_GPU_AVAILABLE:
            pg = param_grids.get('XGBoost', {})
            xgb_gpu_method = globals().get('XGBOOST_GPU_METHOD', 'tree_method')
            if xgb_gpu_method == 'device':
                # XGBoost 2.0+ uses device parameter
                if any(k.startswith('clf__') for k in pg.keys()):
                    pg.setdefault('clf__device', ['cuda'])
                else:
                    pg.setdefault('device', ['cuda'])
            else:
                # Old XGBoost uses tree_method
                if any(k.startswith('clf__') for k in pg.keys()):
                    pg.setdefault('clf__tree_method', ['gpu_hist'])
                    pg.setdefault('clf__predictor', ['gpu_predictor'])
                else:
                    pg.setdefault('tree_method', ['gpu_hist'])
                    pg.setdefault('predictor', ['gpu_predictor'])
            param_grids['XGBoost'] = pg
            print(f"Patched param_grids['XGBoost'] with GPU options (method={xgb_gpu_method}).")
    except Exception as e:
        print("Error patching param_grids:", e)

# Apply patches now if models/param grids already defined
_apply_gpu_patches()

print(f"TF_GPU_AVAILABLE={TF_GPU_AVAILABLE}, MIXED_PRECISION={MIXED_PRECISION_AVAILABLE}, XGBOOST_GPU_AVAILABLE={XGBOOST_GPU_AVAILABLE}, CUML_AVAILABLE={CUML_AVAILABLE}")
print("If models or param_grids are defined later, call _apply_gpu_patches() to apply GPU settings.")

TensorFlow GPUs detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Set memory growth for TensorFlow GPUs.
Enabled mixed precision (mixed_float16).
XGBoost version: 3.1.3
XGBoost GPU support detected (device='cuda' API).
cuML is available.
Default param_grids defined early (can be overridden later).
TF_GPU_AVAILABLE=True, MIXED_PRECISION=True, XGBOOST_GPU_AVAILABLE=True, CUML_AVAILABLE=True
If models or param_grids are defined later, call _apply_gpu_patches() to apply GPU settings.


In [22]:
# --- Define a sklearn-compatible XGBoost wrapper (supports both old and new XGBoost APIs) ---
try:
    import xgboost as xgb
    xgb_version = tuple(int(x) for x in xgb.__version__.split('.')[:2])
    
    class XGBClassifierSK(xgb.XGBClassifier):
        """XGBoost wrapper that handles both old (tree_method) and new (device) APIs."""
        def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=6, random_state=None,
                     use_label_encoder=False, eval_metric='logloss',
                     tree_method=None, predictor=None, device=None, **kwargs):
            # Store parameters before passing to super().__init__
            # This ensures they are available for sklearn's get_params()
            self.tree_method = tree_method
            self.predictor = predictor
            self.device = device
            
            # Handle XGBoost 2.0+ API vs older versions
            if xgb_version >= (2, 0):
                # New API: use 'device' parameter
                if device is not None:
                    kwargs['device'] = device
                # tree_method and predictor are deprecated in 2.0+
            else:
                # Old API: use tree_method/predictor
                if tree_method is not None:
                    kwargs.setdefault('tree_method', tree_method)
                if predictor is not None:
                    kwargs.setdefault('predictor', predictor)
            
            # Remove deprecated parameters that might cause warnings
            self.use_label_encoder = use_label_encoder
            kwargs.pop('use_label_encoder', None)
            
            super().__init__(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth,
                             random_state=random_state, eval_metric=eval_metric, **kwargs)
    
    globals()['XGBClassifierSK'] = XGBClassifierSK
    print(f'Defined XGBoost sklearn-compatible wrapper: XGBClassifierSK (XGBoost version {xgb.__version__})')
except Exception as e:
    print('Failed to define XGBClassifierSK:', e)

Defined XGBoost sklearn-compatible wrapper: XGBClassifierSK (XGBoost version 3.1.3)


In [23]:
from pathlib import Path
import os

# Determine data directory consistently (prefer existing download_dir when present)
if 'download_dir' in globals() and download_dir:
    data_dir = Path(download_dir)
elif IS_KAGGLE:
    data_dir = Path('/kaggle/working/Data')
else:
    data_dir = Path('../Data')

raw_data_dir = data_dir / 'GSE300475_RAW'
raw_data_dir = raw_data_dir.resolve()

# Ensure directory exists (no-op if not writing yet)
os.makedirs(raw_data_dir, exist_ok=True)
print(f"Using raw_data_dir = {raw_data_dir}")

Using raw_data_dir = /kaggle/working/Data/GSE300475_RAW


In [24]:
# --- Auto-apply GPU patches when LOPO is instantiated ---
try:
    import sklearn.model_selection as _skms
    if not getattr(_skms, '_LO_patched_applied', False):
        _LO_orig = _skms.LeaveOneGroupOut
        class _LO_patched(_LO_orig):
            def __init__(self, *args, **kwargs):
                # Ensure GPU patches are applied just before LOPO is constructed
                try:
                    _apply_gpu_patches()
                except Exception as _e:
                    print('Warning: _apply_gpu_patches failed during LOPO patching:', _e)
                super().__init__(*args, **kwargs)
        _skms.LeaveOneGroupOut = _LO_patched
        _skms._LO_patched_applied = True
        print('Patched sklearn.model_selection.LeaveOneGroupOut to auto-apply GPU patches on init')
    else:
        print('LOPO patch already applied')
except Exception as e:
    print('Failed to apply LOPO patch:', e)

Patched sklearn.model_selection.LeaveOneGroupOut to auto-apply GPU patches on init


In [25]:
# --- Data Loading (Robust) ---
import scanpy as sc
import os
import glob
import pandas as pd
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

def _first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

def _glob_pick(folder, patterns, key=None):
    matches = []
    for pat in patterns:
        matches.extend(glob.glob(os.path.join(folder, pat)))
    matches = sorted(set(matches))
    if key:
        key_matches = [m for m in matches if key in os.path.basename(m)]
        if len(key_matches) == 1:
            return key_matches[0]
        if len(key_matches) > 1:
            return key_matches[0]
    if len(matches) == 1:
        return matches[0]
    return None

print("Starting Data Loading...")

# Determine data directory (using extract_dir from Cell 7 if available)
if 'extract_dir' not in globals():
    # Fallback path logic matching Cell 7/8
    base_dir = '/kaggle/working/Data' if IS_KAGGLE else '../Data'
    extract_dir = os.path.join(base_dir, "GSE300475_RAW")

if not os.path.exists(extract_dir):
    print(f"Warning: Directory {extract_dir} does not exist. Please ensure Cell 7 ran successfully.")
else:
    print(f"Searching for data in: {extract_dir}")
    # Find all matrix files
    matrix_files = glob.glob(os.path.join(extract_dir, "*matrix.mtx*"))
    # Also look recursively if structure is nested
    if not matrix_files:
        matrix_files = glob.glob(os.path.join(extract_dir, "**", "*matrix.mtx*"), recursive=True)

    adata_list = []
    
    if not matrix_files:
        print("No matrix.mtx files found or previously loaded.")
        # Check if we can proceed? If this is a re-run, adata might exist.
    else:
        for mat_file in matrix_files:
            try:
                print(f"Processing {os.path.basename(mat_file)}...")
                # Handle formatted loading
                # If file is standard 10x-like (matrix.mtx, genes.tsv, barcodes.tsv) in same folder
                folder = os.path.dirname(mat_file)
                prefix = os.path.basename(mat_file).replace('matrix.mtx', '').replace('.gz', '')
                key = prefix.strip('_')
                
                # Check for accompanying files with same prefix
                genes_path = _first_existing([
                    os.path.join(folder, prefix + 'genes.tsv'),
                    os.path.join(folder, prefix + 'features.tsv'),
                    os.path.join(folder, prefix + 'genes.tsv.gz'),
                    os.path.join(folder, prefix + 'features.tsv.gz'),
                ])
                
                barcodes_path = _first_existing([
                    os.path.join(folder, prefix + 'barcodes.tsv'),
                    os.path.join(folder, prefix + 'barcodes.tsv.gz'),
                ])

                # Fallback to un-prefixed standard 10x naming
                if not genes_path:
                    genes_path = _first_existing([
                        os.path.join(folder, 'genes.tsv'),
                        os.path.join(folder, 'features.tsv'),
                        os.path.join(folder, 'genes.tsv.gz'),
                        os.path.join(folder, 'features.tsv.gz'),
                    ])

                if not barcodes_path:
                    barcodes_path = _first_existing([
                        os.path.join(folder, 'barcodes.tsv'),
                        os.path.join(folder, 'barcodes.tsv.gz'),
                    ])

                # Fallback to any matching files in the folder (use key if present)
                if not genes_path:
                    genes_path = _glob_pick(folder, ['*genes.tsv*', '*features.tsv*'], key=key)
                if not barcodes_path:
                    barcodes_path = _glob_pick(folder, ['*barcodes.tsv*'], key=key)

                if genes_path and barcodes_path and os.path.exists(genes_path) and os.path.exists(barcodes_path):
                    # Load using read_mtx for flexibility with filenames
                    adata_sample = sc.read_mtx(mat_file).T
                    
                    # Annotation
                    genes = pd.read_csv(genes_path, sep='\t', header=None)
                    barcodes = pd.read_csv(barcodes_path, sep='\t', header=None)
                    
                    # Assign var/obs names and sanitize whitespace
                    if genes.shape[1] > 1:
                        var_names = genes.iloc[:,1].astype(str).str.strip().values
                        adata_sample.var['gene_ids'] = genes.iloc[:,0].astype(str).values
                    else:
                        var_names = genes.iloc[:,0].astype(str).str.strip().values
                    adata_sample.var_names = pd.Index(var_names)
                    adata_sample.obs_names = pd.Index(barcodes.iloc[:,0].astype(str).str.strip().values)
                    adata_sample.obs['sample_id'] = prefix.strip('_') if prefix else os.path.basename(folder)
                    
                    # Ensure uniqueness within sample to avoid concat Index errors
                    try:
                        adata_sample.var_names_make_unique()
                        adata_sample.obs_names_make_unique()
                    except Exception:
                        pass
                    
                    adata_list.append(adata_sample)
                    print(f"Loaded {adata_sample.shape[0]} cells from {prefix or folder}")
                else:
                    print(f"Skipping {mat_file}: Missing genes/barcodes files (searched prefix '{prefix}' and fallbacks)")
            except Exception as e:
                print(f"Error loading {mat_file}: {e}")

        # Pre-sanitize all adata samples before concatenation
        for a in adata_list:
            try:
                a.var_names = pd.Index([str(v).strip() for v in a.var_names])
                a.var_names_make_unique()
                a.obs_names = pd.Index([str(v).strip() for v in a.obs_names])
                a.obs_names_make_unique()
            except Exception:
                pass

        if adata_list:
            # Concatenate all samples
            try:
                adata = sc.concat(adata_list, join='outer')
            except Exception as e:
                print('sc.concat failed:', e)
                # Try fallback using AnnData.concatenate with batch info
                try:
                    loaded_batches = [a.obs['sample_id'].unique()[0] for a in adata_list]
                except Exception:
                    loaded_batches = None
                try:
                    if loaded_batches:
                        adata = sc.AnnData.concatenate(*adata_list, join='outer', batch_key='sample_id', batch_categories=loaded_batches)
                    else:
                        adata = sc.AnnData.concatenate(*adata_list, join='outer', batch_key='sample_id')
                except Exception as e2:
                    raise RuntimeError(f"Failed to concatenate AnnData objects: {e}; fallback failed: {e2}")
            adata.obs_names_make_unique()
            # Basic fallback for mitochondrial genes logic (used later)
            adata.var['mt'] = adata.var_names.str.startswith('MT-')
            sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
            print(f"Combined AnnData object created: {adata.shape}")
        else:
            print("Warning: No valid data loaded into adata.")

# Ensure adata exists to prevent downstream crashes
if 'adata' not in globals():
    print("CRITICAL CHECK: adata variable not defined. Downstream cells will fail.")


Starting Data Loading...
Searching for data in: /kaggle/working/Data/GSE300475_RAW
Processing GSM9061670_S6_matrix.mtx.gz...
Loaded 10398 cells from GSM9061670_S6_
Processing GSM9061672_S8_matrix.mtx.gz...
Loaded 12832 cells from GSM9061672_S8_
Processing GSM9061665_S1_matrix.mtx.gz...
Loaded 8931 cells from GSM9061665_S1_
Processing GSM9061668_S4_matrix.mtx.gz...
Loaded 8723 cells from GSM9061668_S4_
Processing GSM9061675_S11_matrix.mtx.gz...
Loaded 9330 cells from GSM9061675_S11_
Processing GSM9061666_S2_matrix.mtx.gz...
Loaded 9069 cells from GSM9061666_S2_
Processing GSM9061667_S3_matrix.mtx.gz...
Loaded 7358 cells from GSM9061667_S3_
Processing GSM9061674_S10_matrix.mtx.gz...
Loaded 9704 cells from GSM9061674_S10_
Processing GSM9061669_S5_matrix.mtx.gz...
Loaded 2912 cells from GSM9061669_S5_
Processing GSM9061671_S7_matrix.mtx.gz...
Loaded 9330 cells from GSM9061671_S7_
Processing GSM9061673_S9_matrix.mtx.gz...
Loaded 11480 cells from GSM9061673_S9_
Combined AnnData object create

## 6. Genetic Sequence Encoding Functions

Define functions for one-hot encoding, k-mer encoding, and physicochemical features extraction for TCR sequences and gene expression patterns.

In [26]:
%%time
# --- Genetic Sequence Encoding Functions ---

def one_hot_encode_sequence(sequence, max_length=50, alphabet='ACDEFGHIKLMNPQRSTVWY'):
    """
    One-hot encode a protein/nucleotide sequence.
    Args:
        sequence: String sequence to encode
        max_length: Maximum sequence length (pad or truncate)
        alphabet: Valid characters in the sequence
    Returns:
        2D numpy array of shape (max_length, len(alphabet))
    """
    if pd.isna(sequence) or sequence == 'NA' or sequence == '':
        return np.zeros((max_length, len(alphabet)))
    
    sequence = str(sequence).upper()[:max_length]  # Truncate if too long
    encoding = np.zeros((max_length, len(alphabet)))
    
    for i, char in enumerate(sequence):
        if char in alphabet:
            char_idx = alphabet.index(char)
            encoding[i, char_idx] = 1
    
    return encoding

def kmer_encode_sequence(sequence, k=3, alphabet='ACDEFGHIKLMNPQRSTVWY'):
    """
    K-mer encoding of sequences.
    """
    if pd.isna(sequence) or sequence == 'NA' or sequence == '':
        return {}
    
    sequence = str(sequence).upper()
    kmers = [sequence[i:i+k] for i in range(len(sequence)-k+1)]
    valid_kmers = [kmer for kmer in kmers if all(c in alphabet for c in kmer)]
    
    return Counter(valid_kmers)

def physicochemical_features(sequence):
    """
    Extract physicochemical properties from protein sequences.
    """
    if pd.isna(sequence) or sequence == 'NA' or sequence == '':
        return {
            'length': 0, 'molecular_weight': 0, 'aromaticity': 0,
            'instability_index': 0, 'isoelectric_point': 0, 'hydrophobicity': 0
        }
    
    try:
        seq = str(sequence).upper()
        # Remove non-standard amino acids
        seq = ''.join([c for c in seq if c in 'ACDEFGHIKLMNPQRSTVWY'])
        
        if len(seq) == 0:
            return {
                'length': 0, 'molecular_weight': 0, 'aromaticity': 0,
                'instability_index': 0, 'isoelectric_point': 0, 'hydrophobicity': 0
            }
        
        bio_seq = Seq(seq)
        analyzer = ProtParam.ProteinAnalysis(str(bio_seq))
        
        return {
            'length': len(seq),
            'molecular_weight': analyzer.molecular_weight(),
            'aromaticity': analyzer.aromaticity(),
            'instability_index': analyzer.instability_index(),
            'isoelectric_point': analyzer.isoelectric_point(),
            'hydrophobicity': analyzer.gravy()
        }
    except:
        return {
            'length': len(str(sequence)) if not pd.isna(sequence) else 0,
            'molecular_weight': 0, 'aromaticity': 0,
            'instability_index': 0, 'isoelectric_point': 0, 'hydrophobicity': 0
        }

def encode_gene_expression_patterns(adata, n_top_genes=1000, train_mask=None):
    """
    Encode gene expression patterns using various dimensionality reduction techniques.
    
    IMPORTANT: To avoid data leakage, pass train_mask to fit transformers only on training data.
    If train_mask is None, fits on all data (use only for exploration, not CV).
    
    Returns:
        tuple: (encodings dict, X_scaled array)
    """
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA, TruncatedSVD
    import umap
    
    # Get highly variable genes if not already computed
    if 'highly_variable' not in adata.var.columns:
        sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes, subset=False)
    
    # Extract expression matrix for highly variable genes - ensure boolean mask
    hvg_mask = np.array(adata.var['highly_variable'].values, dtype=bool)
    
    # Subset adata by HVG mask
    X_full = adata.X
    if hasattr(X_full, 'toarray'):
        X_full = X_full.toarray()
    else:
        X_full = np.asarray(X_full)
    
    # Select HVG columns
    X_hvg = X_full[:, hvg_mask]
    
    # Clean Infs/NaNs (robustness fix)
    X_hvg = np.nan_to_num(X_hvg, nan=0.0, posinf=0.0, neginf=0.0)

    # Standardize the data - FIT ONLY ON TRAINING DATA if mask provided
    scaler = StandardScaler()
    if train_mask is not None:
        scaler.fit(X_hvg[train_mask])
        X_scaled = scaler.transform(X_hvg)
    else:
        X_scaled = scaler.fit_transform(X_hvg)
    
    encodings = {}
    
    # PCA encoding - FIT ONLY ON TRAINING DATA if mask provided
    n_pca = min(50, X_scaled.shape[1], X_scaled.shape[0])
    pca = PCA(n_components=n_pca)
    if train_mask is not None:
        pca.fit(X_scaled[train_mask])
        encodings['pca'] = pca.transform(X_scaled)
    else:
        encodings['pca'] = pca.fit_transform(X_scaled)
    
    # TruncatedSVD for sparse matrices
    n_svd = min(50, X_scaled.shape[1] - 1, X_scaled.shape[0] - 1)
    if n_svd > 0:
        svd = TruncatedSVD(n_components=n_svd, random_state=42)
        if train_mask is not None:
            svd.fit(X_scaled[train_mask])
            encodings['svd'] = svd.transform(X_scaled)
        else:
            encodings['svd'] = svd.fit_transform(X_scaled)
    else:
        encodings['svd'] = np.zeros((X_scaled.shape[0], 1))
    
    # UMAP encoding - UMAP doesn't support clean fit/transform easily for this pipeline, usually unsupervised
    try:
        umap_encoder = umap.UMAP(n_components=20, random_state=42)
        encodings['umap'] = umap_encoder.fit_transform(X_scaled)
    except Exception as e:
        print(f"UMAP failed: {e}")
        encodings['umap'] = np.zeros((X_scaled.shape[0], 20))
    
    return encodings, X_scaled

print("Genetic sequence encoding functions defined successfully!")

Genetic sequence encoding functions defined successfully!
CPU times: user 122 µs, sys: 0 ns, total: 122 µs
Wall time: 128 µs


## 7. Apply Sequence Encoding to TCR CDR3 Sequences

Encode TRA and TRB CDR3 sequences using one-hot, k-mer, and physicochemical methods, and add to AnnData.obsm and obs.

In [27]:
%%time
# --- MEMORY-OPTIMIZED TCR Sequence Encoding ---

# Validate that adata exists
if 'adata' not in globals():
    raise NameError("adata is not defined. Please run the data loading and QC cells first.")
if adata is None:
    raise ValueError("adata is None. Data loading may have failed.")

print("Starting memory-optimized TCR sequence encoding...")
import gc

# Extract TCR CDR3 sequences with robust column handling
# Check for column existence and normalize naming
if 'cdr3_TRA' not in adata.obs.columns:
    if 'CDR3_TRA' in adata.obs.columns:
        adata.obs['cdr3_TRA'] = adata.obs['CDR3_TRA']
    else:
        print("Warning: cdr3_TRA column not found, creating empty column")
        adata.obs['cdr3_TRA'] = ''

if 'cdr3_TRB' not in adata.obs.columns:
    if 'CDR3_TRB' in adata.obs.columns:
        adata.obs['cdr3_TRB'] = adata.obs['CDR3_TRB']
    else:
        print("Warning: cdr3_TRB column not found, creating empty column")
        adata.obs['cdr3_TRB'] = ''

tra_seqs = adata.obs['cdr3_TRA'].fillna('').astype(str).values
trb_seqs = adata.obs['cdr3_TRB'].fillna('').astype(str).values

print(f"TRA sequences: {len(tra_seqs)}, TRB sequences: {len(trb_seqs)}")

# MEMORY FIX: Use smaller k-mer sizes and reduced dimensionality
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD

# Reduce k-mer size from 3 to 2 to reduce feature space
def _kmer_list(seq, k=2):  # Changed from k=3 to k=2
    if len(seq) < k:
        return []
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]

# Convert sequences to k-mer strings
tra_kmer_docs = [' '.join(_kmer_list(s, k=2)) for s in tra_seqs]  # k=2 instead of 3
trb_kmer_docs = [' '.join(_kmer_list(s, k=2)) for s in trb_seqs]

# MEMORY FIX: Limit max features to reduce dimensionality
vec_tra = CountVectorizer(max_features=500)  # Limit to 500 features instead of unlimited
vec_trb = CountVectorizer(max_features=500)

tra_kmer_sparse = vec_tra.fit_transform(tra_kmer_docs)
trb_kmer_sparse = vec_trb.fit_transform(trb_kmer_docs)

# Clean up k-mer docs (no longer needed)
del tra_kmer_docs, trb_kmer_docs
gc.collect()

print(f"TRA k-mer sparse shape: {tra_kmer_sparse.shape}")
print(f"TRB k-mer sparse shape: {trb_kmer_sparse.shape}")

# MEMORY FIX: Reduce dimensions even further using SVD
def _reduce_sparse(sparse_mat, n_components=50):  # Reduced from 200 to 50
    n_comp = min(n_components, max(1, sparse_mat.shape[1]-1))
    try:
        svd = TruncatedSVD(n_components=n_comp, random_state=42)
        return svd.fit_transform(sparse_mat).astype(np.float32)  # Use float32
    except Exception:
        return sparse_mat.toarray().astype(np.float32) if hasattr(sparse_mat, 'toarray') else np.asarray(sparse_mat, dtype=np.float32)

tra_kmer_matrix = _reduce_sparse(tra_kmer_sparse, n_components=50)  # Reduced from 200
trb_kmer_matrix = _reduce_sparse(trb_kmer_sparse, n_components=50)
print(f"TRA k-mer reduced shape: {tra_kmer_matrix.shape}")
print(f"TRB k-mer reduced shape: {trb_kmer_matrix.shape}")

# Clean up sparse matrices
del tra_kmer_sparse, trb_kmer_sparse
gc.collect()

# MEMORY FIX: Reduced one-hot encoding with smaller max length
max_cdr3_length = 15  # Reduced from 20 to 15
alphabet = 'ACDEFGHIKLMNPQRSTVWY'
char_to_idx = {c:i for i,c in enumerate(alphabet)}

def _one_hot_encode_batch(sequences, max_len=max_cdr3_length):
    """Batch one-hot encoding using NumPy for memory efficiency."""
    n_seqs = len(sequences)
    encoding = np.zeros((n_seqs, max_len, len(alphabet)), dtype=np.float16)  # Use float16 instead of float32
    
    for i, seq in enumerate(sequences):
        seq_str = str(seq).upper()[:max_len]  # Truncate
        for j, char in enumerate(seq_str):
            if char in char_to_idx:
                encoding[i, j, char_to_idx[char]] = 1.0
    
    return encoding

tra_one_hot = _one_hot_encode_batch(tra_seqs, max_cdr3_length)
trb_one_hot = _one_hot_encode_batch(trb_seqs, max_cdr3_length)
print(f"TRA one-hot shape: {tra_one_hot.shape} (dtype: {tra_one_hot.dtype})")
print(f"TRB one-hot shape: {trb_one_hot.shape} (dtype: {trb_one_hot.dtype})")

# Clean up sequence arrays
del tra_seqs, trb_seqs
gc.collect()

# MEMORY FIX: Store matrices in obsm (compressed format in AnnData)
adata.obsm['X_tcr_tra_kmer'] = tra_kmer_matrix
adata.obsm['X_tcr_trb_kmer'] = trb_kmer_matrix

# MEMORY FIX: Flatten and store one-hot as float32 for compatibility
adata.obsm['X_tcr_tra_onehot'] = tra_one_hot.reshape(tra_one_hot.shape[0], -1).astype(np.float32)
adata.obsm['X_tcr_trb_onehot'] = trb_one_hot.reshape(trb_one_hot.shape[0], -1).astype(np.float32)

del tra_one_hot, trb_one_hot, tra_kmer_matrix, trb_kmer_matrix
gc.collect()

print("TCR sequence encoding complete and stored in adata.obsm")
print(f"Memory usage reduced by using sparse matrices and dimension reduction")

Starting memory-optimized TCR sequence encoding...
TRA sequences: 100067, TRB sequences: 100067
TRA k-mer sparse shape: (100067, 1)
TRB k-mer sparse shape: (100067, 1)
TRA k-mer reduced shape: (100067, 1)
TRB k-mer reduced shape: (100067, 1)
TRA one-hot shape: (100067, 15, 20) (dtype: float16)
TRB one-hot shape: (100067, 15, 20) (dtype: float16)
TCR sequence encoding complete and stored in adata.obsm
Memory usage reduced by using sparse matrices and dimension reduction
CPU times: user 2.64 s, sys: 39.5 ms, total: 2.68 s
Wall time: 2.67 s


## 8. Advanced TCR Feature Engineering

We refine our TCR feature set by applying vectorized k-mer encoding and reduced one-hot encoding. This step creates the dense feature matrices stored in `adata.obsm` that will be used for downstream machine learning tasks.


## Feature Engineering and Encoding
A core contribution of this work is the engineering of a comprehensive feature set that translates biological sequences into machine-readable vectors. We developed three distinct encoding schemes for the TCR CDR3 amino acid sequences:

1.  **One-Hot Encoding:** This method creates a sparse binary matrix representing the presence or absence of specific amino acids at each position in the sequence. It preserves exact positional information, which is crucial for structural motifs, but results in high-dimensional, sparse vectors.
2.  **K-mer Frequency Encoding:** We decomposed sequences into overlapping substrings of length $k$ (k-mers, with $k=3$). We then calculated the frequency of each unique 3-mer in the sequence. This approach captures short, local structural motifs (e.g., "CAS", "ASS") that may be shared across different TCRs with similar antigen specificity, regardless of their exact position.
3.  **Physicochemical Property Encoding:** To capture the biophysical nature of the TCR-antigen interaction, we mapped each amino acid to a vector of physicochemical properties, including hydrophobicity, molecular weight, isoelectric point, and polarity. We then aggregated these values (e.g., mean, sum) across the CDR3 sequence. This results in a dense, low-dimensional representation that reflects the "binding potential" of the receptor.

These TCR features were concatenated with the top 50 Principal Components (PCs) derived from the gene expression data to form the "Comprehensive" feature set.

In [28]:
%%time
# --- Apply Sequence Encoding to TCR CDR3 Sequences (vectorized k-mer + reduced one-hot) ---

print("Encoding TCR CDR3 sequences (vectorized k-mer + reduced one-hot)...")

# Extract and clean CDR3 sequences
if 'cdr3_TRA' in adata.obs.columns:
    cdr3_TRA = adata.obs['cdr3_TRA'].astype(str).fillna('').str.upper()
else:
    cdr3_TRA = pd.Series([''] * adata.n_obs, index=adata.obs.index)
if 'cdr3_TRB' in adata.obs.columns:
    cdr3_TRB = adata.obs['cdr3_TRB'].astype(str).fillna('').str.upper()
else:
    cdr3_TRB = pd.Series([''] * adata.n_obs, index=adata.obs.index)

valid_aa = 'ACDEFGHIKLMNPQRSTVWY'
def _clean_seq(s):
    return ''.join([c for c in str(s) if c in valid_aa])

tra_seqs = [_clean_seq(s) for s in cdr3_TRA]
trb_seqs = [_clean_seq(s) for s in cdr3_TRB]

# --- Vectorized k-mer encoding using CountVectorizer (sparse) ---
from sklearn.feature_extraction.text import CountVectorizer
k = 3
vec_tra = CountVectorizer(analyzer='char', ngram_range=(k,k))
vec_trb = CountVectorizer(analyzer='char', ngram_range=(k,k))
tra_kmer_sparse = vec_tra.fit_transform(tra_seqs)
trb_kmer_sparse = vec_trb.fit_transform(trb_seqs)
print(f"TRA k-mer sparse shape: {tra_kmer_sparse.shape}")
print(f"TRB k-mer sparse shape: {trb_kmer_sparse.shape}")

# Reduce k-mer sparse matrices with TruncatedSVD to a dense reduced representation (keeps memory low)
from sklearn.decomposition import TruncatedSVD
def _reduce_sparse(sparse_mat, n_components=200):
    n_comp = min(n_components, max(1, sparse_mat.shape[1]-1))
    try:
        svd = TruncatedSVD(n_components=n_comp, random_state=42)
        return svd.fit_transform(sparse_mat)
    except Exception:
        # Fallback to dense (small datasets)
        return sparse_mat.toarray() if hasattr(sparse_mat, 'toarray') else np.asarray(sparse_mat)

tra_kmer_matrix = _reduce_sparse(tra_kmer_sparse, n_components=200)
trb_kmer_matrix = _reduce_sparse(trb_kmer_sparse, n_components=200)
print(f"TRA k-mer reduced shape: {tra_kmer_matrix.shape}")
print(f"TRB k-mer reduced shape: {trb_kmer_matrix.shape}")

# --- Reduced one-hot encoding: limit max length to avoid huge dense matrices ---
max_cdr3_length = 20  # smaller to reduce dimensionality and memory
alphabet = 'ACDEFGHIKLMNPQRSTVWY'
char_to_idx = {c:i for i,c in enumerate(alphabet)}
def _onehot_flat_list(seqs, max_length, alphabet, char_to_idx):
    out = np.zeros((len(seqs), max_length * len(alphabet)), dtype=np.uint8)
    for i, s in enumerate(seqs):
        for j, ch in enumerate(s[:max_length]):
            if ch in char_to_idx:
                out[i, j * len(alphabet) + char_to_idx[ch]] = 1
    return out

tra_onehot_flat = _onehot_flat_list(tra_seqs, max_cdr3_length, alphabet, char_to_idx)
trb_onehot_flat = _onehot_flat_list(trb_seqs, max_cdr3_length, alphabet, char_to_idx)
print(f"TRA one-hot flat shape: {tra_onehot_flat.shape}")
print(f"TRB one-hot flat shape: {trb_onehot_flat.shape}")

# --- Physicochemical properties (unchanged) ---
tra_physico = pd.DataFrame([physicochemical_features(seq) for seq in tra_seqs])
trb_physico = pd.DataFrame([physicochemical_features(seq) for seq in trb_seqs])
print(f"TRA physicochemical features shape: {tra_physico.shape}")
print(f"TRB physicochemical features shape: {trb_physico.shape}")

# Add to AnnData object (reduced, memory-friendly)
adata.obsm['X_tcr_tra_onehot'] = tra_onehot_flat
adata.obsm['X_tcr_trb_onehot'] = trb_onehot_flat
adata.obsm['X_tcr_tra_kmer'] = tra_kmer_matrix
adata.obsm['X_tcr_trb_kmer'] = trb_kmer_matrix

# Add physicochemical features to obs
for col in tra_physico.columns:
    adata.obs[f'tra_{col}'] = tra_physico[col].values
for col in trb_physico.columns:
    adata.obs[f'trb_{col}'] = trb_physico[col].values

print("TCR sequence encoding completed and added to AnnData object!")

# Clean up large temporary objects
import gc
try:
    # delete sparse intermediates and local copies â€” AnnData already stores the reduced matrices
    del tra_kmer_sparse, trb_kmer_sparse
except Exception:
    pass
try:
    # delete other large temporaries that have been copied into `adata.obsm` or `adata.obs`
    for _n in ['tra_kmer_matrix', 'trb_kmer_matrix', 'tra_onehot_flat', 'trb_onehot_flat', 'tra_physico', 'trb_physico', 'tra_seqs', 'trb_seqs', 'vec_tra', 'vec_trb', 'char_to_idx']:
        if _n in globals():
            try:
                del globals()[_n]
            except Exception:
                pass
except Exception:
    pass
gc.collect()

Encoding TCR CDR3 sequences (vectorized k-mer + reduced one-hot)...
TRA k-mer sparse shape: (100067, 1)
TRB k-mer sparse shape: (100067, 1)
TRA k-mer reduced shape: (100067, 1)
TRB k-mer reduced shape: (100067, 1)
TRA one-hot flat shape: (100067, 400)
TRB one-hot flat shape: (100067, 400)
TRA physicochemical features shape: (100067, 6)
TRB physicochemical features shape: (100067, 6)
TCR sequence encoding completed and added to AnnData object!
CPU times: user 1.52 s, sys: 29.1 ms, total: 1.55 s
Wall time: 1.54 s


0

## 9. Gene Expression Encoding

We define and apply a memory-optimized function to encode gene expression patterns. This includes highly variable gene selection, normalization, log-transformation, and dimensionality reduction using PCA and UMAP.


In [29]:
%%time
# MEMORY-OPTIMIZED encode_gene_expression_patterns function using Scanpy's Native Optimized PCA
def encode_gene_expression_patterns(adata, n_top_genes=1500, train_mask=None):
    """
    Encode gene expression patterns using Scanpy's optimized sparse PCA (Arpack)
    and TruncatedSVD. Avoids manual chunking complexity which can be error prone.
    
    Returns:
        tuple: (encodings dict, X_scaled placeholder)
    """
    import numpy as np
    import gc
    from sklearn.decomposition import TruncatedSVD
    import umap
    
    gc.collect()
    
    # 1. HVG Selection (Memory Optimized: Subsample cells if huge)
    # Calculating mean/var on 100k cells x 30k genes can overlap memory.
    # We calculate on a subset of 20k cells to estimate HVGs.
    print("Selecting Highly Variable Genes...")
    
    if adata.n_obs > 20000 and 'highly_variable' not in adata.var.columns:
        # Subsample for HVG calculation only
        idx = np.random.choice(adata.n_obs, 20000, replace=False)
        temp_adata = adata[idx].copy()
        sc.pp.highly_variable_genes(temp_adata, n_top_genes=n_top_genes, subset=False, flavor='seurat')
        # Transfer results back
        adata.var['highly_variable'] = False
        adata.var.loc[temp_adata.var_names, 'highly_variable'] = temp_adata.var['highly_variable']
        del temp_adata
        gc.collect()
    elif 'highly_variable' not in adata.var.columns:
        sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes, subset=False, flavor='seurat')

    # Get HVG Subset (Sparse View or Copy)
    # Scanpy handles views efficiently for PCA
    adata_hvg = adata[:, adata.var['highly_variable']]
    print(f"HVG subset shape: {adata_hvg.shape}")

    # 2. PCA (Scanpy Arpack = Sparse SVD on centered data implicitly)
    # This is much more stable than manual IncrementalPCA on disjoint chunks
    print("Computing PCA (Arpack - Sparse)...")
    sc.pp.pca(adata_hvg, n_comps=30, svd_solver='arpack', zero_center=True, use_highly_variable=False)
    X_pca = adata_hvg.obsm['X_pca']
    
    # 3. TruncatedSVD (LSA - No centering)
    # Good for sparse data comparison
    print("Computing TruncatedSVD...")
    # Use the sparse matrix from the view
    svd = TruncatedSVD(n_components=30, random_state=42, algorithm='randomized')
    X_svd = svd.fit_transform(adata_hvg.X)
    
    # 4. UMAP on PCA (Standard practice)
    print("Computing UMAP on PCA embeddings...")
    umap_reducer = umap.UMAP(
        n_components=10, 
        n_neighbors=15, 
        random_state=42, 
        n_jobs=1, 
        low_memory=True
    )
    X_umap = umap_reducer.fit_transform(X_pca)

    encodings = {
        'pca': X_pca.astype(np.float32),
        'svd': X_svd.astype(np.float32),
        'umap': X_umap.astype(np.float32)
    }
    
    # Clean up
    del adata_hvg
    gc.collect()
    
    # Return nothing for X_scaled (deprecated)
    return encodings, None

# --- Main Preprocessing Block ---

print("Preprocessing gene expression data...")
import scipy.sparse as sp

# 1. Ensure float32 sparse (Crucial for memory)
if not hasattr(adata.X, 'toarray'): # is sparse
    if adata.X.dtype != np.float32:
        print("Converting sparse matrix to float32...")
        adata.X = adata.X.astype(np.float32)
else: # is dense (shouldn't be, but just in case)
    print("Warning: Data is dense. converting to sparse float32...")
    adata.X = sp.csr_matrix(adata.X, dtype=np.float32)

gc.collect()

# 2. Normalize & Log (In-place)
print("Normalizing...")
sc.pp.normalize_total(adata, target_sum=1e4)
print("Log transforming...")
sc.pp.log1p(adata)
gc.collect()

# 3. Clean Infs/NaNs (In-place, memory safe)
if hasattr(adata.X, 'data'):
    mask = np.isinf(adata.X.data)
    if mask.any():
        adata.X.data[mask] = 0
        print(f"Fixed {mask.sum()} infinite values")
    mask = np.isnan(adata.X.data)
    if mask.any():
        adata.X.data[mask] = 0
        print(f"Fixed {mask.sum()} NaN values")

print("Encoding patterns...")

# Apply encoding
try:
    # Use reduced gene set (1500)
    result = encode_gene_expression_patterns(adata, n_top_genes=1500)
    gene_encodings = result[0]

    # Add to AnnData
    for key, val in gene_encodings.items():
        adata.obsm[f'X_gene_{key}'] = val
        print(f"  Added X_gene_{key}")

    del gene_encodings
    gc.collect()
    print("Gene expression encoding completed!")
    
except Exception as e:
    print(f"Error during encoding: {e}")
    import traceback
    traceback.print_exc()
    raise e

Preprocessing gene expression data...
Normalizing...
Log transforming...
Encoding patterns...
Selecting Highly Variable Genes...
HVG subset shape: (100067, 1500)
Computing PCA (Arpack - Sparse)...
Computing TruncatedSVD...
Computing UMAP on PCA embeddings...
  Added X_gene_pca
  Added X_gene_svd
  Added X_gene_umap
Gene expression encoding completed!
CPU times: user 4min 31s, sys: 2.08 s, total: 4min 33s
Wall time: 3min 57s


In [30]:
%%time
# --- Create Combined Multi-Modal Encodings ---
print("Creating combined multi-modal encodings...")
import gc
from scipy import sparse
from sklearn.decomposition import TruncatedSVD

# Combine different encoding modalities
# 1. Gene expression PCA + TCR physicochemical features
gene_pca = None
# Retrieve pre-computed PCA from gene_encodings dict
pca_data = gene_encodings.get('pca', None) if 'gene_encodings' in globals() and isinstance(gene_encodings, dict) else None

if pca_data is not None and isinstance(pca_data, (np.ndarray, list)):
    gene_pca = np.asarray(pca_data)
elif 'X_gene_pca' in adata.obsm:
    gene_pca = adata.obsm['X_gene_pca']

# Fallback or pad
if gene_pca is not None:
    if gene_pca.ndim == 1:
        gene_pca = gene_pca.reshape(-1, 1)
    if gene_pca.shape[1] >= 20:
        gene_pca = gene_pca[:, :20]
    else:
        # Pad to 20 components
        pad_cols = 20 - gene_pca.shape[1]
        gene_pca = np.pad(gene_pca, ((0, 0), (0, pad_cols)), mode='constant')
else:
    print("Warning: PCA data not available; using zeros.")
    gene_pca = np.zeros((adata.n_obs, 20))

tcr_features = np.column_stack([
    adata.obs[['tra_length', 'tra_molecular_weight', 'tra_hydrophobicity']].fillna(0).values,
    adata.obs[['trb_length', 'trb_molecular_weight', 'trb_hydrophobicity']].fillna(0).values
])

combined_gene_tcr = np.column_stack([gene_pca, tcr_features])
adata.obsm['X_combined_gene_tcr'] = combined_gene_tcr.astype(np.float32)

# 2. Gene expression UMAP + TCR k-mer features (reduced)
gene_umap = gene_encodings.get('umap', None) if 'gene_encodings' in globals() and isinstance(gene_encodings, dict) else None
if gene_umap is None and 'X_gene_umap' in adata.obsm:
    gene_umap = adata.obsm['X_gene_umap']
if gene_umap is None:
    gene_umap = np.zeros((adata.n_obs, 2))

# Stack TRA and TRB k-mer matrices EFFICIENTLY
tra_kmer = adata.obsm.get('X_tcr_tra_kmer', None)
trb_kmer = adata.obsm.get('X_tcr_trb_kmer', None)

if tra_kmer is not None and trb_kmer is not None:
    if sparse.issparse(tra_kmer) or sparse.issparse(trb_kmer):
        # Ensure both are sparse before stacking to avoid densification
        if not sparse.issparse(tra_kmer): tra_kmer = sparse.csr_matrix(tra_kmer)
        if not sparse.issparse(trb_kmer): trb_kmer = sparse.csr_matrix(trb_kmer)
        tcr_kmer_combined = sparse.hstack([tra_kmer, trb_kmer])
    else:
        tcr_kmer_combined = np.column_stack([tra_kmer, trb_kmer])
else:
    tcr_kmer_combined = np.zeros((adata.n_obs, 1)) # Dummy

# Robust dimensional reduction for k-mer features using TruncatedSVD (Safe for OOM)
print(f"Reducing k-mer features: {tcr_kmer_combined.shape}")
n_comp_kmer = min(10, tcr_kmer_combined.shape[1] - 1)
if n_comp_kmer > 0:
    tcr_svd = TruncatedSVD(n_components=n_comp_kmer, random_state=42, algorithm='randomized')
    tcr_kmer_reduced = tcr_svd.fit_transform(tcr_kmer_combined)
else:
    tcr_kmer_reduced = np.zeros((adata.n_obs, 10))

combined_gene_tcr_kmer = np.column_stack([gene_umap, tcr_kmer_reduced])
adata.obsm['X_combined_gene_tcr_kmer'] = combined_gene_tcr_kmer.astype(np.float32)

print(f"Combined gene-TCR encoding shape: {combined_gene_tcr.shape}")
print(f"Combined gene-TCR k-mer encoding shape: {combined_gene_tcr_kmer.shape}")

# Clear memory
del tcr_kmer_combined
gc.collect()

# --- Dimensionality Reduction on Combined Data ---
print("Computing dimensionality reduction on combined data (UMAP only)...")

# UMAP on combined data
umap_combined = umap.UMAP(n_components=2, random_state=42, n_jobs=1) # Single Job for RAM safety
adata.obsm['X_umap_combined'] = umap_combined.fit_transform(combined_gene_tcr)

Creating combined multi-modal encodings...
Reducing k-mer features: (100067, 2)
Combined gene-TCR encoding shape: (100067, 26)
Combined gene-TCR k-mer encoding shape: (100067, 11)
Computing dimensionality reduction on combined data (UMAP only)...
CPU times: user 3min 4s, sys: 269 ms, total: 3min 5s
Wall time: 2min 3s


## 10. Multi-Modal Feature Combination

We combine the encoded gene expression features (PCA/UMAP) with the TCR features (physicochemical properties and k-mers) to create unified multi-modal representations for the learning algorithms.


## 11. Unsupervised Clustering with Leiden Algorithm

To identify distinct cell populations within the immune landscape, we employ the Leiden algorithm. This community detection method improves upon the Louvain algorithm by guaranteeing well-connected communities and is the standard for single-cell RNA-seq analysis. We perform clustering at multiple resolutions to capture both broad cell types and fine-grained states.


In [31]:
# HDBSCAN/sklearn compatibility patch â€” run before clustering
import sys, subprocess, inspect

# Ensure hdbscan is available (not strictly necessary if already installed earlier)
try:
    import hdbscan
except Exception:
    print("hdbscan not installed â€” installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "hdbscan"]) 
    import importlib
    importlib.invalidate_caches()
    import hdbscan

# Patch the check_array reference used inside hdbscan to accept the older keyword
try:
    import sklearn.utils.validation as sk_validation
    from hdbscan import hdbscan_ as _hdbscan_mod
    sig = inspect.signature(sk_validation.check_array)
    if 'ensure_all_finite' in sig.parameters and 'force_all_finite' not in sig.parameters:
        orig = getattr(_hdbscan_mod, 'check_array', None) or sk_validation.check_array
        def _patched_check_array(*args, **kwargs):
            if 'force_all_finite' in kwargs and 'ensure_all_finite' not in kwargs:
                kwargs['ensure_all_finite'] = kwargs.pop('force_all_finite')
            return orig(*args, **kwargs)
        _hdbscan_mod.check_array = _patched_check_array
        print("Patched hdbscan.check_array to accept 'force_all_finite' for this runtime.")
    else:
        print("No patch required for sklearn.check_array signature.")
except Exception as e:
    print("Compatibility patch could not be applied:", type(e).__name__, e)


No patch required for sklearn.check_array signature.


## Unsupervised Machine Learning Analysis (Updated)

This section has been updated to utilize the `clustering.py` implementation for Leiden clustering, replacing the previous K-Means/DBSCAN/Agglomerative comparison.

**Changes:**
- Imported `clustering.py` module.
- Used `clustering.preprocess_data(adata)` for data preprocessing.
- Used `clustering.perform_clustering(adata)` for Leiden clustering at multiple resolutions.
- Calculated silhouette scores for Leiden clusters to maintain compatibility with the "best clustering" selection logic.
- Renamed Leiden cluster columns to `leiden_cluster_{resolution}` to ensure compatibility with downstream feature selection filters.
- Retained TCR sequence-specific clustering and Gene Expression Module Discovery.

**Note:**
- Ensure `clustering.py` is in the python path (Code/ directory).
- The "best clustering" is now selected from the Leiden results based on silhouette score.

In [ ]:
%%time
%pip install scipy
%pip install leidenalg

# Check if we should skip unsupervised learning
if globals().get('SKIP_UNSUPERVISED_LEARNING', False) or globals().get('SKIP_TO_DEEP_LEARNING', False):
    print("SKIPPING: Unsupervised Learning (Leiden, UMAP, etc.)")
    print("Set SKIP_UNSUPERVISED_LEARNING=False in the configuration cell to run this section.")
    
    # Still need to do minimal preprocessing for deep learning
    import scanpy as sc
    import numpy as np
    import pandas as pd
    import gc
    from scipy import sparse
    from sklearn.decomposition import TruncatedSVD
    from sklearn.preprocessing import LabelEncoder
    
    print("Running MINIMAL preprocessing for deep learning...")
    
    # Normalize and log-transform
    if 'log1p' not in adata.uns:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
    
    # Find HVGs
    if 'highly_variable' not in adata.var.columns:
        sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
    
    # Compute PCA if needed
    if 'X_pca' not in adata.obsm:
        hvg_mask = adata.var['highly_variable'].values
        n_hvgs = hvg_mask.sum()
        if sparse.issparse(adata.X):
            X_hvg = adata.X[:, hvg_mask]
            n_components = min(50, n_hvgs - 1, X_hvg.shape[0] - 1)
            svd = TruncatedSVD(n_components=n_components, random_state=42, algorithm='arpack')
            adata.obsm['X_pca'] = svd.fit_transform(X_hvg).astype(np.float32)
            del X_hvg, svd
            gc.collect()
    
    # Fix column names (response, patient_id) - ROBUST VERSION
    if 'response' not in adata.obs.columns and 'Response' in adata.obs.columns:
        adata.obs['response'] = adata.obs['Response']
    
    # First check if patient_id already exists
    patient_id_found = False
    for col_name in ['patient_id', 'Patient_ID', 'PatientID']:
        if col_name in adata.obs.columns:
            if col_name != 'patient_id':
                adata.obs['patient_id'] = adata.obs[col_name]
            patient_id_found = True
            print(f"  Found patient_id in column '{col_name}'")
            break
    
    # If patient_id not found, derive from sample_id using metadata mapping
    if not patient_id_found and 'sample_id' in adata.obs.columns:
        print("  patient_id not found directly. Deriving from sample_id...")
        
        # Recreate metadata_df mapping (same as in data loading cell)
        # This is the authoritative mapping from GEO GSE300475
        metadata_records = [
            {'sample_id': 'GSM9061665', 'Patient_ID': 'PT1', 'Timepoint': 'Pre', 'Response': 'Responder'},
            {'sample_id': 'GSM9061666', 'Patient_ID': 'PT1', 'Timepoint': 'D21', 'Response': 'Responder'},
            {'sample_id': 'GSM9061667', 'Patient_ID': 'PT1', 'Timepoint': 'D42', 'Response': 'Responder'},
            {'sample_id': 'GSM9061668', 'Patient_ID': 'PT2', 'Timepoint': 'Pre', 'Response': 'Responder'},
            {'sample_id': 'GSM9061669', 'Patient_ID': 'PT2', 'Timepoint': 'D21', 'Response': 'Responder'},
            {'sample_id': 'GSM9061670', 'Patient_ID': 'PT2', 'Timepoint': 'D42', 'Response': 'Responder'},
            {'sample_id': 'GSM9061671', 'Patient_ID': 'PT3', 'Timepoint': 'Pre', 'Response': 'Non-Responder'},
            {'sample_id': 'GSM9061672', 'Patient_ID': 'PT3', 'Timepoint': 'D21', 'Response': 'Non-Responder'},
            {'sample_id': 'GSM9061673', 'Patient_ID': 'PT4', 'Timepoint': 'Pre', 'Response': 'Non-Responder'},
            {'sample_id': 'GSM9061674', 'Patient_ID': 'PT4', 'Timepoint': 'D21', 'Response': 'Non-Responder'},
            {'sample_id': 'GSM9061675', 'Patient_ID': 'PT4', 'Timepoint': 'D42', 'Response': 'Non-Responder'},
        ]
        metadata_df_local = pd.DataFrame(metadata_records)
        
        # Create sample_id to patient_id mapping
        sample_to_patient = dict(zip(metadata_df_local['sample_id'], metadata_df_local['Patient_ID']))
        
        # Map sample_id to patient_id
        adata.obs['patient_id'] = adata.obs['sample_id'].map(sample_to_patient)
        
        # Check for unmapped values
        n_mapped = adata.obs['patient_id'].notna().sum()
        n_total = len(adata.obs)
        print(f"  Mapped {n_mapped}/{n_total} cells to patient_id")
        
        if n_mapped == 0:
            # Try parsing sample_id - maybe format is different (e.g., batch column)
            print("  Warning: No direct matches. Checking for batch column...")
            if 'batch' in adata.obs.columns:
                adata.obs['patient_id'] = adata.obs['batch'].map(sample_to_patient)
                n_mapped = adata.obs['patient_id'].notna().sum()
                print(f"  Mapped {n_mapped}/{n_total} cells using batch column")
        
        # Also derive response if missing
        if 'response' not in adata.obs.columns or adata.obs['response'].isna().all():
            sample_to_response = dict(zip(metadata_df_local['sample_id'], metadata_df_local['Response']))
            adata.obs['response'] = adata.obs['sample_id'].map(sample_to_response)
            if adata.obs['response'].isna().all() and 'batch' in adata.obs.columns:
                adata.obs['response'] = adata.obs['batch'].map(sample_to_response)
            print(f"  Derived response column: {adata.obs['response'].value_counts().to_dict()}")
        
        patient_id_found = adata.obs['patient_id'].notna().any()
    
    if not patient_id_found:
        print("  WARNING: Could not derive patient_id. Downstream patient-level analysis may fail.")
        print(f"  Available columns: {list(adata.obs.columns)}")
    else:
        print(f"  patient_id distribution: {adata.obs['patient_id'].value_counts().to_dict()}")
    
    # Create supervised_mask for downstream
    if 'response' in adata.obs.columns:
        supervised_mask = adata.obs['response'].isin(['Responder', 'Non-Responder']).values
    else:
        supervised_mask = np.ones(adata.n_obs, dtype=bool)
    
    print(f"Minimal preprocessing complete. supervised_mask: {supervised_mask.sum()} samples")
    
else:
    # --- Full Unsupervised Machine Learning Analysis ---
    print("Applying unsupervised machine learning algorithms...")

    import scanpy as sc
    import numpy as np
    import pandas as pd
    import gc  # For garbage collection
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from scipy.cluster.hierarchy import dendrogram, linkage
    from scipy import sparse
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from pathlib import Path
    from IPython.display import display
    
    # Ensure output directory exists
    if IS_KAGGLE:
        Path('/kaggle/working/Processed_Data').mkdir(parents=True, exist_ok=True)
    else:
        Path('Processed_Data').mkdir(parents=True, exist_ok=True)

    # Quick memory cleanup
    for _v in ['adata_list','adata_sample','metadata_list','metadata_df',
               'tra_kmer_sparse','trb_kmer_sparse','tra_kmer_matrix','trb_kmer_matrix',
               'vec_tra','vec_trb','tra_seqs','trb_seqs','tra_kmeans','trb_kmeans',
               'tra_kmer_scaled','trb_kmer_scaled','tra_scaler','trb_scaler','gene_kmeans',
               'gene_pca','gene_expression_modules','tra_clusters','trb_clusters']:
        if _v in globals():
            try:
                del globals()[_v]
            except Exception:
                pass
    gc.collect()

    np.random.seed(42)

    # Fix column names (response, patient_id) BEFORE processing
    if 'response' not in adata.obs.columns and 'Response' in adata.obs.columns:
        adata.obs['response'] = adata.obs['Response']
        print("  Renamed 'Response' column to 'response'")
    
    for col_name in ['Patient_ID', 'PatientID']:
        if col_name in adata.obs.columns and 'patient_id' not in adata.obs.columns:
            adata.obs['patient_id'] = adata.obs[col_name]
            print(f"  Renamed '{col_name}' column to 'patient_id'")
            break

    print("Preprocessing data (memory-efficient mode)...")

    # 1. Normalize and log-transform (these keep sparse)
    if 'log1p' not in adata.uns:
        print("  Normalizing...")
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        gc.collect()

    # 2. Find highly variable genes (does NOT densify)
    if 'highly_variable' not in adata.var.columns:
        print("  Finding highly variable genes...")
        sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
        gc.collect()

    # 3. MEMORY-EFFICIENT PCA using TruncatedSVD on sparse HVG subset
    if 'X_pca' not in adata.obsm:
        print("Computing PCA (sparse-friendly via TruncatedSVD on HVGs)...")
        
        from sklearn.decomposition import TruncatedSVD
        
        hvg_mask = adata.var['highly_variable'].values
        n_hvgs = hvg_mask.sum()
        print(f"  Using {n_hvgs} highly variable genes")
        
        if sparse.issparse(adata.X):
            X_hvg = adata.X[:, hvg_mask]
            print(f"  HVG matrix shape: {X_hvg.shape}, sparse: {sparse.issparse(X_hvg)}")
            
            n_components = min(50, n_hvgs - 1, X_hvg.shape[0] - 1)
            print(f"  Running TruncatedSVD with {n_components} components...")
            
            svd = TruncatedSVD(n_components=n_components, random_state=42, algorithm='arpack')
            X_pca = svd.fit_transform(X_hvg)
            
            adata.obsm['X_pca'] = X_pca.astype(np.float32)
            adata.uns['pca'] = {
                'variance_ratio': svd.explained_variance_ratio_,
                'variance': svd.explained_variance_,
            }
            loadings = np.zeros((adata.n_vars, n_components), dtype=np.float32)
            loadings[hvg_mask, :] = svd.components_.T.astype(np.float32)
            adata.varm['PCs'] = loadings
            
            del X_hvg, svd, X_pca, loadings
            gc.collect()
            print(f"  PCA complete. Variance explained: {adata.uns['pca']['variance_ratio'].sum():.2%}")
            
        else:
            print("  Data is dense, scaling HVGs only...")
            X_hvg = adata.X[:, hvg_mask].copy()
            scaler = StandardScaler(with_mean=True, with_std=True)
            X_hvg_scaled = scaler.fit_transform(X_hvg)
            del X_hvg
            gc.collect()
            
            from sklearn.decomposition import PCA
            n_components = min(50, n_hvgs - 1, X_hvg_scaled.shape[0] - 1)
            pca = PCA(n_components=n_components, random_state=42)
            X_pca = pca.fit_transform(X_hvg_scaled)
            
            adata.obsm['X_pca'] = X_pca.astype(np.float32)
            adata.uns['pca'] = {
                'variance_ratio': pca.explained_variance_ratio_,
                'variance': pca.explained_variance_,
            }
            loadings = np.zeros((adata.n_vars, n_components), dtype=np.float32)
            loadings[hvg_mask, :] = pca.components_.T.astype(np.float32)
            adata.varm['PCs'] = loadings
            
            del X_hvg_scaled, pca, X_pca, loadings, scaler
            gc.collect()
        
        # Memory cleanup after PCA
        try:
            if getattr(adata, 'raw', None) is not None:
                del adata.raw
        except Exception:
            pass
        try:
            if hasattr(adata, 'layers') and len(adata.layers) > 0:
                adata.layers.clear()
        except Exception:
            pass
        gc.collect()
        print("  Memory cleanup after PCA complete.")

    # Neighbors
    print("Computing neighbors...")
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, random_state=42)
    gc.collect()

    # 2. Perform Clustering (Leiden) - Use fewer resolutions for speed
    print("Performing Leiden clustering...")
    resolutions = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]  # Reduced from 26 to 6
    best_res = 0.1
    target_clusters = 7
    best_diff = float('inf')

    for res in resolutions:
        key = f'leiden_{res}'
        try:
            sc.tl.leiden(adata, resolution=res, key_added=key, random_state=42)
            n_clust = len(adata.obs[key].unique())
            print(f"Resolution {res}: {n_clust} clusters")
            if abs(n_clust - target_clusters) < best_diff:
                best_diff = abs(n_clust - target_clusters)
                best_res = res
        except Exception as e:
            print(f"Leiden failed for resolution {res}: {e}")
        gc.collect()

    print(f"Selected resolution: {best_res}")
    if f'leiden_{best_res}' in adata.obs:
        adata.obs['leiden'] = adata.obs[f'leiden_{best_res}']

    # 3. TCR Sequence Clustering
    print("Performing TCR sequence-specific clustering...")
    if 'X_tcr_tra_kmer' in adata.obsm:
        tra_scaler = StandardScaler()
        tra_kmer_scaled = tra_scaler.fit_transform(adata.obsm['X_tcr_tra_kmer'])
        tra_kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
        adata.obs['tra_kmer_clusters'] = pd.Categorical(tra_kmeans.fit_predict(tra_kmer_scaled))
        del tra_kmer_scaled, tra_kmeans, tra_scaler
        gc.collect()

    if 'X_tcr_trb_kmer' in adata.obsm:
        trb_scaler = StandardScaler()
        trb_kmer_scaled = trb_scaler.fit_transform(adata.obsm['X_tcr_trb_kmer'])
        trb_kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
        adata.obs['trb_kmer_clusters'] = pd.Categorical(trb_kmeans.fit_predict(trb_kmer_scaled))
        del trb_kmer_scaled, trb_kmeans, trb_scaler
        gc.collect()

    # 4. Gene Expression Module Discovery
    print("Discovering gene expression modules...")
    gene_pca = adata.obsm.get('X_gene_pca', adata.obsm.get('X_pca'))
    if gene_pca is not None:
        gene_kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
        adata.obs['gene_expression_modules'] = pd.Categorical(gene_kmeans.fit_predict(gene_pca))
        del gene_pca, gene_kmeans
        gc.collect()

    # 5. Visualization
    print("Creating visualizations...")
    sc.tl.umap(adata, random_state=42)
    
    color_keys = []
    if 'leiden' in adata.obs:
        color_keys.append('leiden')
    if 'response' in adata.obs.columns:
        color_keys.append('response')
    
    if color_keys:
        sc.pl.umap(adata, color=color_keys, show=False)
        plt.show()

    # --- Create supervised_mask for downstream cells ---
    if 'response' in adata.obs.columns:
        supervised_mask = adata.obs['response'].isin(['Responder', 'Non-Responder']).values
        print(f"\nCreated supervised_mask: {supervised_mask.sum()} samples with valid response labels")
    else:
        supervised_mask = np.ones(adata.n_obs, dtype=bool)
        print("\nWarning: No response column found. supervised_mask includes all cells.")

    print("Unsupervised machine learning analysis completed!")

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 41.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Applying unsupervised machine learning algorithms...
Preprocessing data (memory-efficient mode)...
Computing PCA (sparse-friendly via TruncatedSVD on HVGs)...
  Using 1500 highly variable genes
  HVG matrix shape: (100067, 1500), sparse: True
  Running TruncatedSVD with 50 components...
  PCA complete. Variance explained: 70.37%
  Memory cleanup after PCA complete.
Computing neighbors...
Performing Leiden clustering...
Resolution 0.01: 4 clusters
Resolution 0.05: 6 clusters
Resolution 0.1: 7 clusters
Resolution 0.2: 11 clusters
Resolution 0.5: 14 clusters
Resolution 1.0: 23 clusters
Selected resolution: 0.1
Performing TCR sequence-specific clustering...
Discovering gene expression modules...
Creating visualizations...

Unsupervised machine learning analysis completed!
CPU times: u

In [33]:
# --- Memory cleanup (after Leiden clustering, before dendrogram) ---
# This frees large temporary matrices (one-hot encodings, neighbor/connectivity matrices)
# while keeping UMAP for dendrogram/visualization.
print('\nRunning memory cleanup after Leiden clustering (before dendrogram)...')
try:
    import psutil, os
    proc = psutil.Process(os.getpid())
    print(f"Memory before cleanup: {proc.memory_info().rss // (1024**2)} MB")
except Exception:
    print('psutil not available; skipping memory before measurement')

def _fallback_cleanup(drop_onehot=False, drop_raw=False, drop_obsm_umap_tsne=False, verbose=True):
    """Basic cleanup fallback when cleanup_after_clustering is unavailable."""
    if 'adata' not in globals():
        return
    if hasattr(adata, 'obsp'):
        for _k in list(adata.obsp.keys()):
            try:
                del adata.obsp[_k]
            except Exception:
                pass
    if drop_onehot and hasattr(adata, 'obsm'):
        for _key in ['X_tcr_tra_onehot', 'X_tcr_trb_onehot']:
            if _key in adata.obsm:
                try:
                    del adata.obsm[_key]
                except Exception:
                    pass
    if drop_obsm_umap_tsne and hasattr(adata, 'obsm'):
        for _key in list(adata.obsm.keys()):
            _lk = _key.lower()
            if 'umap' in _lk or 'tsne' in _lk:
                try:
                    del adata.obsm[_key]
                except Exception:
                    pass
    if drop_raw and getattr(adata, 'raw', None) is not None:
        adata.raw = None
    if verbose:
        print('Fallback cleanup completed.')

# Conservative cleanup: drop TCR one-hot arrays and obsp connectivities/distances
# Keep one-hot encodings by default to avoid KeyError in downstream feature engineering
if 'cleanup_after_clustering' in globals():
    try:
        cleanup_after_clustering(drop_onehot=False, drop_raw=False, drop_obsm_umap_tsne=False, verbose=True)
    except Exception as e:
        print('cleanup_after_clustering failed, using fallback cleanup:', e)
        _fallback_cleanup(drop_onehot=False, drop_raw=False, drop_obsm_umap_tsne=False, verbose=True)
else:
    _fallback_cleanup(drop_onehot=False, drop_raw=False, drop_obsm_umap_tsne=False, verbose=True)

try:
    proc = psutil.Process(os.getpid())
    print(f"Memory after cleanup: {proc.memory_info().rss // (1024**2)} MB")
except Exception:
    pass

import gc
gc.collect()



Running memory cleanup after Leiden clustering (before dendrogram)...
Memory before cleanup: 4409 MB
Fallback cleanup completed.
Memory after cleanup: 4409 MB


9

In [34]:
# --- Label/patient derivation and supervised availability helpers ---
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
if '_ensure_int_labels' not in globals():
    def _ensure_int_labels(y):
        y_arr = np.asarray(y)
        if np.issubdtype(y_arr.dtype, np.integer):
            return y_arr
        if y_arr.size == 0:
            return y_arr.astype(np.int64)
        if np.all(np.isfinite(y_arr)) and np.all(np.equal(y_arr, np.floor(y_arr))):
            return y_arr.astype(np.int64)
        raise ValueError("Labels must be integer or integer-like floats.")
if '_normalize_response_value' not in globals():
    def _normalize_response_value(val):
        if pd.isna(val):
            return 'Unknown'
        s = str(val).strip().lower()
        if s == '':
            return 'Unknown'
        if 'non' in s and 'responder' in s:
            return 'Non-Responder'
        if 'responder' in s:
            return 'Responder'
        return 'Unknown'
if '_ensure_response_and_patient' not in globals():
    def _ensure_response_and_patient(adata):
        # Normalize existing columns if present
        if 'response' not in adata.obs.columns and 'Response' in adata.obs.columns:
            adata.obs['response'] = adata.obs['Response']
        if 'patient_id' not in adata.obs.columns:
            for _col in ['Patient_ID', 'PatientID']:
                if _col in adata.obs.columns:
                    adata.obs['patient_id'] = adata.obs[_col]
                    break
        # Determine metadata mapping
        md = None
        if 'metadata_df' in globals() and isinstance(metadata_df, pd.DataFrame) and not metadata_df.empty:
            md = metadata_df.copy()
        else:
            # Fallback to hard-coded metadata list (same as in Cell 15)
            _metadata_list = [
                {'S_Number': 'S1',  'GEX_Sample_ID': 'GSM9061665', 'TCR_Sample_ID': 'GSM9061687', 'Patient_ID': 'PT1',  'Timepoint': 'Baseline',   'Response': 'Responder',     'In_Data': 'Yes'},
                {'S_Number': 'S2',  'GEX_Sample_ID': 'GSM9061666', 'TCR_Sample_ID': 'GSM9061688', 'Patient_ID': 'PT1',  'Timepoint': 'Post-Tx',    'Response': 'Responder',     'In_Data': 'Yes'},
                {'S_Number': 'S3',  'GEX_Sample_ID': 'GSM9061667', 'TCR_Sample_ID': 'GSM9061689', 'Patient_ID': 'PT1',  'Timepoint': 'Recurrence', 'Response': 'Responder',     'In_Data': 'Yes'},
                {'S_Number': 'S4',  'GEX_Sample_ID': 'GSM9061668', 'TCR_Sample_ID': 'GSM9061690', 'Patient_ID': 'PT2',  'Timepoint': 'Baseline',   'Response': 'Responder',     'In_Data': 'Yes'},
                {'S_Number': 'S5',  'GEX_Sample_ID': 'GSM9061669', 'TCR_Sample_ID': 'GSM9061691', 'Patient_ID': 'PT2',  'Timepoint': 'Post-Tx',    'Response': 'Responder',     'In_Data': 'Yes'},
                {'S_Number': 'S6',  'GEX_Sample_ID': 'GSM9061670', 'TCR_Sample_ID': 'GSM9061692', 'Patient_ID': 'PT3',  'Timepoint': 'Baseline',   'Response': 'Non-Responder', 'In_Data': 'Yes'},
                {'S_Number': 'S7',  'GEX_Sample_ID': 'GSM9061671', 'TCR_Sample_ID': 'GSM9061693', 'Patient_ID': 'PT3',  'Timepoint': 'Post-Tx',    'Response': 'Non-Responder', 'In_Data': 'Yes'},
                {'S_Number': 'S8',  'GEX_Sample_ID': 'GSM9061672', 'TCR_Sample_ID': None,         'Patient_ID': 'PT3',  'Timepoint': 'Recurrence', 'Response': 'Non-Responder', 'In_Data': 'GEX only'},
                {'S_Number': 'S9',  'GEX_Sample_ID': 'GSM9061673', 'TCR_Sample_ID': 'GSM9061694', 'Patient_ID': 'PT4',  'Timepoint': 'Baseline',   'Response': 'Non-Responder', 'In_Data': 'Yes'},
                {'S_Number': 'S10', 'GEX_Sample_ID': 'GSM9061674', 'TCR_Sample_ID': 'GSM9061695', 'Patient_ID': 'PT4',  'Timepoint': 'Post-Tx',    'Response': 'Non-Responder', 'In_Data': 'Yes'},
                {'S_Number': 'S11', 'GEX_Sample_ID': 'GSM9061675', 'TCR_Sample_ID': 'GSM9061696', 'Patient_ID': 'PT4',  'Timepoint': 'Recurrence', 'Response': 'Non-Responder', 'In_Data': 'Yes'},
            ]
            md = pd.DataFrame(_metadata_list)
        # Identify mapping columns in metadata
        sample_col = None
        for _c in ['sample_id', 'GEX_Sample_ID', 'GSM_ID', 'GEO_ID', 'Sample_ID']:
            if _c in md.columns:
                sample_col = _c
                break
        patient_col = None
        for _c in ['patient_id', 'Patient_ID', 'PatientID']:
            if _c in md.columns:
                patient_col = _c
                break
        response_col = None
        for _c in ['response', 'Response']:
            if _c in md.columns:
                response_col = _c
                break
        # Determine sample ID series from adata
        sample_series = None
        for _c in ['sample_id', 'batch']:
            if _c in adata.obs.columns:
                sample_series = adata.obs[_c].astype(str)
                break
        if md is not None and sample_series is not None and sample_col is not None:
            sample_key = sample_series.str.split('_').str[0]
            md_sample = md[sample_col].astype(str)
            if patient_col is not None:
                patient_map = dict(zip(md_sample, md[patient_col]))
                if 'patient_id' not in adata.obs.columns:
                    adata.obs['patient_id'] = sample_key.map(patient_map)
                else:
                    adata.obs['patient_id'] = adata.obs['patient_id'].where(adata.obs['patient_id'].notna(), sample_key.map(patient_map))
            if response_col is not None:
                resp_map = dict(zip(md_sample, md[response_col]))
                if 'response' not in adata.obs.columns:
                    adata.obs['response'] = sample_key.map(resp_map)
                else:
                    adata.obs['response'] = adata.obs['response'].where(adata.obs['response'].notna(), sample_key.map(resp_map))
        # Normalize response labels
        if 'response' in adata.obs.columns:
            adata.obs['response'] = adata.obs['response'].apply(_normalize_response_value)
        # Coverage reporting
        if 'response' in adata.obs.columns:
            resp_counts = adata.obs['response'].value_counts(dropna=False).to_dict()
            print(f"Response distribution: {resp_counts}")
        if 'patient_id' in adata.obs.columns:
            mapped = adata.obs['patient_id'].notna().sum()
            print(f"Patient_id coverage: {mapped}/{len(adata.obs)}")
if '_get_supervised_mask_and_labels' not in globals():
    def _get_supervised_mask_and_labels(adata):
        if 'response' not in adata.obs.columns:
            print("WARNING: response column missing. No supervised labels available.")
            supervised_mask = np.zeros(adata.n_obs, dtype=bool)
            return supervised_mask, pd.Series([], dtype=object), None, {}, False
        y_all = adata.obs['response'].astype(str)
        supervised_mask = y_all.isin(['Responder', 'Non-Responder']).values
        y_supervised = y_all[supervised_mask]
        if len(y_supervised) == 0:
            print("WARNING: No labeled samples found for supervised learning.")
            return supervised_mask, y_supervised, None, {}, False
        class_counts = y_supervised.value_counts().to_dict()
        if len(class_counts) < 2 or min(class_counts.values()) < 2:
            print(f"WARNING: Insufficient class balance for supervised learning: {class_counts}")
            return supervised_mask, y_supervised, None, class_counts, False
        le = LabelEncoder()
        return supervised_mask, y_supervised, le, class_counts, True
# Ensure response/patient labels are present and normalized
if 'adata' not in globals() or adata is None:
    raise NameError("adata is not defined. Please run the data loading cells first.")
_ensure_response_and_patient(adata)
supervised_mask, y_supervised, label_encoder, class_counts, SUPERVISED_AVAILABLE = _get_supervised_mask_and_labels(adata)
globals()['SUPERVISED_AVAILABLE'] = SUPERVISED_AVAILABLE
if SUPERVISED_AVAILABLE:
    y_encoded = label_encoder.fit_transform(y_supervised)
    y_encoded = _ensure_int_labels(y_encoded)
    print(f"Working with {int(supervised_mask.sum())} samples for supervised learning")
    print(f"Class distribution: {class_counts}")
else:
    print("WARNING: Supervised labels not available or insufficient. Skipping supervised-only steps.")
    supervised_mask = np.ones(adata.n_obs, dtype=bool)
    y_supervised = pd.Series(['Unknown'] * adata.n_obs, index=adata.obs.index)
    y_encoded = np.array([], dtype=np.int64)

Response distribution: {'Non-Responder': 63074, 'Responder': 36993}
Patient_id coverage: 100067/100067
Working with 100067 samples for supervised learning
Class distribution: {'Non-Responder': 63074, 'Responder': 36993}


In [35]:
%%time
# --- Comprehensive Feature Engineering ---

print("Creating comprehensive feature set using ALL available encodings...")

# --- 1. Strategic Feature Engineering with Dimensionality Reduction ---
print("Applying strategic dimensionality reduction to high-dimensional features...")

# --- Label/patient derivation and supervised availability helpers ---
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

if '_ensure_int_labels' not in globals():
    def _ensure_int_labels(y):
        y_arr = np.asarray(y)
        if np.issubdtype(y_arr.dtype, np.integer):
            return y_arr
        if y_arr.size == 0:
            return y_arr.astype(np.int64)
        if np.all(np.isfinite(y_arr)) and np.all(np.equal(y_arr, np.floor(y_arr))):
            return y_arr.astype(np.int64)
        raise ValueError("Labels must be integer or integer-like floats.")

if '_normalize_response_value' not in globals():
    def _normalize_response_value(val):
        if pd.isna(val):
            return 'Unknown'
        s = str(val).strip().lower()
        if s == '':
            return 'Unknown'
        if 'non' in s and 'responder' in s:
            return 'Non-Responder'
        if 'responder' in s:
            return 'Responder'
        return 'Unknown'

if '_ensure_response_and_patient' not in globals():
    def _ensure_response_and_patient(adata):
        # Normalize existing columns if present
        if 'response' not in adata.obs.columns and 'Response' in adata.obs.columns:
            adata.obs['response'] = adata.obs['Response']
        if 'patient_id' not in adata.obs.columns:
            for _col in ['Patient_ID', 'PatientID']:
                if _col in adata.obs.columns:
                    adata.obs['patient_id'] = adata.obs[_col]
                    break

        # Determine metadata mapping
        md = None
        if 'metadata_df' in globals() and isinstance(metadata_df, pd.DataFrame) and not metadata_df.empty:
            md = metadata_df.copy()
        else:
            # Fallback to hard-coded metadata list (same as in Cell 15)
            _metadata_list = [
                {'S_Number': 'S1',  'GEX_Sample_ID': 'GSM9061665', 'TCR_Sample_ID': 'GSM9061687', 'Patient_ID': 'PT1',  'Timepoint': 'Baseline',   'Response': 'Responder',     'In_Data': 'Yes'},
                {'S_Number': 'S2',  'GEX_Sample_ID': 'GSM9061666', 'TCR_Sample_ID': 'GSM9061688', 'Patient_ID': 'PT1',  'Timepoint': 'Post-Tx',    'Response': 'Responder',     'In_Data': 'Yes'},
                {'S_Number': 'S3',  'GEX_Sample_ID': 'GSM9061667', 'TCR_Sample_ID': 'GSM9061689', 'Patient_ID': 'PT1',  'Timepoint': 'Recurrence', 'Response': 'Responder',     'In_Data': 'Yes'},
                {'S_Number': 'S4',  'GEX_Sample_ID': 'GSM9061668', 'TCR_Sample_ID': 'GSM9061690', 'Patient_ID': 'PT2',  'Timepoint': 'Baseline',   'Response': 'Responder',     'In_Data': 'Yes'},
                {'S_Number': 'S5',  'GEX_Sample_ID': 'GSM9061669', 'TCR_Sample_ID': 'GSM9061691', 'Patient_ID': 'PT2',  'Timepoint': 'Post-Tx',    'Response': 'Responder',     'In_Data': 'Yes'},
                {'S_Number': 'S6',  'GEX_Sample_ID': 'GSM9061670', 'TCR_Sample_ID': 'GSM9061692', 'Patient_ID': 'PT3',  'Timepoint': 'Baseline',   'Response': 'Non-Responder', 'In_Data': 'Yes'},
                {'S_Number': 'S7',  'GEX_Sample_ID': 'GSM9061671', 'TCR_Sample_ID': 'GSM9061693', 'Patient_ID': 'PT3',  'Timepoint': 'Post-Tx',    'Response': 'Non-Responder', 'In_Data': 'Yes'},
                {'S_Number': 'S8',  'GEX_Sample_ID': 'GSM9061672', 'TCR_Sample_ID': None,         'Patient_ID': 'PT3',  'Timepoint': 'Recurrence', 'Response': 'Non-Responder', 'In_Data': 'GEX only'},
                {'S_Number': 'S9',  'GEX_Sample_ID': 'GSM9061673', 'TCR_Sample_ID': 'GSM9061694', 'Patient_ID': 'PT4',  'Timepoint': 'Baseline',   'Response': 'Non-Responder', 'In_Data': 'Yes'},
                {'S_Number': 'S10', 'GEX_Sample_ID': 'GSM9061674', 'TCR_Sample_ID': 'GSM9061695', 'Patient_ID': 'PT4',  'Timepoint': 'Post-Tx',    'Response': 'Non-Responder', 'In_Data': 'Yes'},
                {'S_Number': 'S11', 'GEX_Sample_ID': 'GSM9061675', 'TCR_Sample_ID': 'GSM9061696', 'Patient_ID': 'PT4',  'Timepoint': 'Recurrence', 'Response': 'Non-Responder', 'In_Data': 'Yes'},
            ]
            md = pd.DataFrame(_metadata_list)

        # Identify mapping columns in metadata
        sample_col = None
        for _c in ['sample_id', 'GEX_Sample_ID', 'GSM_ID', 'GEO_ID', 'Sample_ID']:
            if _c in md.columns:
                sample_col = _c
                break
        patient_col = None
        for _c in ['patient_id', 'Patient_ID', 'PatientID']:
            if _c in md.columns:
                patient_col = _c
                break
        response_col = None
        for _c in ['response', 'Response']:
            if _c in md.columns:
                response_col = _c
                break

        # Determine sample ID series from adata
        sample_series = None
        for _c in ['sample_id', 'batch']:
            if _c in adata.obs.columns:
                sample_series = adata.obs[_c].astype(str)
                break

        if md is not None and sample_series is not None and sample_col is not None:
            sample_key = sample_series.str.split('_').str[0]
            md_sample = md[sample_col].astype(str)

            if patient_col is not None:
                patient_map = dict(zip(md_sample, md[patient_col]))
                if 'patient_id' not in adata.obs.columns:
                    adata.obs['patient_id'] = sample_key.map(patient_map)
                else:
                    adata.obs['patient_id'] = adata.obs['patient_id'].where(adata.obs['patient_id'].notna(), sample_key.map(patient_map))

            if response_col is not None:
                resp_map = dict(zip(md_sample, md[response_col]))
                if 'response' not in adata.obs.columns:
                    adata.obs['response'] = sample_key.map(resp_map)
                else:
                    adata.obs['response'] = adata.obs['response'].where(adata.obs['response'].notna(), sample_key.map(resp_map))

        # Normalize response labels
        if 'response' in adata.obs.columns:
            adata.obs['response'] = adata.obs['response'].apply(_normalize_response_value)

        # Coverage reporting
        if 'response' in adata.obs.columns:
            resp_counts = adata.obs['response'].value_counts(dropna=False).to_dict()
            print(f"Response distribution: {resp_counts}")
        if 'patient_id' in adata.obs.columns:
            mapped = adata.obs['patient_id'].notna().sum()
            print(f"Patient_id coverage: {mapped}/{len(adata.obs)}")

if '_get_supervised_mask_and_labels' not in globals():
    def _get_supervised_mask_and_labels(adata):
        if 'response' not in adata.obs.columns:
            print("WARNING: response column missing. No supervised labels available.")
            supervised_mask = np.zeros(adata.n_obs, dtype=bool)
            return supervised_mask, pd.Series([], dtype=object), None, {}, False
        y_all = adata.obs['response'].astype(str)
        supervised_mask = y_all.isin(['Responder', 'Non-Responder']).values
        y_supervised = y_all[supervised_mask]
        if len(y_supervised) == 0:
            print("WARNING: No labeled samples found for supervised learning.")
            return supervised_mask, y_supervised, None, {}, False
        class_counts = y_supervised.value_counts().to_dict()
        if len(class_counts) < 2 or min(class_counts.values()) < 2:
            print(f"WARNING: Insufficient class balance for supervised learning: {class_counts}")
            return supervised_mask, y_supervised, None, class_counts, False
        le = LabelEncoder()
        return supervised_mask, y_supervised, le, class_counts, True

# Ensure response/patient labels are present and normalized
if 'adata' not in globals() or adata is None:
    raise NameError("adata is not defined. Please run the data loading cells first.")

_ensure_response_and_patient(adata)

supervised_mask, y_supervised, label_encoder, class_counts, SUPERVISED_AVAILABLE = _get_supervised_mask_and_labels(adata)
globals()['SUPERVISED_AVAILABLE'] = SUPERVISED_AVAILABLE

if SUPERVISED_AVAILABLE:
    y_encoded = label_encoder.fit_transform(y_supervised)
    y_encoded = _ensure_int_labels(y_encoded)
    print(f"Working with {int(supervised_mask.sum())} samples for supervised learning")
    print(f"Class distribution: {class_counts}")
else:
    print("WARNING: Supervised labels not available or insufficient. Skipping supervised-only steps.")
    supervised_mask = np.ones(adata.n_obs, dtype=bool)
    y_supervised = pd.Series(['Unknown'] * adata.n_obs, index=adata.obs.index)
    y_encoded = np.array([], dtype=np.int64)

# --- Reduce high-dimensional k-mer features using variance-based selection ---
# Check if k-mer features exist
has_tra_kmer = 'X_tcr_tra_kmer' in adata.obsm
has_trb_kmer = 'X_tcr_trb_kmer' in adata.obsm

if has_tra_kmer:
    tra_kmer_supervised = adata.obsm['X_tcr_tra_kmer'][supervised_mask]
else:
    print("Warning: X_tcr_tra_kmer not found. Using placeholder.")
    tra_kmer_supervised = np.zeros((sum(supervised_mask), 100))

if has_trb_kmer:
    trb_kmer_supervised = adata.obsm['X_tcr_trb_kmer'][supervised_mask]
else:
    print("Warning: X_tcr_trb_kmer not found. Using placeholder.")
    trb_kmer_supervised = np.zeros((sum(supervised_mask), 100))

# Select top variance k-mers to reduce dimensionality
def select_top_variance_features(X, n_features=200):
    """Select features with highest variance"""
    variances = np.var(X, axis=0)
    n_features = min(n_features, X.shape[1])  # Don't select more features than exist
    top_indices = np.argsort(variances)[-n_features:]
    return X[:, top_indices], top_indices

print("Reducing k-mer features by variance selection...")
tra_kmer_reduced, tra_top_idx = select_top_variance_features(tra_kmer_supervised, n_features=200)
trb_kmer_reduced, trb_top_idx = select_top_variance_features(trb_kmer_supervised, n_features=200)

print(f"TRA k-mers reduced from {tra_kmer_supervised.shape[1]} to {tra_kmer_reduced.shape[1]}")
print(f"TRB k-mers reduced from {trb_kmer_supervised.shape[1]} to {trb_kmer_reduced.shape[1]}")

# --- 2. Create strategic feature combinations ---
feature_sets = {}

# Helper function to safely get obsm arrays
def _get_obsm_or_zeros(adata, key, mask, n_cols):
    if key in adata.obsm:
        arr = adata.obsm[key][mask]
        return arr[:, :min(n_cols, arr.shape[1])]
    return np.zeros((sum(mask), n_cols))

# Get gene features (try X_gene_pca first, then X_pca)
if 'X_gene_pca' in adata.obsm:
    gene_features = adata.obsm['X_gene_pca'][supervised_mask]
elif 'X_pca' in adata.obsm:
    gene_features = adata.obsm['X_pca'][supervised_mask]
else:
    print("Warning: No gene PCA features found.")
    gene_features = np.zeros((sum(supervised_mask), 50))

# TCR physicochemical features
tcr_physico_cols_tra = ['tra_length', 'tra_molecular_weight', 'tra_hydrophobicity']
tcr_physico_cols_trb = ['trb_length', 'trb_molecular_weight', 'trb_hydrophobicity']

tra_physico = adata.obs[[c for c in tcr_physico_cols_tra if c in adata.obs.columns]].fillna(0)[supervised_mask].values \
    if any(c in adata.obs.columns for c in tcr_physico_cols_tra) else np.zeros((sum(supervised_mask), 3))
trb_physico = adata.obs[[c for c in tcr_physico_cols_trb if c in adata.obs.columns]].fillna(0)[supervised_mask].values \
    if any(c in adata.obs.columns for c in tcr_physico_cols_trb) else np.zeros((sum(supervised_mask), 3))

# Ensure 3 columns each
if tra_physico.shape[1] < 3:
    tra_physico = np.hstack([tra_physico, np.zeros((tra_physico.shape[0], 3 - tra_physico.shape[1]))])
if trb_physico.shape[1] < 3:
    trb_physico = np.hstack([trb_physico, np.zeros((trb_physico.shape[0], 3 - trb_physico.shape[1]))])

tcr_physico = np.column_stack([tra_physico, trb_physico])

# QC features
qc_cols = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
available_qc = [c for c in qc_cols if c in adata.obs.columns]
if available_qc:
    qc_features = adata.obs[available_qc].fillna(0)[supervised_mask].values
else:
    qc_features = np.zeros((sum(supervised_mask), 3))

# Ensure 3 columns for QC
if qc_features.shape[1] < 3:
    qc_features = np.hstack([qc_features, np.zeros((qc_features.shape[0], 3 - qc_features.shape[1]))])

# Basic features (gene expression + physicochemical)
feature_sets['basic'] = np.column_stack([
    gene_features[:, :min(20, gene_features.shape[1])],  # Top 20 gene PCA components
    tcr_physico,
    qc_features
])

# Enhanced gene expression
feature_sets['gene_enhanced'] = np.column_stack([
    gene_features,  # All gene PCA components
    _get_obsm_or_zeros(adata, 'X_gene_svd', supervised_mask, 30),  # Top 30 SVD components
    _get_obsm_or_zeros(adata, 'X_gene_umap', supervised_mask, 20),  # All 20 UMAP components
    tcr_physico,
    qc_features
])

# TCR sequence enhanced
feature_sets['tcr_enhanced'] = np.column_stack([
    gene_features[:, :min(20, gene_features.shape[1])],  # Top 20 gene PCA
    tra_kmer_reduced,  # Top 200 TRA k-mers
    trb_kmer_reduced,  # Top 200 TRB k-mers
    tcr_physico,
    qc_features
])

# Comprehensive (reduced) - Only gene PCA + top k-mers + physicochemical
feature_sets['comprehensive'] = np.column_stack([
    gene_features[:, :min(15, gene_features.shape[1])],  # Top 15 gene PCA
    tra_kmer_reduced[:, :min(50, tra_kmer_reduced.shape[1])],  # Top 50 TRA k-mers
    trb_kmer_reduced[:, :min(50, trb_kmer_reduced.shape[1])],  # Top 50 TRB k-mers
    tcr_physico,
    qc_features
])

print(f"\nFeature set dimensions:")
for name, features in feature_sets.items():
    print(f"  â€¢ {name}: {features.shape}")

print("Comprehensive feature engineering completed!")

Creating comprehensive feature set using ALL available encodings...
Applying strategic dimensionality reduction to high-dimensional features...
Response distribution: {'Non-Responder': 63074, 'Responder': 36993}
Patient_id coverage: 100067/100067
Working with 100067 samples for supervised learning
Class distribution: {'Non-Responder': 63074, 'Responder': 36993}
Reducing k-mer features by variance selection...
TRA k-mers reduced from 1 to 1
TRB k-mers reduced from 1 to 1

Feature set dimensions:
  â€¢ basic: (100067, 29)
  â€¢ gene_enhanced: (100067, 79)
  â€¢ tcr_enhanced: (100067, 31)
  â€¢ comprehensive: (100067, 26)
Comprehensive feature engineering completed!
CPU times: user 339 ms, sys: 13 ms, total: 352 ms
Wall time: 350 ms


In [36]:
# --- Correlation Analysis of Top Features ---
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Validate that adata exists
if 'adata' not in globals() or adata is None:
    raise NameError("adata is not defined. Please run the data loading cells first.")

# Ensure supervised_mask is defined
if 'supervised_mask' not in globals():
    if 'response' in adata.obs.columns:
        supervised_mask = adata.obs['response'].isin(['Responder', 'Non-Responder']).values
    elif 'Response' in adata.obs.columns:
        supervised_mask = adata.obs['Response'].isin(['Responder', 'Non-Responder']).values
    else:
        supervised_mask = np.ones(adata.n_obs, dtype=bool)
        print("Warning: No response column found. Using all cells.")

# Ensure tcr_physico and qc_features are defined
if 'tcr_physico' not in globals():
    # Extract TRA physicochemical features
    tra_cols = ['tra_length', 'tra_molecular_weight', 'tra_hydrophobicity']
    if all(col in adata.obs.columns for col in tra_cols):
        tra_physico = adata.obs[tra_cols].fillna(0)[supervised_mask].values
    else:
        tra_physico = np.zeros((np.sum(supervised_mask), 3))
    
    # Extract TRB physicochemical features
    trb_cols = ['trb_length', 'trb_molecular_weight', 'trb_hydrophobicity']
    if all(col in adata.obs.columns for col in trb_cols):
        trb_physico = adata.obs[trb_cols].fillna(0)[supervised_mask].values
    else:
        trb_physico = np.zeros((np.sum(supervised_mask), 3))
    
    # Combine TRA and TRB features
    tcr_physico = np.hstack([tra_physico, trb_physico])

if 'qc_features' not in globals():
    qc_cols = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
    available_qc = [col for col in qc_cols if col in adata.obs.columns]
    if available_qc:
        qc_features = adata.obs[available_qc].fillna(0)[supervised_mask].values
    else:
        qc_features = np.zeros((np.sum(supervised_mask), 3))

# Select a subset of features for the heatmap
# We'll take the top 10 Gene PCs, top 5 physicochemical, and QC metrics
# Ensure we have the data available
if 'X_gene_pca' in adata.obsm:
    gene_pcs = adata.obsm['X_gene_pca'][supervised_mask][:, :min(10, adata.obsm['X_gene_pca'].shape[1])]
    gene_names = [f"Gene_PC{i+1}" for i in range(gene_pcs.shape[1])]
elif 'X_pca' in adata.obsm:
    gene_pcs = adata.obsm['X_pca'][supervised_mask][:, :min(10, adata.obsm['X_pca'].shape[1])]
    gene_names = [f"Gene_PC{i+1}" for i in range(gene_pcs.shape[1])]
else:
    gene_pcs = np.zeros((np.sum(supervised_mask), 10))
    gene_names = [f"Placeholder_PC{i+1}" for i in range(10)]

heatmap_features = np.column_stack([
    gene_pcs,
    tcr_physico,
    qc_features
])
heatmap_feature_names = gene_names + \
                        ['TRA_Len', 'TRA_MW', 'TRA_Hydro', 'TRB_Len', 'TRB_MW', 'TRB_Hydro'] + \
                        ['n_genes', 'total_counts', 'pct_mt']

# Calculate correlation matrix
corr_matrix = np.corrcoef(heatmap_features, rowvar=False)

# Plot
plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0,
            xticklabels=heatmap_feature_names, yticklabels=heatmap_feature_names,
            linewidths=0.5, linecolor='gray', cbar_kws={"shrink": .8})
plt.title("Feature Correlation Matrix (Top Gene PCs + TCR Features)", fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Supervised Classification of Immunotherapy Response
The core predictive task was formulated as a binary classification problem: predicting the patient response label (Responder vs. Non-Responder) for each individual cell. We evaluated a diverse suite of algorithms:
*   **Logistic Regression:** A linear baseline model.
*   **Decision Trees:** A simple, interpretable non-linear model.
*   **Random Forest:** An ensemble of decision trees that reduces overfitting.
*   **XGBoost (Extreme Gradient Boosting):** A highly optimized gradient boosting framework known for strong performance on tabular data.

### Experimental Setup
We designed our experiments to isolate the predictive value of different data modalities. We trained and evaluated models on four nested feature sets:
1.  **Baseline:** Technical covariates only (e.g., mitochondrial percentage, library size).
2.  **Gene-Enhanced:** Baseline + Gene Expression PCs.
3.  **TCR-Enhanced:** Baseline + TCR Encodings (One-hot, K-mer, Physicochemical).
4.  **Comprehensive:** Baseline + Gene Expression PCs + TCR Encodings.

### Validation Strategy (Updated)
To obtain patient-level generalization estimates and to avoid data leakage between cells from the same patient, we use a Leave-One-Patient-Out (LOPO) cross-validation as the outer evaluation loop. Hyperparameter tuning is performed within the training partitions using GroupKFold (grouped by patient) when possible, falling back to stratified folds only when the number of training patients is too small for grouped splits. Feature scaling and imputation are fit on training partitions only and applied to held-out patient data to ensure leakage-free evaluation.

In [37]:
%pip install scipy
import scipy

Note: you may need to restart the kernel to use updated packages.


In [38]:
%%time
# Skip entire LOPO CV block when SKIP_TO_DEEP_LEARNING master switch is enabled
if globals().get('SKIP_TO_DEEP_LEARNING', False):
    print("SKIPPING: Patient-level LOPO CV because SKIP_TO_DEEP_LEARNING is active (FAST MODE).")
    # ensure variables referenced later won't raise NameError if needed
    lopo_summary_rows = []
else:
    # --- Patient-level LOPO CV (Leakage-safe) [OPTIMIZED] ---
    print("Starting patient-level LOPO CV with leakage-safe pipelines (Optimized for Speed/Accuracy)...")

    from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, StratifiedKFold, GridSearchCV, RandomizedSearchCV
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
    import numpy as np
    import pandas as pd
    import xgboost as xgb
    from pathlib import Path
    import gc, time, joblib

    # --- Optimization Settings ---
    USE_RANDOM_SEARCH = True  # Use RandomizedSearchCV for speed
    N_ITER_SEARCH = 15        # Max hyperparameter combinations to try per fold
    N_JOBS_CV = -1            # Parallelize Cross-Validation (uses all cores)
    N_JOBS_MODEL = -1

    # Prepare grouping variable (patient) and supervised mask
    # Robust column detection for patient_id
    patient_id_col = None
    if 'patient_id' in adata.obs.columns:
        patient_id_col = 'patient_id'
    elif 'Patient_ID' in adata.obs.columns:
        adata.obs['patient_id'] = adata.obs['Patient_ID']  # Create lowercase copy
        patient_id_col = 'patient_id'
    elif 'PatientID' in adata.obs.columns:
        adata.obs['patient_id'] = adata.obs['PatientID']  # Create lowercase copy
        patient_id_col = 'patient_id'
    else:
        # Fallback 1: infer from sample_id using metadata_df
        sample_col = None
        for _c in ['sample_id', 'Sample_ID', 'GEX_Sample_ID', 'sample', 'Sample']:
            if _c in adata.obs.columns:
                sample_col = _c
                break

        if sample_col is not None and 'metadata_df' in globals():
            if 'GEX_Sample_ID' in metadata_df.columns and 'Patient_ID' in metadata_df.columns:
                sample_to_patient = (
                    metadata_df[['GEX_Sample_ID', 'Patient_ID']]
                    .dropna()
                    .drop_duplicates()
                    .set_index('GEX_Sample_ID')['Patient_ID']
                )
                adata.obs['patient_id'] = adata.obs[sample_col].map(sample_to_patient)
            else:
                md_cols = {c.lower(): c for c in metadata_df.columns}
                if 'gex_sample_id' in md_cols and 'patient_id' in md_cols:
                    sample_to_patient = (
                        metadata_df[[md_cols['gex_sample_id'], md_cols['patient_id']]]
                        .dropna()
                        .drop_duplicates()
                        .set_index(md_cols['gex_sample_id'])[md_cols['patient_id']]
                    )
                    adata.obs['patient_id'] = adata.obs[sample_col].map(sample_to_patient)

        # Fallback 2: parse patient id from sample_id strings (e.g., "PT1")
        if 'patient_id' not in adata.obs.columns or adata.obs['patient_id'].isna().all():
            if sample_col is not None:
                adata.obs['patient_id'] = adata.obs[sample_col].astype(str).str.extract(r'(PT\d+)')[0]

        if 'patient_id' in adata.obs.columns and adata.obs['patient_id'].notna().any():
            patient_id_col = 'patient_id'
        elif adata.n_obs == 0:
            adata.obs['patient_id'] = pd.Series(index=adata.obs.index, dtype='object')
            patient_id_col = 'patient_id'
        else:
            raise KeyError(
                "No patient ID column found. Tried direct columns, metadata_df mapping, and parsing from sample_id."
            )

    groups_all = np.array(adata.obs[patient_id_col][supervised_mask])
    unique_patients = np.unique(groups_all)
    print(f"Supervised patients: {len(unique_patients)} -> {unique_patients}")

    # --- EARLY VALIDATION: Check for empty supervised set ---
    if len(groups_all) == 0 or len(unique_patients) == 0:
        print("WARNING: No supervised samples found (supervised_mask is empty).")
        print("Skipping patient-level LOPO CV and deep learning evaluation to prevent memory waste and errors.")
        print("This can happen if no samples have valid 'response' annotations.")
        lopo_summary_rows = []
        processed_data_dir = Path('/kaggle/working/Processed_Data') if globals().get('IS_KAGGLE', False) else Path('Processed_Data')
        processed_data_dir.mkdir(parents=True, exist_ok=True)
        try:
            entity_ids_all = np.asarray(adata.obs_names[supervised_mask], dtype=str)
        except Exception:
            entity_ids_all = np.asarray([f'cell_{i}' for i in range(len(y_encoded))], dtype=object)
        if len(entity_ids_all) != len(y_encoded):
            entity_ids_all = np.asarray([f'cell_{i}' for i in range(len(y_encoded))], dtype=object)
        dl_results_rows = []
    else:
        # Per-patient response summary
        patient_response_df = (
            adata.obs[supervised_mask][[patient_id_col, 'response']]
            .reset_index()
            .drop_duplicates(subset=patient_id_col)
            .set_index(patient_id_col)
        )
        print("Per-patient response counts:")
        print(patient_response_df['response'].value_counts())

        # --- Memory cleanup ---
        _start_cleanup = time.time()
        print("Cleaning up temporary variables and large matrices before ML.")
        # Flags (defaults)
        DROP_ONEHOT_OBSM = False
        DROP_RAW = False
        DROP_OBSM_UMAP_TSNE = True

        _vars_to_delete = [
            'tra_onehot','trb_onehot','tra_onehot_flat','trb_onehot_flat',
            'onehot_tra_reduced','onehot_trb_reduced','onehot_trb_pca','onehot_trb_reduced_new',
            'tmp','tmp1','tmp2','seq_scaler','seq_scaler_full','seq_scaler_flat','length_results'
        ]
        for _v in _vars_to_delete:
            if _v in globals():
                try:
                    del globals()[_v]
                except Exception: pass

        try:
            if hasattr(adata, 'obsp'):
                for _k in list(adata.obsp.keys()): 
                    try: del adata.obsp[_k]
                    except: pass
            for _k in ['neighbors', 'umap']:
                if _k in adata.uns: 
                    try: del adata.uns[_k]
                    except: pass
            if DROP_OBSM_UMAP_TSNE:
                for _key in list(adata.obsm.keys()):
                    _lk = _key.lower()
                    if 'umap' in _lk or 'tsne' in _lk or (_lk == 'x_pca' and 'x_gene_pca' not in _lk):
                        try: del adata.obsm[_key]
                        except: pass
            if DROP_ONEHOT_OBSM:
                for _key in ['X_tcr_tra_onehot', 'X_tcr_trb_onehot']:
                     if _key in adata.obsm: 
                         try: del adata.obsm[_key]
                         except: pass
            if DROP_RAW and getattr(adata, 'raw', None) is not None:
                 adata.raw = None
        except Exception as _e:
            print('Error while pruning adata structures:', _e)

        try:
            import tensorflow.keras.backend as K
            K.clear_session()
        except Exception: pass
        gc.collect()

        # --- Define Models & Optimized Hyperparameters ---
        # Defined here to ensure robust execution without dependency on other cells
        param_grids = {
            'Logistic Regression': {'C': [0.1, 1, 10], 'penalty': ['l2'], 'solver': ['liblinear']},
            'Decision Tree': {'max_depth': [5, 10], 'min_samples_split': [5, 10], 'min_samples_leaf': [2, 4]},
            'Random Forest': {'n_estimators': [100], 'max_depth': [10, 20], 'min_samples_split': [5, 10]}, # Reduced grid
            'XGBoost': {
                'max_depth': [3, 5], 
                'learning_rate': [0.05, 0.1], 
                'subsample': [0.8, 1.0], 
                'colsample_bytree': [0.8, 1.0], 
                'n_estimators': [100]
            }
        }

        models_eval = {
            'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, solver='liblinear'),
            'Decision Tree': DecisionTreeClassifier(random_state=42),
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=N_JOBS_MODEL),
            'XGBoost': (lambda: (globals().get('XGBClassifierSK', xgb.XGBClassifier)(
                random_state=42, 
                use_label_encoder=False, 
                eval_metric='logloss',
                n_jobs=N_JOBS_MODEL,
                **({'tree_method':'gpu_hist','predictor':'gpu_predictor'} 
                   if globals().get('XGBOOST_GPU_AVAILABLE', False) 
                   else {'tree_method':'hist'}) # Optimization: Use 'hist' on CPU which is much faster than 'exact'
            )))()
        }
        _apply_gpu_patches()

        # Adapt param_grids to pipeline format (prefix 'clf__')
        param_grid_pipeline = {m: {f'clf__{k}': v for k, v in g.items()} for m, g in param_grids.items()}

        logo = LeaveOneGroupOut()
        lopo_summary_rows = []
        processed_data_dir = Path('/kaggle/working/Processed_Data') if globals().get('IS_KAGGLE', False) else Path('Processed_Data')
        processed_data_dir.mkdir(parents=True, exist_ok=True)
        try:
            entity_ids_all = np.asarray(adata.obs_names[supervised_mask], dtype=str)
        except Exception:
            entity_ids_all = np.asarray([f'cell_{i}' for i in range(len(y_encoded))], dtype=object)
        if len(entity_ids_all) != len(y_encoded):
            entity_ids_all = np.asarray([f'cell_{i}' for i in range(len(y_encoded))], dtype=object)

        # Iterate feature sets
        for feature_name, X_features in feature_sets.items():
            print(f"\n=== Feature set: {feature_name} (shape={X_features.shape}) ===")
            X = X_features
            y = y_encoded
            groups = groups_all

            accum = {m: {'y_true': [], 'y_pred': [], 'y_proba': [], 'groups': []} for m in models_eval.keys()}
            cleared_prediction_files = set()

            for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
                held_patient = np.unique(groups[test_idx])
                print(f"LOPO fold {fold_idx+1}/{len(unique_patients)} -- held patient(s): {held_patient}")

                X_tr, X_te = X[train_idx], X[test_idx]
                y_tr, y_te = y[train_idx], y[test_idx]
                groups_tr = groups[train_idx]
                
                n_train_groups = len(np.unique(groups_tr))
                inner_n_splits = min(3, n_train_groups) if n_train_groups >= 2 else 1

                for model_name, base_model in models_eval.items():
                    pipeline = Pipeline([
                        ('imputer', SimpleImputer(strategy='mean')),
                        ('scaler', StandardScaler()),
                        ('clf', base_model)
                    ])

                    # Hyperparameter tuning
                    # Use RandomizedSearchCV to cap the maximum time spent on regular algorithms
                    if model_name in param_grid_pipeline:
                        # Determine strategy
                        grid_params = param_grid_pipeline[model_name]
                        grid_size = np.prod([len(v) for v in grid_params.values()])
                        
                        # If grid is small enough, use GridSearch. If large, use RandomizedSearchCV
                        if USE_RANDOM_SEARCH and grid_size > N_ITER_SEARCH:
                            search_impl = RandomizedSearchCV(pipeline, grid_params, n_iter=N_ITER_SEARCH, 
                                                           cv=inner_n_splits if inner_n_splits > 1 else StratifiedKFold(3),
                                                           scoring='accuracy', n_jobs=N_JOBS_CV, random_state=42)
                        else:
                            search_impl = GridSearchCV(pipeline, grid_params, 
                                                     cv=inner_n_splits if inner_n_splits > 1 else StratifiedKFold(3),
                                                     scoring='accuracy', n_jobs=N_JOBS_CV)

                        # Fit
                        if inner_n_splits >= 2:
                            search_impl.fit(X_tr, y_tr, groups=groups_tr)
                        else: 
                            # Fallback for few groups
                            search_impl.fit(X_tr, y_tr)
                            
                        best_model = search_impl.best_estimator_
                    else:
                        best_model = pipeline.fit(X_tr, y_tr)

                    # Save model weights
                    try:
                        model_dir = Path('Output/Models/LOPO')
                        model_dir.mkdir(parents=True, exist_ok=True)
                        sanitized_model_name = model_name.replace(' ', '_')
                        model_filename = model_dir / f'model_{sanitized_model_name}_{feature_name}_fold{fold_idx}.joblib'
                        joblib.dump(best_model, model_filename)
                    except Exception as e:
                        print(f"Failed to save model {model_name}: {e}")

                    # Predict
                    y_pred = best_model.predict(X_te)
                    try:
                        y_pred_proba = best_model.predict_proba(X_te)[:, 1]
                    except Exception:
                        try:
                            d = best_model.decision_function(X_te)
                            y_pred_proba = d[:, 1] if d.ndim > 1 else d
                        except:
                            y_pred_proba = np.zeros(len(y_pred))

                    held_patient_str = '|'.join(map(str, np.atleast_1d(held_patient).tolist()))
                    sanitized_model_name = model_name.replace(' ', '_').replace('/', '_')
                    cell_detail_path = processed_data_dir / f'lopo_cell_predictions_{feature_name}_{sanitized_model_name}.csv'
                    patient_detail_path = processed_data_dir / f'lopo_patient_predictions_{feature_name}_{sanitized_model_name}.csv'
                    for _path in (cell_detail_path, patient_detail_path):
                        if _path not in cleared_prediction_files and _path.exists():
                            _path.unlink()
                        cleared_prediction_files.add(_path)

                    entity_slice = entity_ids_all[test_idx] if len(entity_ids_all) == len(y) else np.asarray([f'cell_{i}' for i in test_idx], dtype=object)
                    fold_pred_df = pd.DataFrame({
                        'model': model_name,
                        'feature_set': feature_name,
                        'evaluation_level': 'cell',
                        'fold_id': fold_idx + 1,
                        'held_out_patient': held_patient_str,
                        'patient': np.asarray(groups[test_idx]).astype(str),
                        'entity_id': np.asarray(entity_slice).astype(str),
                        'y_true': y_te,
                        'y_pred': y_pred,
                        'y_proba': y_pred_proba,
                    })
                    fold_pred_df.to_csv(cell_detail_path, mode='a', header=not cell_detail_path.exists(), index=False)

                    patient_fold_df = fold_pred_df.groupby('patient', as_index=False).agg({'y_true': 'first', 'y_proba': 'mean'})
                    patient_fold_df['y_pred'] = (patient_fold_df['y_proba'] >= 0.5).astype(int)
                    patient_fold_df['model'] = model_name
                    patient_fold_df['feature_set'] = feature_name
                    patient_fold_df['evaluation_level'] = 'patient'
                    patient_fold_df['fold_id'] = fold_idx + 1
                    patient_fold_df['held_out_patient'] = patient_fold_df['patient'].astype(str)
                    patient_fold_df['entity_id'] = patient_fold_df['patient'].astype(str)
                    patient_fold_df = patient_fold_df[['model', 'feature_set', 'evaluation_level', 'fold_id', 'held_out_patient', 'patient', 'entity_id', 'y_true', 'y_pred', 'y_proba']]
                    patient_fold_df.to_csv(patient_detail_path, mode='a', header=not patient_detail_path.exists(), index=False)

                    # Accumulate
                    accum[model_name]['y_true'].extend(y_te.tolist())
                    accum[model_name]['y_pred'].extend(y_pred.tolist())
                    accum[model_name]['y_proba'].extend(y_pred_proba.tolist())
                    accum[model_name]['groups'].extend(groups[test_idx].tolist())

        # --- Aggregation & Reporting ---
        for model_name, data_dict in accum.items():
            y_true_all = np.array(data_dict['y_true'])
            y_pred_all = np.array(data_dict['y_pred'])
            y_proba_all = np.array(data_dict['y_proba'])
            groups_all_pred = np.array(data_dict.get('groups', []), dtype=object)

            if len(y_true_all) == 0: continue

            # Cell-level metrics
            acc = accuracy_score(y_true_all, y_pred_all)
            prec = precision_score(y_true_all, y_pred_all, zero_division=0)
            rec = recall_score(y_true_all, y_pred_all, zero_division=0)
            f1s = f1_score(y_true_all, y_pred_all, zero_division=0)
            try: auc = roc_auc_score(y_true_all, y_proba_all)
            except: auc = float('nan')
            cm = confusion_matrix(y_true_all, y_pred_all)
            if cm.size == 4:
                tn, fp, fn, tp = cm.ravel()
                spec = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
                npv = tn / (tn + fn) if (tn + fn) > 0 else float('nan')
            else: spec, npv = float('nan'), float('nan')

            lopo_summary_rows.append({
                'feature_set': feature_name, 'model': model_name, 'evaluation_level': 'cell',
                'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1s, 'auc': auc,
                'specificity': spec, 'npv': npv, 'n_patients': len(unique_patients), 'n_cells': X_features.shape[0]
            })

            # Patient-level aggregation
            try:
                pred_df = pd.DataFrame({'patient': groups_all_pred, 'y_true': y_true_all, 'y_proba': y_proba_all})
                patient_summary = pred_df.groupby('patient').agg({'y_proba': 'mean', 'y_true': 'first'}).reset_index()
                patient_summary['y_pred'] = (patient_summary['y_proba'] >= 0.5).astype(int)
                patient_summary['model'] = model_name
                patient_summary['feature_set'] = feature_name
                patient_summary['evaluation_level'] = 'patient'
                patient_summary['entity_id'] = patient_summary['patient'].astype(str)
                patient_summary['held_out_patient'] = patient_summary['patient'].astype(str)

                y_t, y_p = patient_summary['y_true'], patient_summary['y_pred']
                try: auc_p = roc_auc_score(y_t, patient_summary['y_proba'])
                except: auc_p = float('nan')
                
                lopo_summary_rows.append({
                    'feature_set': feature_name, 'model': model_name, 'evaluation_level': 'patient',
                    'accuracy': accuracy_score(y_t, y_p), 'precision': precision_score(y_t, y_p, zero_division=0),
                    'recall': recall_score(y_t, y_p, zero_division=0), 'f1': f1_score(y_t, y_p, zero_division=0),
                    'auc': auc_p, 'n_patients': len(patient_summary), 'n_cells': X_features.shape[0]
                })
                
                sanitized_model_name = model_name.replace(' ', '_').replace('/', '_')
                p_out = processed_data_dir / f'lopo_patient_prediction_summary_{feature_name}_{sanitized_model_name}.csv'
                patient_summary.to_csv(p_out, index=False)
            except Exception as e:
                print(f"Failed patient-level metrics: {e}")

    lopo_df = pd.DataFrame(lopo_summary_rows)
    output_path = processed_data_dir / 'lopo_results.csv'
    processed_data_dir.mkdir(exist_ok=True)
    lopo_df.to_csv(output_path, index=False)
    print(f"LOPO results saved to: {output_path}")
    display(lopo_df)

Starting patient-level LOPO CV with leakage-safe pipelines (Optimized for Speed/Accuracy)...
Supervised patients: 4 -> ['PT1' 'PT2' 'PT3' 'PT4']
Per-patient response counts:
response
Non-Responder    2
Responder        2
Name: count, dtype: int64
Cleaning up temporary variables and large matrices before ML.
Patched models_eval['XGBoost'] to use GPU (method=device).
Patched models_eval['Random Forest'] to use n_jobs=-1.
Patched param_grids['XGBoost'] with GPU options (method=device).
Patched models_eval['XGBoost'] to use GPU (method=device).
Patched models_eval['Random Forest'] to use n_jobs=-1.
Patched param_grids['XGBoost'] with GPU options (method=device).

=== Feature set: basic (shape=(100067, 29)) ===
LOPO fold 1/4 -- held patient(s): ['PT1']


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:26:13] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:26:13] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:26:13] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:26:13] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [17:26:14] WARNING: /works

LOPO fold 2/4 -- held patient(s): ['PT2']


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:30:20] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:30:20] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:30:20] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:30:20] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:30:21] WARNING: /w

LOPO fold 3/4 -- held patient(s): ['PT3']


...

LOPO fold 4/4 -- held patient(s): ['PT4']


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:31:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:31:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:31:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:31:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:31:02] WARNING: /w

LOPO results saved to: Processed_Data/lopo_results.csv


,feature_set,model,evaluation_level,accuracy,precision,recall,f1,auc,specificity,npv,n_patients,n_cells
0,comprehensive,Logistic Regression,cell,0.873775,0.843630,0.808396,0.825637,0.933421,0.912119,0.890311,4,100067
1,comprehensive,Logistic Regression,patient,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,4,100067
2,comprehensive,Decision Tree,cell,0.817043,0.762231,0.734085,0.747893,0.854360,0.865697,0.847346,4,100067
3,comprehensive,Decision Tree,patient,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,4,100067
4,comprehensive,Random Forest,cell,0.856786,0.835336,0.763009,0.797536,0.930442,0.911786,0.867722,4,100067
5,comprehensive,Random Forest,patient,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,4,100067
6,comprehensive,XGBoost,cell,0.884038,0.858066,0.822345,0.839826,0.952615,0.920221,0.898288,4,100067
7,comprehensive,XGBoost,patient,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,4,100067


CPU times: user 29min 19s, sys: 9.7 s, total: 29min 29s
Wall time: 1h 8min 12s


## 13. Deep Learning Multi-Modal Classification

We implement and compare three advanced deep learning architectures to classify patient response based on the integrated multi-modal features:

1.  **1D CNN (Convolutional Neural Network):** Captures local patterns and dependencies within the feature vector.
2.  **BiLSTM (Bidirectional Long Short-Term Memory):** Models sequential dependencies in both directions, effective for capturing context in feature sequences.
3.  **Transformer Encoder:** Utilizes self-attention mechanisms to weigh the importance of different features dynamically, enabling the model to focus on the most relevant biological signals regardless of their position in the input.

These models are trained using a Leave-One-Patient-Out (LOPO) cross-validation strategy to ensure robust and generalizable performance assessment.


In [39]:
%%time
# --- Advanced Multimodal Deep Learning (MLP / CNN / RNN / BiLSTM / Transformer)
# This cell implements leakage-safe validation for several deep architectures.
# OPTIMIZATION: Automatic Device Config (TPU > GPU > CPU)
# OPTIMIZATION: Switched to GroupKFold (5 splits) instead of LOPO to prevent timeouts.

if globals().get('SKIP_TO_DEEP_LEARNING', False):
    print("SKIPPING: Patient-level LOPO CV because SKIP_TO_DEEP_LEARNING is active (FAST MODE).")
else:
    # --- EARLY VALIDATION: Check if supervised data exists ---
    if 'supervised_mask' not in globals() or supervised_mask.sum() == 0:
        print("WARNING: No supervised samples available (supervised_mask is empty or undefined).")
        print("Skipping deep learning evaluation to prevent memory waste and errors.")
        dl_results_rows = []
    else:
        import itertools
        import time
        import math
        import random
        import gc
        import tracemalloc
        import numpy as np
        import pandas as pd
        import tensorflow as tf
        from tensorflow import keras
        from tensorflow.keras import layers, regularizers
        from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
        from sklearn.model_selection import GroupKFold, StratifiedKFold, LeaveOneGroupOut, GroupShuffleSplit
        from sklearn.preprocessing import StandardScaler
        from sklearn.utils.class_weight import compute_class_weight
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
        from pathlib import Path
        from joblib import Parallel, delayed
        try:
            from tqdm.auto import tqdm
        except ImportError:
            def tqdm(x, **kwargs): return x

        # Deterministic seeds
        SEED = 42
        np.random.seed(SEED)
        random.seed(SEED)
        tf.random.set_seed(SEED)
        
        # --- Robust Device Configuration: TPU > GPU > CPU ---
        def configure_tf_strategy():
            """Detects hardware and returns the appropriate DistributionStrategy."""
            try:
                # 1. Try TPU
                tpu = tf.distribute.cluster_resolver.TPUClusterResolver() # Throws if no TPU
                print(f"Running on TPU: {tpu.master()}")
                tf.config.experimental_connect_to_cluster(tpu)
                tf.tpu.experimental.initialize_tpu_system(tpu)
                strategy = tf.distribute.TPUStrategy(tpu)
                print("  TPU Strategy initialized.")
                return strategy
            except ValueError:
                pass # No TPU found

            # 2. Try GPU
            gpus = tf.config.list_physical_devices('GPU')
            if gpus:
                print(f"Found {len(gpus)} GPU(s): {gpus}")
                try:
                    # Enable memory growth
                    for gpu in gpus:
                        tf.config.experimental.set_memory_growth(gpu, True)
                    
                    # Use mixed precision on GPU for speed & memory
                    tf.keras.mixed_precision.set_global_policy('mixed_float16')
                    print("  Mixed precision enabled for GPU.")
                except RuntimeError as e:
                    print(f"  GPU Config Error: {e}")
                
                # Explicit GPU strategies to guarantee accelerator placement.
                if len(gpus) > 1:
                    strategy = tf.distribute.MirroredStrategy()
                    print("  Using MirroredStrategy on multiple GPUs.")
                else:
                    strategy = tf.distribute.OneDeviceStrategy(device='/GPU:0')
                    print("  Using OneDeviceStrategy on GPU:0.")
                return strategy

            # 3. Fallback to CPU
            print("No GPU/TPU detected. Using CPU.")
            return tf.distribute.OneDeviceStrategy(device='/CPU:0')

        # Initialize Strategy
        STRATEGY = configure_tf_strategy()
        print(f"Number of replicas: {STRATEGY.num_replicas_in_sync}")

        # Start memory tracking
        tracemalloc.start()
        _mem_start = tracemalloc.get_traced_memory()
        _time_start = time.time()

        print("TensorFlow version:", tf.__version__)

        # Performance helper: tf.data builder + AUTOTUNE
        AUTOTUNE = tf.data.AUTOTUNE
        def make_tf_dataset(inputs, targets, sample_weight=None, batch_size=32, shuffle=False):
            """Create a tf.data.Dataset from numpy inputs/targets with optional sample weights.
            Supports single-array inputs or list/tuple multi-inputs (e.g., [seq, gene]).
            """
            if inputs is None:
                raise ValueError("make_tf_dataset: inputs is None")

            # Ensure numpy types and dtypes
            if isinstance(inputs, (list, tuple)):
                arrays = [np.asarray(x).astype('float32') for x in inputs]
                labels = np.asarray(targets).astype('float32')
                if sample_weight is None:
                    ds = tf.data.Dataset.from_tensor_slices((tuple(arrays), labels))
                else:
                    ds = tf.data.Dataset.from_tensor_slices((tuple(arrays), labels, np.asarray(sample_weight).astype('float32')))
            else:
                arr = np.asarray(inputs).astype('float32')
                labels = np.asarray(targets).astype('float32')
                if sample_weight is None:
                    ds = tf.data.Dataset.from_tensor_slices((arr, labels))
                else:
                    ds = tf.data.Dataset.from_tensor_slices((arr, labels, np.asarray(sample_weight).astype('float32')))

            if shuffle:
                ds = ds.shuffle(buffer_size=2048, seed=SEED)
            ds = ds.batch(batch_size)
            ds = ds.prefetch(AUTOTUNE)
            return ds

        # Helper: prepare sequence arrays if available (robust, auto-detect channels/seq_len)
        def prepare_onehot_sequences(adata, mask, n_channels=20):
            """
            Returns (tra_seq, trb_seq, seq_len) or (None,None,None).
            Tries precomputed one-hot arrays first, then falls back to on-the-fly CDR3 one-hot encoding.
            """
            if 'X_tcr_tra_onehot' in adata.obsm and 'X_tcr_trb_onehot' in adata.obsm:
                tra_flat = adata.obsm['X_tcr_tra_onehot'][mask]
                trb_flat = adata.obsm['X_tcr_trb_onehot'][mask]
                try:
                    if hasattr(tra_flat, 'toarray'):
                        tra_flat = tra_flat.toarray()
                    if hasattr(trb_flat, 'toarray'):
                        trb_flat = trb_flat.toarray()

                    tra_flat = np.asarray(tra_flat)
                    trb_flat = np.asarray(trb_flat)

                    if tra_flat.ndim == 2 and trb_flat.ndim == 2:
                        if tra_flat.shape[0] != trb_flat.shape[0]:
                            min_n = min(tra_flat.shape[0], trb_flat.shape[0])
                            tra_flat = tra_flat[:min_n]
                            trb_flat = trb_flat[:min_n]

                        total_cols = tra_flat.shape[1]
                        if total_cols % n_channels == 0:
                            seq_len = total_cols // n_channels
                            try:
                                return (
                                    tra_flat.reshape(tra_flat.shape[0], seq_len, n_channels),
                                    trb_flat.reshape(trb_flat.shape[0], seq_len, n_channels),
                                    seq_len,
                                )
                            except Exception:
                                pass

                        candidates = []
                        for nc in range(1, 33):
                            if total_cols % nc == 0:
                                sl = total_cols // nc
                                if 3 <= sl <= 200:
                                    candidates.append((nc, sl))

                        if candidates:
                            preferred = [20, 15, 10, 8, 5, 4, 2, 1]
                            chosen = None
                            for p in preferred:
                                for nc, sl in candidates:
                                    if nc == p:
                                        chosen = (nc, sl)
                                        break
                                if chosen is not None:
                                    break
                            if chosen is None:
                                chosen = max(candidates, key=lambda x: x[0])

                            nc, seq_len = chosen
                            try:
                                tra_seq = tra_flat.reshape(tra_flat.shape[0], seq_len, nc)
                                trb_seq = trb_flat.reshape(trb_flat.shape[0], seq_len, nc)
                                print(f'prepare_onehot_sequences: using precomputed one-hot (n_channels={nc}, seq_len={seq_len}).')
                                return tra_seq, trb_seq, seq_len
                            except Exception:
                                pass
                except Exception as e:
                    print(f"prepare_onehot_sequences precomputed path failed: {e}")

            # Fallback path: build compact one-hot tensors directly from CDR3 columns.
            tra_col = 'cdr3_TRA' if 'cdr3_TRA' in adata.obs.columns else ('CDR3_TRA' if 'CDR3_TRA' in adata.obs.columns else None)
            trb_col = 'cdr3_TRB' if 'cdr3_TRB' in adata.obs.columns else ('CDR3_TRB' if 'CDR3_TRB' in adata.obs.columns else None)
            if tra_col is None and trb_col is None:
                return None, None, None

            try:
                alphabet = 'ACDEFGHIKLMNPQRSTVWY'
                a2i = {aa: i for i, aa in enumerate(alphabet)}
                seq_len_fb = 48

                tra_vals = adata.obs.loc[mask, tra_col].fillna('').astype(str).str.upper().values if tra_col else None
                trb_vals = adata.obs.loc[mask, trb_col].fillna('').astype(str).str.upper().values if trb_col else None

                n = len(tra_vals) if tra_vals is not None else len(trb_vals)
                if n <= 0:
                    return None, None, None
                if tra_vals is None:
                    tra_vals = np.array([''] * n)
                if trb_vals is None:
                    trb_vals = np.array([''] * n)

                def _encode(vals):
                    out = np.zeros((len(vals), seq_len_fb, len(alphabet)), dtype=np.float32)
                    for i, seq in enumerate(vals):
                        seq = ''.join(ch for ch in str(seq) if ch in a2i)[:seq_len_fb]
                        for p, ch in enumerate(seq):
                            out[i, p, a2i[ch]] = 1.0
                    return out

                tra_seq = _encode(tra_vals)
                trb_seq = _encode(trb_vals)
                print('prepare_onehot_sequences: built fallback one-hot from CDR3 strings.')
                return tra_seq, trb_seq, seq_len_fb
            except Exception as e:
                print(f'prepare_onehot_sequences fallback failed: {e}')
                return None, None, None


        # Model builders (Modified to not compile immediately so we can compile inside strategy scope if needed)
        def create_compiled_model(model_fn, *args, lr=1e-3, **kwargs):
            """Creates and compiles a model within the current strategy scope."""
            with STRATEGY.scope():
                model = model_fn(*args, **kwargs)
                # Use jit_compile=True for XLA optimization if not on TPU (TPU implies XLA)
                jit = False if isinstance(STRATEGY, tf.distribute.TPUStrategy) else True
                model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), 
                            loss='binary_crossentropy', 
                            metrics=[keras.metrics.AUC(name='auc'), 'accuracy'],
                            jit_compile=jit)
            return model

        def build_mlp_graph(input_dim, hidden1=128, hidden2=64, dropout=0.3, l2_reg=1e-3):
            inp = keras.Input(shape=(input_dim,), name='gene_input')
            x = layers.Dense(hidden1, kernel_regularizer=regularizers.l2(l2_reg))(inp)
            x = layers.BatchNormalization()(x)
            x = layers.Activation('relu')(x)
            x = layers.Dropout(dropout)(x)
            x = layers.Dense(hidden2, kernel_regularizer=regularizers.l2(l2_reg))(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation('relu')(x)
            x = layers.Dropout(dropout)(x)
            out = layers.Dense(1, activation='sigmoid')(x)
            return keras.Model(inputs=inp, outputs=out)

        def build_cnn_graph(seq_len, n_channels, gene_dim=None, conv_filters=64, kernel_size=5, dropout=0.3, l2_reg=1e-3):
            seq_in = keras.Input(shape=(seq_len, n_channels), name='seq_input')
            x = layers.Conv1D(conv_filters, kernel_size, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(seq_in)
            x = layers.Conv1D(conv_filters, kernel_size, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(x)
            x = layers.GlobalMaxPooling1D()(x)
            
            if gene_dim is not None:
                gene_in = keras.Input(shape=(gene_dim,), name='gene_input')
                g = layers.Dense(64, activation='relu')(gene_in)
                x = layers.concatenate([x, g])
                out_in = [seq_in, gene_in]
            else:
                out_in = seq_in
                
            x = layers.Dropout(dropout)(x)
            x = layers.Dense(64, activation='relu')(x)
            out = layers.Dense(1, activation='sigmoid')(x)
            return keras.Model(inputs=out_in, outputs=out)

        
        def build_rnn_graph(seq_len, n_channels, gene_dim=None, rnn_units=96, dropout=0.3, l2_reg=1e-3):
            seq_in = keras.Input(shape=(seq_len, n_channels), name='seq_input')
            x = layers.SimpleRNN(rnn_units, return_sequences=False, kernel_regularizer=regularizers.l2(l2_reg))(seq_in)

            if gene_dim is not None:
                gene_in = keras.Input(shape=(gene_dim,), name='gene_input')
                g = layers.Dense(64, activation='relu')(gene_in)
                x = layers.concatenate([x, g])
                out_in = [seq_in, gene_in]
            else:
                out_in = seq_in

            x = layers.Dropout(dropout)(x)
            x = layers.Dense(64, activation='relu')(x)
            out = layers.Dense(1, activation='sigmoid')(x)
            return keras.Model(inputs=out_in, outputs=out)

        def build_bilstm_graph(seq_len, n_channels, gene_dim=None, lstm_units=128, dropout=0.3, l2_reg=1e-3):
            seq_in = keras.Input(shape=(seq_len, n_channels), name='seq_input')
            x = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=False, kernel_regularizer=regularizers.l2(l2_reg)))(seq_in)
            
            if gene_dim is not None:
                gene_in = keras.Input(shape=(gene_dim,), name='gene_input')
                g = layers.Dense(64, activation='relu')(gene_in)
                x = layers.concatenate([x, g])
                out_in = [seq_in, gene_in]
            else:
                out_in = seq_in
                
            x = layers.Dropout(dropout)(x)
            x = layers.Dense(64, activation='relu')(x)
            out = layers.Dense(1, activation='sigmoid')(x)
            return keras.Model(inputs=out_in, outputs=out)

        # Small Transformer encoder block
        class TransformerBlock(layers.Layer):
            def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
                super(TransformerBlock, self).__init__(**kwargs)
                self.embed_dim = embed_dim
                self.num_heads = num_heads
                self.ff_dim = ff_dim
                self.rate = rate
                self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
                self.ffn = keras.Sequential([layers.Dense(ff_dim, activation='relu'), layers.Dense(embed_dim)])
                self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
                self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
                self.dropout1 = layers.Dropout(rate)
                self.dropout2 = layers.Dropout(rate)
                
            def call(self, inputs, training=None):
                attn_output = self.att(inputs, inputs)
                attn_output = self.dropout1(attn_output, training=training)
                out1 = self.layernorm1(inputs + attn_output)
                ffn_output = self.ffn(out1)
                ffn_output = self.dropout2(ffn_output, training=training)
                return self.layernorm2(out1 + ffn_output)
                
            def get_config(self):
                config = super().get_config()
                config.update({
                    "embed_dim": self.embed_dim,
                    "num_heads": self.num_heads,
                    "ff_dim": self.ff_dim,
                    "rate": self.rate,
                })
                return config

        def build_transformer_graph(seq_len, n_channels, gene_dim=None, embed_dim=64, num_heads=4, ff_dim=128, dropout=0.1):
            seq_in = keras.Input(shape=(seq_len, n_channels), name='seq_input')
            # project channels to embed_dim
            x = layers.Dense(embed_dim)(seq_in)
            x = TransformerBlock(embed_dim, num_heads, ff_dim, rate=dropout)(x)
            x = layers.GlobalAveragePooling1D()(x)
            
            if gene_dim is not None:
                gene_in = keras.Input(shape=(gene_dim,), name='gene_input')
                g = layers.Dense(64, activation='relu')(gene_in)
                x = layers.concatenate([x, g])
                inputs_list = [seq_in, gene_in]
            else:
                inputs_list = seq_in
                
            x = layers.Dropout(dropout)(x)
            x = layers.Dense(64, activation='relu')(x)
            out = layers.Dense(1, activation='sigmoid')(x)
            return keras.Model(inputs=inputs_list, outputs=out)

        # --- Parallel Training Helper ---
        def train_eval_single_config(cfg_idx, config, use_gene, use_seq, 
                                    X_tr_gene, X_val_gene, 
                                    X_tr_seq, X_val_seq, 
                                    X_tr_flat, X_val_flat, 
                                    y_train, y_val, class_weights):
            """
            Train and evaluate a single model configuration for one inner split.
            """
            arch, hu, dr, lr, bs, epochs = config
            
            # 1. Check validity of config for current data availability
            if arch in ['CNN','RNN','BiLSTM','Transformer'] and not use_seq:
                return cfg_idx, -1.0
                
            try:
                fit_inputs = None
                val_inputs = None
                model = None
                
                # 2. Build Model & Inputs (Using Strategy Scope via create_compiled_model)
                if arch == 'MLP':
                    if use_gene and X_tr_gene is not None:
                        fit_inputs = X_tr_gene
                        val_inputs = X_val_gene
                        input_dim = fit_inputs.shape[1]
                        model = create_compiled_model(build_mlp_graph, input_dim, hidden1=hu, hidden2=max(32, hu//2), dropout=dr, l2_reg=1e-3, lr=lr)
                    elif use_seq and X_tr_flat is not None:
                        fit_inputs = X_tr_flat
                        val_inputs = X_val_flat
                        input_dim = fit_inputs.shape[1]
                        model = create_compiled_model(build_mlp_graph, input_dim, hidden1=hu, hidden2=max(32, hu//2), dropout=dr, l2_reg=1e-3, lr=lr)
                    else:
                        return cfg_idx, -1.0

                elif arch == 'CNN':
                    fit_inputs = [X_tr_seq, X_tr_gene] if use_gene else X_tr_seq
                    val_inputs = [X_val_seq, X_val_gene] if use_gene else X_val_seq
                    gene_dim_val = (X_tr_gene.shape[1] if use_gene else None)
                    model = create_compiled_model(build_cnn_graph, X_tr_seq.shape[1], X_tr_seq.shape[2], gene_dim=gene_dim_val, conv_filters=hu, kernel_size=5, dropout=dr, l2_reg=1e-3, lr=lr)

                elif arch == 'RNN':
                    fit_inputs = [X_tr_seq, X_tr_gene] if use_gene else X_tr_seq
                    val_inputs = [X_val_seq, X_val_gene] if use_gene else X_val_seq
                    gene_dim_val = (X_tr_gene.shape[1] if use_gene else None)
                    model = create_compiled_model(build_rnn_graph, X_tr_seq.shape[1], X_tr_seq.shape[2], gene_dim=gene_dim_val, rnn_units=max(32, hu // 2), dropout=dr, l2_reg=1e-3, lr=lr)

                elif arch == 'BiLSTM':
                    fit_inputs = [X_tr_seq, X_tr_gene] if use_gene else X_tr_seq
                    val_inputs = [X_val_seq, X_val_gene] if use_gene else X_val_seq
                    gene_dim_val = (X_tr_gene.shape[1] if use_gene else None)
                    model = create_compiled_model(build_bilstm_graph, X_tr_seq.shape[1], X_tr_seq.shape[2], gene_dim=gene_dim_val, lstm_units=hu, dropout=dr, l2_reg=1e-3, lr=lr)
                    
                else: # Transformer
                    fit_inputs = [X_tr_seq, X_tr_gene] if use_gene else X_tr_seq
                    val_inputs = [X_val_seq, X_val_gene] if use_gene else X_val_seq
                    gene_dim_val = (X_tr_gene.shape[1] if use_gene else None)
                    model = create_compiled_model(build_transformer_graph, X_tr_seq.shape[1], X_tr_seq.shape[2], gene_dim=gene_dim_val, embed_dim=max(32, hu//2), num_heads=4, ff_dim=hu, dropout=dr, lr=lr)

                # Train
                es = keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, verbose=0)
                rlr = ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=2, min_lr=1e-6, verbose=0)

                # Convert to tf.data datasets for efficient pipeline + GPU utilization
                try:
                    sw_train = None
                    sw_val = None
                    if class_weights:
                        sw_train = np.array([class_weights.get(int(y), 1.0) for y in y_train], dtype='float32')
                        sw_val = np.array([class_weights.get(int(y), 1.0) for y in y_val], dtype='float32')

                    train_ds = make_tf_dataset(fit_inputs, y_train, sample_weight=sw_train, batch_size=bs, shuffle=True)
                    val_ds = make_tf_dataset(val_inputs, y_val, sample_weight=sw_val, batch_size=bs, shuffle=False)

                    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=[es, rlr], verbose=0)
                except Exception as e:
                    # fallback to original array-based fit if dataset conversion fails
                    history = model.fit(fit_inputs, y_train, validation_data=(val_inputs, y_val), epochs=epochs, batch_size=bs, class_weight=class_weights, callbacks=[es, rlr], verbose=0)

                # Retrieve metric
                val_metric = max(history.history['val_auc']) if 'val_auc' in history.history else 0.0

                # Cleanup
                del model
                keras.backend.clear_session()
                gc.collect()
                
                return cfg_idx, val_metric

            except Exception as e:
                # print(f"Config {config} failed: {e}") # debug
                return cfg_idx, -1.0

        # Manual hyperparameter grid for DL
        from itertools import product

        # Heuristic default batch size scaled by number of replicas
        DEFAULT_BATCH = int(32 * max(1, STRATEGY.num_replicas_in_sync))
        if DEFAULT_BATCH < 8:
            DEFAULT_BATCH = 8
        print(f"Default DL batch size set to: {DEFAULT_BATCH}")

        # Optimized Grid: Further reduced search space to prevent timeout
        dl_param_grid = {
            'arch': ['MLP', 'CNN', 'RNN', 'BiLSTM', 'Transformer'],
            'hidden_units': [128], # Fixed size
            'dropout': [0.3],      # Fixed dropout
            'lr': [1e-3],          # Fixed LR, rely on RLR
            'batch_size': [DEFAULT_BATCH],
            'epochs': [25],        # Reduced max epochs slightly
        }
        grid_items = list(product(dl_param_grid['arch'], dl_param_grid['hidden_units'], dl_param_grid['dropout'], dl_param_grid['lr'], dl_param_grid['batch_size'], dl_param_grid['epochs']))
        print(f"DL hyperparameter combinations per fold: {len(grid_items)}")

        # Prepare inputs
        supervised_mask_local = supervised_mask # from prior cells
        X_gene_all = adata.obsm['X_gene_pca'][supervised_mask_local]
        tra_seq_all, trb_seq_all, seq_len = prepare_onehot_sequences(adata, supervised_mask_local)
        use_sequence = tra_seq_all is not None and trb_seq_all is not None
        
        if use_sequence:
            # concatenate TRA+TRB channels along the channel axis
            X_seq_all = np.concatenate([tra_seq_all, trb_seq_all], axis=2) # shape (N, seq_len, n_channels*2)
            n_channels_combined = X_seq_all.shape[2]
        else:
            X_seq_all = None
            n_channels_combined = None

        y_all = y_encoded
        
        # Robust column detection for patient_id
        patient_id_col_local = None
        if 'patient_id' in adata.obs.columns:
            patient_id_col_local = 'patient_id'
        elif 'Patient_ID' in adata.obs.columns:
            if 'patient_id' not in adata.obs.columns:
                adata.obs['patient_id'] = adata.obs['Patient_ID']
            patient_id_col_local = 'patient_id'
        elif 'PatientID' in adata.obs.columns:
            if 'patient_id' not in adata.obs.columns:
                adata.obs['patient_id'] = adata.obs['PatientID']
            patient_id_col_local = 'patient_id'
        else:
            raise KeyError("No patient ID column found (tried 'patient_id', 'Patient_ID', 'PatientID')")
        
        groups_all_local = np.array(adata.obs[patient_id_col_local][supervised_mask_local])

        # Force strict alignment across gene/sequence/labels/groups so sequence models cannot fail on index mismatch.
        lengths = [len(y_all), X_gene_all.shape[0], len(groups_all_local)]
        if use_sequence and X_seq_all is not None:
            lengths.append(X_seq_all.shape[0])
        common_n = int(min(lengths))
        if common_n <= 1:
            raise ValueError('Not enough aligned supervised samples for deep learning.')

        if X_gene_all.shape[0] != common_n:
            X_gene_all = X_gene_all[:common_n]
        if len(y_all) != common_n:
            y_all = y_all[:common_n]
        if len(groups_all_local) != common_n:
            groups_all_local = groups_all_local[:common_n]
        if use_sequence and X_seq_all is not None and X_seq_all.shape[0] != common_n:
            X_seq_all = X_seq_all[:common_n]

        try:
            entity_ids_all_local = np.asarray(adata.obs_names[supervised_mask_local], dtype=str)
        except Exception:
            entity_ids_all_local = np.asarray([f'cell_{i}' for i in range(len(y_all))], dtype=object)
        if len(entity_ids_all_local) >= common_n:
            entity_ids_all_local = entity_ids_all_local[:common_n]
        else:
            entity_ids_all_local = np.asarray([f'cell_{i}' for i in range(common_n)], dtype=object)

        processed_data_dir = Path('/kaggle/working/Processed_Data') if globals().get('IS_KAGGLE', False) else Path('Processed_Data')
        processed_data_dir.mkdir(parents=True, exist_ok=True)

        unique_patients = np.unique(groups_all_local)

        # Outer Validation strategy: Use GroupKFold (5 splits) if possible, else LOPO
        if len(unique_patients) >= 5:
            n_outer_splits = 5
            outer_cv = GroupKFold(n_splits=n_outer_splits)
            print(f"Using GroupKFold with {n_outer_splits} splits for outer validation (faster than LOPO).")
        else:
            outer_cv = LeaveOneGroupOut()
            print("Using LeaveOneGroupOut for outer validation (few patients).")
            
        dl_results_rows = []

        for feature_name in ['sequence_structure', 'comprehensive', 'tcr_enhanced']:
            # Select appropriate X inputs for DL
            print(f"\n=== DL evaluation using feature set: {feature_name} ===")
            if feature_name == 'sequence_structure' and use_sequence:
                # We will use gene PCs + sequence input
                use_gene = True
                use_seq = True
                X_gene = X_gene_all
                X_seq = X_seq_all
            elif feature_name == 'comprehensive':
                # use gene + reduced sequence PCA features if sequence onehot unavailable
                use_gene = True
                use_seq = use_sequence
                X_gene = X_gene_all
                X_seq = X_seq_all
            elif feature_name == 'tcr_enhanced' and use_sequence:
                use_gene = False
                use_seq = True
                X_gene = None
                X_seq = X_seq_all
            else:
                # fallback to gene-only MLP
                use_gene = True
                use_seq = False
                X_gene = X_gene_all
                X_seq = None

            # accumulators per architecture
            accum_arch = {}
            for arch in ['MLP','CNN','RNN','BiLSTM','Transformer']:
                accum_arch[arch] = {'y_true': [], 'y_pred': [], 'y_proba': [], 'groups': []}
            cleared_prediction_files = set()

            # Clearer Logging with TQDM
            splits = list(outer_cv.split(X_gene if X_gene is not None else np.zeros((len(y_all),1)), y_all, groups_all_local))
            print(f"  Starting validation with {len(splits)} folds...")
            
            for fold_idx, (train_idx, test_idx) in tqdm(enumerate(splits), total=len(splits), desc=f"CV {feature_name}"):
                held = np.unique(groups_all_local[test_idx])
                # print(f"Fold {fold_idx+1}/{len(splits)}")

                # Split inputs
                if use_gene:
                    X_tr_gene = X_gene[train_idx]
                    X_te_gene = X_gene[test_idx]
                    # Standard scaling fits only on training
                    scaler = StandardScaler().fit(X_tr_gene)
                    X_tr_gene_scaled = scaler.transform(X_tr_gene)
                    X_te_gene_scaled = scaler.transform(X_te_gene)
                else:
                    X_tr_gene_scaled = None
                    X_te_gene_scaled = None

                if use_seq:
                    X_tr_seq = X_seq[train_idx]
                    X_te_seq = X_seq[test_idx]
                else:
                    X_tr_seq = None
                    X_te_seq = None

                y_tr = y_all[train_idx]
                y_te = y_all[test_idx]
                groups_tr = groups_all_local[train_idx]

                # Compute class weights
                classes = np.unique(y_tr)
                cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_tr)
                class_weight_dict = {int(c): float(w) for c,w in zip(classes, cw)}

                # Train each architecture per fold so CNN/BiLSTM/Transformer always execute.
                for arch in ['MLP', 'CNN', 'RNN', 'BiLSTM', 'Transformer']:
                    if arch in ['CNN', 'RNN', 'BiLSTM', 'Transformer'] and not use_seq:
                        continue

                    # Pick architecture-specific config; keep deterministic and lightweight.
                    arch_cfgs = [cfg for cfg in grid_items if cfg[0] == arch]
                    if not arch_cfgs:
                        continue
                    best_cfg = arch_cfgs[0]
                    _, hu, dr, lr, bs, epochs = best_cfg

                    try:
                        fit_inputs = None
                        test_inputs = None
                        model = None

                        if arch == 'MLP':
                            if use_gene and X_tr_gene_scaled is not None:
                                model = create_compiled_model(build_mlp_graph, X_tr_gene_scaled.shape[1], hidden1=hu, hidden2=max(32, hu//2), dropout=dr, l2_reg=1e-3, lr=lr)
                                fit_inputs = X_tr_gene_scaled
                                test_inputs = X_te_gene_scaled
                            elif use_seq and X_tr_seq is not None:
                                X_tr_flat = X_tr_seq.reshape(X_tr_seq.shape[0], -1)
                                X_te_flat = X_te_seq.reshape(X_te_seq.shape[0], -1)
                                seq_scaler_full = StandardScaler().fit(X_tr_flat)
                                X_tr_flat_scaled = seq_scaler_full.transform(X_tr_flat)
                                X_te_flat_scaled = seq_scaler_full.transform(X_te_flat)
                                model = create_compiled_model(build_mlp_graph, X_tr_flat_scaled.shape[1], hidden1=hu, hidden2=max(32, hu//2), dropout=dr, l2_reg=1e-3, lr=lr)
                                fit_inputs = X_tr_flat_scaled
                                test_inputs = X_te_flat_scaled
                            else:
                                continue
                        elif arch == 'CNN':
                            model = create_compiled_model(build_cnn_graph, X_tr_seq.shape[1], X_tr_seq.shape[2], gene_dim=(X_tr_gene_scaled.shape[1] if use_gene else None), conv_filters=hu, kernel_size=5, dropout=dr, l2_reg=1e-3, lr=lr)
                            fit_inputs = [X_tr_seq, X_tr_gene_scaled] if use_gene else X_tr_seq
                            test_inputs = [X_te_seq, X_te_gene_scaled] if use_gene else X_te_seq
                        elif arch == 'RNN':
                            model = create_compiled_model(build_rnn_graph, X_tr_seq.shape[1], X_tr_seq.shape[2], gene_dim=(X_tr_gene_scaled.shape[1] if use_gene else None), rnn_units=max(32, hu//2), dropout=dr, l2_reg=1e-3, lr=lr)
                            fit_inputs = [X_tr_seq, X_tr_gene_scaled] if use_gene else X_tr_seq
                            test_inputs = [X_te_seq, X_te_gene_scaled] if use_gene else X_te_seq
                        elif arch == 'BiLSTM':
                            model = create_compiled_model(build_bilstm_graph, X_tr_seq.shape[1], X_tr_seq.shape[2], gene_dim=(X_tr_gene_scaled.shape[1] if use_gene else None), lstm_units=hu, dropout=dr, l2_reg=1e-3, lr=lr)
                            fit_inputs = [X_tr_seq, X_tr_gene_scaled] if use_gene else X_tr_seq
                            test_inputs = [X_te_seq, X_te_gene_scaled] if use_gene else X_te_seq
                        else:  # Transformer
                            model = create_compiled_model(build_transformer_graph, X_tr_seq.shape[1], X_tr_seq.shape[2], gene_dim=(X_tr_gene_scaled.shape[1] if use_gene else None), embed_dim=max(32, hu//2), num_heads=4, ff_dim=hu, dropout=dr, lr=lr)
                            fit_inputs = [X_tr_seq, X_tr_gene_scaled] if use_gene else X_tr_seq
                            test_inputs = [X_te_seq, X_te_gene_scaled] if use_gene else X_te_seq

                        es = keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=6, restore_best_weights=True, verbose=0)
                        rlr = ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=2, min_lr=1e-6, verbose=0)

                        try:
                            sw_full = np.array([class_weight_dict.get(int(yv), 1.0) for yv in y_tr], dtype='float32')
                            n_samples = len(y_tr)
                            if n_samples >= 8:
                                rng = np.random.RandomState(SEED + fold_idx)
                                idx = np.arange(n_samples)
                                rng.shuffle(idx)
                                n_val = max(1, int(n_samples * 0.1))
                                val_idx = idx[:n_val]
                                tr_idx_local = idx[n_val:]

                                def _slice_inputs(inp, indices):
                                    if isinstance(inp, (list, tuple)):
                                        return [x[indices] for x in inp]
                                    return inp[indices]

                                tr_inputs = _slice_inputs(fit_inputs, tr_idx_local)
                                va_inputs = _slice_inputs(fit_inputs, val_idx)
                                train_ds = make_tf_dataset(tr_inputs, y_tr[tr_idx_local], sample_weight=sw_full[tr_idx_local], batch_size=bs, shuffle=True)
                                val_ds = make_tf_dataset(va_inputs, y_tr[val_idx], sample_weight=sw_full[val_idx], batch_size=bs, shuffle=False)
                                model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=[es, rlr], verbose=0)
                            else:
                                train_ds = make_tf_dataset(fit_inputs, y_tr, sample_weight=sw_full, batch_size=bs, shuffle=True)
                                model.fit(train_ds, epochs=epochs, callbacks=[es, rlr], verbose=0)
                        except Exception:
                            model.fit(fit_inputs, y_tr, validation_split=0.1, epochs=epochs, batch_size=bs, class_weight=class_weight_dict, callbacks=[es, rlr], verbose=0)

                        try:
                            dl_model_dir = Path('Output/Models/DL')
                            dl_model_dir.mkdir(parents=True, exist_ok=True)
                            dl_model_path = dl_model_dir / f'dl_model_{feature_name}_{arch}_fold{fold_idx}.keras'
                            model.save(dl_model_path)
                        except Exception as e:
                            print(f"Failed to save DL model ({arch}): {e}")

                        y_test_proba = model.predict(test_inputs, verbose=0).flatten()
                        y_test_pred = (y_test_proba > 0.5).astype(int)
                    except Exception as e:
                        print(f"  Training/eval failed for fold {fold_idx} arch {arch}: {e}")
                        y_test_proba = np.zeros(len(y_te), dtype=float)
                        y_test_pred = np.zeros(len(y_te), dtype=int)
                    finally:
                        keras.backend.clear_session()
                        gc.collect()

                    held_patient_str = '|'.join(map(str, np.atleast_1d(held).tolist()))
                    sanitized_arch = arch.replace(' ', '_').replace('/', '_')
                    cell_detail_path = processed_data_dir / f'dl_cell_predictions_{feature_name}_{sanitized_arch}.csv'
                    patient_detail_path = processed_data_dir / f'dl_patient_predictions_{feature_name}_{sanitized_arch}.csv'
                    for _path in (cell_detail_path, patient_detail_path):
                        if _path not in cleared_prediction_files and _path.exists():
                            _path.unlink()
                        cleared_prediction_files.add(_path)

                    entity_slice = entity_ids_all_local[test_idx] if len(entity_ids_all_local) == len(y_all) else np.asarray([f'cell_{i}' for i in test_idx], dtype=object)
                    fold_pred_df = pd.DataFrame({
                        'model': arch,
                        'architecture': arch,
                        'feature_set': feature_name,
                        'evaluation_level': 'cell',
                        'fold_id': fold_idx + 1,
                        'held_out_patient': held_patient_str,
                        'patient': np.asarray(groups_all_local[test_idx]).astype(str),
                        'entity_id': np.asarray(entity_slice).astype(str),
                        'y_true': y_te,
                        'y_pred': y_test_pred,
                        'y_proba': y_test_proba,
                    })
                    fold_pred_df.to_csv(cell_detail_path, mode='a', header=not cell_detail_path.exists(), index=False)

                    patient_fold_df = fold_pred_df.groupby('patient', as_index=False).agg({'y_true': 'first', 'y_proba': 'mean'})
                    patient_fold_df['y_pred'] = (patient_fold_df['y_proba'] >= 0.5).astype(int)
                    patient_fold_df['model'] = arch
                    patient_fold_df['architecture'] = arch
                    patient_fold_df['feature_set'] = feature_name
                    patient_fold_df['evaluation_level'] = 'patient'
                    patient_fold_df['fold_id'] = fold_idx + 1
                    patient_fold_df['held_out_patient'] = patient_fold_df['patient'].astype(str)
                    patient_fold_df['entity_id'] = patient_fold_df['patient'].astype(str)
                    patient_fold_df = patient_fold_df[['model', 'architecture', 'feature_set', 'evaluation_level', 'fold_id', 'held_out_patient', 'patient', 'entity_id', 'y_true', 'y_pred', 'y_proba']]
                    patient_fold_df.to_csv(patient_detail_path, mode='a', header=not patient_detail_path.exists(), index=False)

                    accum_arch[arch]['y_true'].extend(y_te.tolist())
                    accum_arch[arch]['y_pred'].extend(y_test_pred.tolist())
                    accum_arch[arch]['y_proba'].extend(y_test_proba.tolist())
                    accum_arch[arch]['groups'].extend(groups_all_local[test_idx].tolist())

                # --- OOM FIX: release TF graph and numpy arrays between LOPO folds ---
                keras.backend.clear_session()
                gc.collect()

            # After LOPO folds compute aggregated metrics per architecture
            for arch, data in accum_arch.items():
                y_true_all = np.array(data['y_true'])
                y_pred_all = np.array(data['y_pred'])
                y_proba_all = np.array(data['y_proba'])
                if len(y_true_all) == 0:
                    continue
                acc = accuracy_score(y_true_all, y_pred_all)
                prec = precision_score(y_true_all, y_pred_all, zero_division=0)
                rec = recall_score(y_true_all, y_pred_all, zero_division=0)
                f1s = f1_score(y_true_all, y_pred_all, zero_division=0)
                try:
                    auc = roc_auc_score(y_true_all, y_proba_all)
                except Exception:
                    auc = float('nan')
                cm = confusion_matrix(y_true_all, y_pred_all)
                if cm.size == 4:
                    tn, fp, fn, tp = cm.ravel()
                    spec = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
                    npv = tn / (tn + fn) if (tn + fn) > 0 else float('nan')
                else:
                    spec = float('nan')
                    npv = float('nan')

                dl_results_rows.append({
                    'feature_set': feature_name,
                    'architecture': arch,
                    'model': arch,
                    'evaluation_level': 'cell',
                    'accuracy': acc,
                    'precision': prec,
                    'recall': rec,
                    'f1': f1s,
                    'auc': auc,
                    'specificity': spec,
                    'npv': npv,
                    'n_patients': len(unique_patients),
                    'n_cells': X_gene.shape[0] if X_gene is not None else (X_seq.shape[0] if X_seq is not None else 0),
                })

                # --- Patient-level aggregation for DL architecture ---
                try:
                    groups_arr = np.array(data.get('groups', []), dtype=object)
                    pred_df = pd.DataFrame({'patient': groups_arr, 'y_true': data['y_true'], 'y_proba': data['y_proba']})
                    patient_summary = pred_df.groupby('patient').agg({'y_proba': 'mean', 'y_true': 'first'}).reset_index()
                    patient_summary['y_pred'] = (patient_summary['y_proba'] >= 0.5).astype(int)
                    
                    # Assign vars before calc
                    y_true_pat = patient_summary['y_true'].values
                    y_pred_pat = patient_summary['y_pred'].values
                    y_proba_pat = patient_summary['y_proba'].values

                    acc_p = accuracy_score(y_true_pat, y_pred_pat)
                    prec_p = precision_score(y_true_pat, y_pred_pat, zero_division=0)
                    rec_p = recall_score(y_true_pat, y_pred_pat, zero_division=0)
                    f1s_p = f1_score(y_true_pat, y_pred_pat, zero_division=0)
                    try:
                        auc_p = roc_auc_score(y_true_pat, y_proba_pat)
                    except Exception:
                        auc_p = float('nan')
                    cm_p = confusion_matrix(y_true_pat, y_pred_pat)
                    if cm_p.size == 4:
                        tn, fp, fn, tp = cm_p.ravel()
                        spec_p = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
                        npv_p = tn / (tn + fn) if (tn + fn) > 0 else float('nan')
                    else:
                        spec_p = float('nan')
                        npv_p = float('nan')

                    dl_results_rows.append({
                        'feature_set': feature_name,
                        'architecture': arch,
                        'model': arch,
                        'evaluation_level': 'patient',
                        'accuracy': acc_p,
                        'precision': prec_p,
                        'recall': rec_p,
                        'f1': f1s_p,
                        'auc': auc_p,
                        'specificity': spec_p,
                        'npv': npv_p,
                        'n_patients': len(patient_summary),
                        'n_cells': X_gene.shape[0] if X_gene is not None else (X_seq.shape[0] if X_seq is not None else 0),
                    })
                except Exception as e:
                    # Skip if patient-level aggregation fails due to insufficient data
                    print(f"Patient aggregation failed for {arch}: {e}")
                    pass

        # Create final dataframes if data exists
        if dl_results_rows:
            dl_df = pd.DataFrame(dl_results_rows)
            output_path = processed_data_dir / 'dl_results.csv'
            processed_data_dir.mkdir(exist_ok=True)
            dl_df.to_csv(output_path, index=False)
            print(f"Deep learning results saved to: {output_path}")
            display(dl_df)
        else:
            print("No deep learning results to save (insufficient supervised data or model failures)")


Found 1 GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
  Mixed precision enabled for GPU.
  Using OneDeviceStrategy on GPU:0.
Number of replicas: 1
TensorFlow version: 2.19.0
Default DL batch size set to: 32
DL hyperparameter combinations per fold: 5
Patched models_eval['XGBoost'] to use GPU (method=device).
Patched models_eval['Random Forest'] to use n_jobs=-1.
Patched param_grids['XGBoost'] with GPU options (method=device).
Using LeaveOneGroupOut for outer validation (few patients).

=== DL evaluation using feature set: sequence_structure ===


I0000 00:00:1773081076.321365      25 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14317 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


  Starting validation with 4 folds...


CV sequence_structure:   0%|          | 0/4 [00:00<?, ?it/s]

I0000 00:00:1773081084.536952    3412 service.cc:152] XLA service 0x7c073c005f80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773081084.537008    3412 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773081085.111176    3412 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1773081088.183095    3412 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

CV comprehensive:   0%|          | 0/4 [00:00<?, ?it/s]


=== DL evaluation using feature set: tcr_enhanced ===
  Starting validation with 4 folds...


CV tcr_enhanced:   0%|          | 0/4 [00:00<?, ?it/s]

,feature_set,architecture,evaluation_level,accuracy,precision,recall,f1,auc,specificity,npv,n_patients,n_cells
0,sequence_structure,MLP,cell,0.915097,0.861628,0.917714,0.888787,0.972361,0.913562,0.949824,4,100067
1,sequence_structure,MLP,patient,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4,100067
2,sequence_structure,CNN,cell,0.909651,0.855027,0.909875,0.881599,0.969218,0.909519,0.945075,4,100067
3,sequence_structure,CNN,patient,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4,100067
4,sequence_structure,RNN,cell,0.910680,0.856544,0.910956,0.882912,0.969671,0.910518,0.945755,4,100067
5,sequence_structure,RNN,patient,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4,100067
6,sequence_structure,BiLSTM,cell,0.913148,0.857367,0.917741,0.886527,0.971397,0.910454,0.949677,4,100067
7,sequence_structure,BiLSTM,patient,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4,100067
8,sequence_structure,Transformer,cell,0.910130,0.854898,0.911632,0.882354,0.968504,0.909249,0.946073,4,100067
9,sequence_structure,Transformer,patient,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4,100067


CPU times: user 7h 29min 23s, sys: 32min 4s, total: 8h 1min 27s
Wall time: 6h 16min 2s


In [40]:
%%time
# --- Homology-aware, leakage-safe, lightweight sequence benchmark ---
# Sequence-similarity clusters are built first, then whole clusters are assigned to CV folds.

import gc
import hashlib
import random
import warnings
from collections import defaultdict
from pathlib import Path
import os
import multiprocessing

# Configure CPU threading to use all available cores
try:
    N_CPUS = multiprocessing.cpu_count()
except Exception:
    N_CPUS = os.cpu_count() or 1
os.environ['OMP_NUM_THREADS'] = str(N_CPUS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_CPUS)
os.environ['MKL_NUM_THREADS'] = str(N_CPUS)
os.environ['VECLIB_MAXIMUM_THREADS'] = str(N_CPUS)
os.environ['NUMEXPR_NUM_THREADS'] = str(N_CPUS)
try:
    import mkl
    mkl.set_num_threads(N_CPUS)
except Exception:
    pass
print(f'Configured CPU threading: {N_CPUS} threads')

import numpy as np
import pandas as pd
from scipy import sparse as sp
from sklearn.base import clone
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except Exception:
    HAS_SGKF = False

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

if 'adata' not in globals() or adata is None:
    raise NameError("adata is not defined. Please run upstream cells first.")

if '_ensure_response_and_patient' in globals():
    _ensure_response_and_patient(adata)

obs = adata.obs.copy()
if 'response' not in obs.columns:
    raise KeyError("Missing 'response' column in adata.obs.")

patient_col = next((c for c in ['patient_id', 'Patient_ID', 'PatientID'] if c in obs.columns), None)
if patient_col is None:
    raise KeyError("No patient ID column found (tried 'patient_id', 'Patient_ID', 'PatientID').")

valid_mask = obs['response'].astype(str).isin(['Responder', 'Non-Responder']).values
valid_indices = np.where(valid_mask)[0]

if valid_mask.sum() < 40:
    print(f"Insufficient supervised samples: {valid_mask.sum()}. Skipping benchmark.")
else:
    y_text_all = obs.loc[valid_mask, 'response'].astype(str).values
    le = LabelEncoder()
    y_all = le.fit_transform(y_text_all).astype(np.int64)

    tra_col = 'cdr3_TRA' if 'cdr3_TRA' in obs.columns else None
    trb_col = 'cdr3_TRB' if 'cdr3_TRB' in obs.columns else None
    if tra_col is None and trb_col is None:
        raise KeyError("Neither 'cdr3_TRA' nor 'cdr3_TRB' was found in adata.obs.")

    tra_vals = obs.loc[valid_mask, tra_col].fillna('').astype(str).str.upper().values if tra_col else np.array([''] * valid_mask.sum())
    trb_vals = obs.loc[valid_mask, trb_col].fillna('').astype(str).str.upper().values if trb_col else np.array([''] * valid_mask.sum())
    combined_seq_all = np.array([(a + 'X' + b).replace('NAN', '') for a, b in zip(tra_vals, trb_vals)], dtype=object)

    MAX_SAMPLES = 12000
    if len(y_all) > MAX_SAMPLES:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=MAX_SAMPLES, random_state=SEED)
        keep_idx, _ = next(sss.split(np.zeros(len(y_all)), y_all))
        keep_idx = np.sort(keep_idx)
    else:
        keep_idx = np.arange(len(y_all))

    y = y_all[keep_idx]
    y_text = y_text_all[keep_idx]
    combined_seq = combined_seq_all[keep_idx]
    selected_indices = valid_indices[keep_idx]

    if len(y_all) > MAX_SAMPLES:
        print(f"Subsampled to {len(y)} rows for lightweight benchmarking.")

    def _clean_seq(seq, max_len=96):
        aa = ''.join(ch for ch in str(seq) if ch.isalpha())
        if not aa:
            return 'X'
        return aa[:max_len]

    def seq_to_kmers(seq, k=3, max_len=96):
        s = _clean_seq(seq, max_len=max_len)
        if len(s) < k:
            return s
        return ' '.join(s[i:i + k] for i in range(len(s) - k + 1))

    def seq_to_kset(seq, k=3, max_len=96):
        s = _clean_seq(seq, max_len=max_len)
        if len(s) < k:
            return {s}
        return {s[i:i + k] for i in range(len(s) - k + 1)}

    kmer_corpus = [seq_to_kmers(s, k=3, max_len=96) for s in combined_seq]
    kmer_sets = [seq_to_kset(s, k=3, max_len=96) for s in combined_seq]

    tfidf = TfidfVectorizer(token_pattern=r'[^ ]+', min_df=2, max_features=6000, dtype=np.float32)
    X_sparse = tfidf.fit_transform(kmer_corpus)
    if X_sparse.shape[1] == 0:
        tfidf = TfidfVectorizer(token_pattern=r'[^ ]+', min_df=1, max_features=4000, dtype=np.float32)
        X_sparse = tfidf.fit_transform(kmer_corpus)

    svd_dim = int(max(8, min(192, X_sparse.shape[1] - 1))) if X_sparse.shape[1] > 1 else 8
    svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
    X_svd = svd.fit_transform(X_sparse).astype(np.float32)
    seq_len_feature = np.array([len(_clean_seq(s, max_len=96).replace('X', '')) for s in combined_seq], dtype=np.float32).reshape(-1, 1)
    X_dense = np.hstack([X_svd, seq_len_feature]).astype(np.float32)

    def cluster_sequences_by_similarity(kset_list, identity_threshold=0.35, max_postings=1200):
        # Approximate sequence identity threshold (~35%) with 3-mer Jaccard threshold.
        jaccard_threshold = max(0.18, identity_threshold - 0.15)
        n = len(kset_list)
        if n == 0:
            return np.array([], dtype=np.int32)
        parent = np.arange(n, dtype=np.int32)
        size = np.ones(n, dtype=np.int32)

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra == rb:
                return
            if size[ra] < size[rb]:
                ra, rb = rb, ra
            parent[rb] = ra
            size[ra] += size[rb]

        postings = defaultdict(list)
        for idx, tokens in enumerate(kset_list):
            for tok in tokens:
                postings[tok].append(idx)

        for i, ks_i in enumerate(kset_list):
            if not ks_i:
                continue
            cand_overlap = defaultdict(int)
            for tok in ks_i:
                ids = postings.get(tok, [])
                if len(ids) > max_postings:
                    continue
                for j in ids:
                    if j <= i:
                        continue
                    cand_overlap[j] += 1

            len_i = len(ks_i)
            for j, inter in cand_overlap.items():
                ks_j = kset_list[j]
                if not ks_j:
                    continue
                union_size = len_i + len(ks_j) - inter
                if union_size <= 0:
                    continue
                jacc = inter / union_size
                if jacc >= jaccard_threshold:
                    union(i, j)

        roots = np.fromiter((find(i) for i in range(n)), dtype=np.int32, count=n)
        _, groups = np.unique(roots, return_inverse=True)
        return groups.astype(np.int32)

    homology_groups = cluster_sequences_by_similarity(kmer_sets, identity_threshold=0.35)
    n_groups = len(np.unique(homology_groups))
    print(f"Homology-aware clustering created {n_groups} sequence-similarity groups (target identity threshold ~35%).")

    if n_groups < 2:
        print("Not enough homology groups for grouped validation. Skipping benchmark.")
    else:
        # --- Removed pretrained ESM2 baseline per request. Integrate canonical DeepTCR if available. ---
        class_counts = np.bincount(y)
        imbalance_ratio = (class_counts.min() / class_counts.max()) if class_counts.max() > 0 else 1.0
        use_balanced = imbalance_ratio < 0.8
        class_weight_mode = 'balanced' if use_balanced else None
        rf_class_weight_mode = 'balanced_subsample' if use_balanced else None
        print(f"Class balance ratio: {imbalance_ratio:.3f}; class weights enabled: {use_balanced}")

        # Standard classical model specs (no pretrained ESM features)
        model_specs = {
            'LogReg_kmer': {
                'X': X_sparse,
                'estimator': LogisticRegression(
                    max_iter=1800,
                    class_weight=class_weight_mode,
                    solver='saga',
                    random_state=SEED,
                    n_jobs=-1,
                ),
                'param_dist': {'C': [0.05, 0.1, 0.5, 1.0, 2.0, 5.0]},
            },
            'LinearSVM_kmer': {
                'X': X_sparse,
                'estimator': LinearSVC(class_weight=class_weight_mode, random_state=SEED),
                'param_dist': {'C': [0.05, 0.1, 0.5, 1.0, 2.0, 5.0]},
            },
            'RF_svd': {
                'X': X_dense,
                'estimator': RandomForestClassifier(
                    n_estimators=240,
                    class_weight=rf_class_weight_mode,
                    random_state=SEED,
                    n_jobs=-1,
                ),
                'param_dist': {
                    'max_depth': [None, 10, 20],
                    'min_samples_leaf': [1, 2, 4],
                    'max_features': ['sqrt', 0.5],
                },
            },
            'RBF_SVM_svd': {
                'X': X_dense,
                'estimator': Pipeline([
                    ('scaler', StandardScaler()),
                    ('clf', SVC(kernel='rbf', probability=True, class_weight=class_weight_mode, random_state=SEED)),
                ]),
                'param_dist': {
                    'clf__C': [0.5, 1.0, 2.0, 4.0],
                    'clf__gamma': ['scale', 0.05, 0.01],
                },
            },
        }

        # Prepare DeepTCR input files (sequences per sample and labels per sample)
        try:
            patient_vals_all = obs.loc[valid_mask, patient_col].astype(str).values
            patient_vals = patient_vals_all[keep_idx]
        except Exception:
            # Fallback: use unique index per row
            patient_vals = np.array([f"sample_{i}" for i in range(len(y))], dtype=object)

        seq_arr_clean = np.asarray([_clean_seq(s, max_len=96) for s in combined_seq], dtype=object)
        seq_df = pd.DataFrame({'sample_id': patient_vals, 'sequence': seq_arr_clean})
        seq_agg = seq_df.groupby(['sample_id', 'sequence']).size().reset_index(name='count')

        unique_patients = np.unique(patient_vals)
        labels_by_sample = {}
        for pid in unique_patients:
            idxs = np.where(patient_vals == pid)[0]
            labels_for_pid = y[idxs]
            lab = int(np.bincount(labels_for_pid).argmax()) if len(labels_for_pid) > 0 else 0
            labels_by_sample[pid] = lab
        labels_df = pd.DataFrame({'sample_id': list(labels_by_sample.keys()), 'label': list(labels_by_sample.values())})

        deeptcr_input_dir = Path('Processed_Data') / 'deeptcr_input'
        deeptcr_input_dir.mkdir(parents=True, exist_ok=True)
        seq_agg.to_csv(deeptcr_input_dir / 'deeptcr_sequences.csv', index=False)
        labels_df.to_csv(deeptcr_input_dir / 'deeptcr_labels.csv', index=False)
        print(f"Wrote DeepTCR input files to {deeptcr_input_dir}")

        # Attempt to import and run DeepTCR if installed; otherwise provide instructions.
        import importlib, subprocess, sys
        deeptcr_mod = None
        for name in ('DeepTCR', 'deeptcr', 'deep_tcr', 'deepTCR'):
            try:
                deeptcr_mod = importlib.import_module(name)
                print(f"Imported DeepTCR module '{name}'")
                break
            except Exception:
                deeptcr_mod = None
        if deeptcr_mod is None:
            print("DeepTCR not installed. To install, run:\n    pip install DeepTCR\nOr follow the project's GitHub installation instructions. DeepTCR integration skipped.")
        else:
            print("DeepTCR module found. Attempting to locate trainer API via introspection...")
            trainer_instance = None
            # Try common class name
            if hasattr(deeptcr_mod, 'DeepTCR'):
                try:
                    trainer_instance = getattr(deeptcr_mod, 'DeepTCR')()
                    print("Instantiated DeepTCR() from module.")
                except Exception as e:
                    print(f"Could not instantiate DeepTCR(): {e}")
                    trainer_instance = None
            # Fallback: try to find callable classes/functions
            if trainer_instance is None:
                for attr in dir(deeptcr_mod):
                    if attr.lower().startswith('deep') or 'train' in attr.lower():
                        obj = getattr(deeptcr_mod, attr)
                        if callable(obj):
                            try:
                                inst = obj()
                                trainer_instance = inst
                                print(f"Instantiated {attr}() from module.")
                                break
                            except Exception:
                                pass

            if trainer_instance is None:
                print("Could not create a DeepTCR trainer instance automatically. Please run training manually with the generated CSVs:")
                print(f"  sequences: {deeptcr_input_dir / 'deeptcr_sequences.csv'}")
                print(f"  labels:    {deeptcr_input_dir / 'deeptcr_labels.csv'}")
                print("Suggested pseudo-commands (replace with package-specific API):")
                print("from DeepTCR.DeepTCR import DeepTCR\nd = DeepTCR()\nd.Load_Data('deeptcr_sequences.csv','deeptcr_labels.csv')\nd.Train(epochs=20, outdir='deeptcr_model')")
            else:
                # Try to call common loader/train methods
                called_loader = False
                for load_m in ('Load_Data', 'LoadData', 'Load_Sequences', 'Load_TCR', 'Load_Immuno', 'load_data', 'load_sequences'):
                    if hasattr(trainer_instance, load_m):
                        try:
                            getattr(trainer_instance, load_m)(str(deeptcr_input_dir / 'deeptcr_sequences.csv'), str(deeptcr_input_dir / 'deeptcr_labels.csv'))
                            print(f"Called {load_m} with sequences/labels CSVs")
                            called_loader = True
                            break
                        except Exception as e:
                            print(f"Calling {load_m} failed: {e}")
                if called_loader:
                    for train_m in ('Train', 'train', 'Fit', 'fit'):
                        if hasattr(trainer_instance, train_m):
                            try:
                                getattr(trainer_instance, train_m)(epochs=20, outdir=str(deeptcr_input_dir / 'deeptcr_model'))
                                print(f"Called {train_m} on trainer (outdir: {deeptcr_input_dir / 'deeptcr_model'})")
                                break
                            except Exception as e:
                                print(f"Calling {train_m} failed: {e}")
                else:
                    print("Loaded DeepTCR trainer but couldn't automatically call loader/train methods. Inspect the trainer object and run training manually as in the suggested pseudo-commands above.")

        # Continue with classical benchmarking (model_specs) below.
        outer_splits = min(4, n_groups)
        if outer_splits < 2:
            print("Not enough homology groups for outer CV.")
        else:
            if HAS_SGKF:
                outer_cv = StratifiedGroupKFold(n_splits=outer_splits, shuffle=True, random_state=SEED)
                split_name = 'StratifiedGroupKFold over homology clusters'
            else:
                outer_cv = GroupKFold(n_splits=outer_splits)
                split_name = 'GroupKFold over homology clusters'

            N_ITER_SEARCH = 4
            print(f"Split strategy: {split_name}")
            print(f"Hyperparameter tuning: RandomizedSearchCV with n_iter={N_ITER_SEARCH}, inner grouped CV on train-only homology groups.")

            def safe_proba(estimator, X_eval):
                if hasattr(estimator, 'predict_proba'):
                    proba = estimator.predict_proba(X_eval)[:, 1]
                elif hasattr(estimator, 'decision_function'):
                    scores = estimator.decision_function(X_eval)
                    proba = 1.0 / (1.0 + np.exp(-scores))
                else:
                    proba = np.asarray(estimator.predict(X_eval), dtype=float)
                proba = np.asarray(proba, dtype=float).ravel()
                proba = np.nan_to_num(proba, nan=0.5, posinf=1.0, neginf=0.0)
                return np.clip(proba, 0.0, 1.0)

            def bootstrap_mean_ci(values, n_boot=300, alpha=0.95):
                vals = np.asarray(values, dtype=float)
                vals = vals[np.isfinite(vals)]
                if vals.size == 0:
                    return np.nan, np.nan, np.nan, np.nan
                if vals.size == 1:
                    return float(vals[0]), 0.0, float(vals[0]), float(vals[0])
                boots = np.empty(n_boot, dtype=np.float64)
                for i in range(n_boot):
                    sample = RNG.choice(vals, size=vals.size, replace=True)
                    boots[i] = sample.mean()
                lo = float(np.quantile(boots, (1 - alpha) / 2))
                hi = float(np.quantile(boots, 1 - (1 - alpha) / 2))
                return float(vals.mean()), float(vals.std(ddof=1)), lo, hi

            rows = []
            fold_auc_by_model = {}
            best_params_log = []

            for model_name, spec in model_specs.items():
                X_model = spec['X']
                base_est = spec['estimator']
                param_dist = spec['param_dist']

                y_true_all, y_pred_all, y_proba_all = [], [], []
                fold_aucs = []

                for fold_idx, (tr_idx, te_idx) in enumerate(outer_cv.split(np.zeros(len(y)), y, groups=homology_groups), start=1):
                    if np.intersect1d(homology_groups[tr_idx], homology_groups[te_idx]).size > 0:
                        raise RuntimeError(f"Leakage detected in fold {fold_idx}: shared homology groups.")

                    X_tr, X_te = X_model[tr_idx], X_model[te_idx]
                    y_tr, y_te = y[tr_idx], y[te_idx]
                    g_tr = homology_groups[tr_idx]

                    if len(np.unique(y_tr)) < 2:
                        const_pred = int(np.bincount(y_tr).argmax()) if len(y_tr) else 0
                        y_proba = np.full(len(y_te), float(const_pred), dtype=float)
                        y_pred = np.full(len(y_te), const_pred, dtype=int)
                        y_true_all.extend(y_te.tolist())
                        y_pred_all.extend(y_pred.tolist())
                        y_proba_all.extend(y_proba.tolist())
                        fold_aucs.append(np.nan)
                        continue

                    est = clone(base_est)
                    best_est = est
                    inner_n_groups = len(np.unique(g_tr))
                    inner_splits = min(3, inner_n_groups)

                    try:
                        if inner_splits >= 2 and len(param_dist) > 0:
                            if HAS_SGKF:
                                inner_cv = StratifiedGroupKFold(n_splits=inner_splits, shuffle=True, random_state=SEED + fold_idx)
                            else:
                                inner_cv = GroupKFold(n_splits=inner_splits)

                            space_size = int(np.prod([len(v) for v in param_dist.values()]))
                            n_iter = min(N_ITER_SEARCH, max(1, space_size))

                            search = RandomizedSearchCV(
                                estimator=est,
                                param_distributions=param_dist,
                                n_iter=n_iter,
                                scoring='roc_auc',
                                cv=inner_cv,
                                n_jobs=-1,
                                random_state=SEED,
                                refit=True,
                            )
                            search.fit(X_tr, y_tr, groups=g_tr)
                            best_est = search.best_estimator_
                            best_params_log.append({'model': model_name, 'fold': fold_idx, 'best_params': search.best_params_})
                        else:
                            best_est.fit(X_tr, y_tr)
                    except Exception as fit_err:
                        print(f"[{model_name}] fold {fold_idx}: fallback to direct fit ({fit_err})")
                        best_est = clone(base_est)
                        best_est.fit(X_tr, y_tr)

                    y_proba = safe_proba(best_est, X_te)
                    y_pred = (np.asarray(y_proba) >= 0.5).astype(int)

                    y_true_all.extend(y_te.tolist())
                    y_pred_all.extend(y_pred.tolist())
                    y_proba_all.extend(np.asarray(y_proba, dtype=float).tolist())

                    try:
                        fold_auc = roc_auc_score(y_te, y_proba)
                    except Exception:
                        fold_auc = np.nan
                    fold_aucs.append(fold_auc)

                y_true_all = np.asarray(y_true_all)
                y_pred_all = np.asarray(y_pred_all)
                y_proba_all = np.asarray(y_proba_all)

                auc_all = roc_auc_score(y_true_all, y_proba_all) if len(np.unique(y_true_all)) > 1 else np.nan
                mean_auc, std_auc, ci_lo, ci_hi = bootstrap_mean_ci(fold_aucs, n_boot=300, alpha=0.95)

                rows.append({
                    'model': model_name,
                    'n_samples': len(y_true_all),
                    'n_homology_groups': n_groups,
                    'accuracy': accuracy_score(y_true_all, y_pred_all),
                    'precision': precision_score(y_true_all, y_pred_all, zero_division=0),
                    'recall': recall_score(y_true_all, y_pred_all, zero_division=0),
                    'f1': f1_score(y_true_all, y_pred_all, zero_division=0),
                    'auc': auc_all,
                    'fold_auc_mean': mean_auc,
                    'fold_auc_std': std_auc,
                    'fold_auc_ci95_low': ci_lo,
                    'fold_auc_ci95_high': ci_hi,
                    'outer_folds': outer_splits,
                    'inner_search': f'RandomizedSearchCV(n_iter={N_ITER_SEARCH}, cv<=3, scoring=roc_auc)',
                    'imbalance_handling': 'class_weight enabled' if use_balanced else 'none',
                    'split_strategy': split_name,
                })
                fold_auc_by_model[model_name] = np.asarray(fold_aucs, dtype=float)

            results_df = pd.DataFrame(rows).sort_values('fold_auc_mean', ascending=False).reset_index(drop=True)
            results_df['pipeline_section'] = 'homology_sequence_benchmark'
            display(results_df)

            # Integrate benchmark and deep-learning outputs into one ML table when available.
            integrated_frames = [results_df.copy()]
            if 'dl_df' in globals() and isinstance(dl_df, pd.DataFrame) and len(dl_df) > 0:
                dl_merge = dl_df.copy()
                rename_map = {
                    'architecture': 'model',
                    'auc': 'fold_auc_mean',
                    'accuracy': 'accuracy',
                    'precision': 'precision',
                    'recall': 'recall',
                    'f1': 'f1',
                }
                for src_col, dst_col in rename_map.items():
                    if src_col in dl_merge.columns and src_col != dst_col:
                        dl_merge[dst_col] = dl_merge[src_col]
                dl_merge['pipeline_section'] = 'deep_learning'
                dl_merge['split_strategy'] = dl_merge.get('split_strategy', 'GroupKFold over patient groups')
                integrated_frames.append(dl_merge)

            ml_integrated_results_df = pd.concat(integrated_frames, ignore_index=True, sort=False)

            try:
                from scipy.stats import wilcoxon
                if len(results_df) >= 2:
                    m1 = results_df.loc[0, 'model']
                    m2 = results_df.loc[1, 'model']
                    a1 = fold_auc_by_model[m1]
                    a2 = fold_auc_by_model[m2]
                    keep = np.isfinite(a1) & np.isfinite(a2)
                    if keep.sum() >= 3:
                        _, pval = wilcoxon(a1[keep], a2[keep], zero_method='wilcox', alternative='two-sided')
                        print(f"Wilcoxon top-2 fold AUC ({m1} vs {m2}): p={pval:.4g}")
                    else:
                        print("Wilcoxon test skipped (insufficient finite fold AUC pairs).")
            except Exception as e:
                print(f"Statistical test skipped: {e}")

            out_dir = Path('Processed_Data')
            out_dir.mkdir(parents=True, exist_ok=True)
            results_df.to_csv(out_dir / 'homology_aware_benchmark_results.csv', index=False)
            ml_integrated_results_df.to_csv(out_dir / 'ml_integrated_results.csv', index=False)

            if best_params_log:
                pd.DataFrame(best_params_log).to_csv(out_dir / 'homology_aware_best_params.csv', index=False)

            # Lightweight biological insight extraction: ranked predictive genes with identifier mapping.
            try:
                X_gene_full = adata.X[selected_indices]
                n_gene_candidates = min(800, adata.n_vars)

                if sp.issparse(X_gene_full):
                    X_gene_full = X_gene_full.tocsr()
                    mean = np.asarray(X_gene_full.mean(axis=0)).ravel()
                    sq_mean = np.asarray(X_gene_full.multiply(X_gene_full).mean(axis=0)).ravel()
                    var = np.maximum(sq_mean - mean * mean, 0.0)
                else:
                    X_gene_arr = np.asarray(X_gene_full)
                    var = np.var(X_gene_arr, axis=0)

                top_var_idx = np.argpartition(var, -n_gene_candidates)[-n_gene_candidates:]
                top_var_idx = np.sort(top_var_idx)

                if sp.issparse(X_gene_full):
                    X_gene_top = X_gene_full[:, top_var_idx].toarray().astype(np.float32)
                else:
                    X_gene_top = np.asarray(X_gene_full[:, top_var_idx], dtype=np.float32)

                etc = ExtraTreesClassifier(
                    n_estimators=220,
                    random_state=SEED,
                    n_jobs=-1,
                    class_weight=rf_class_weight_mode,
                    max_features='sqrt',
                )
                etc.fit(X_gene_top, y)
                imp = np.asarray(etc.feature_importances_, dtype=np.float64)

                order = np.argsort(imp)[::-1]
                top_n = min(40, len(order))
                order = order[:top_n]

                gene_names = np.asarray(adata.var_names).astype(str)[top_var_idx][order]
                gene_imp = imp[order]
                gene_df = pd.DataFrame({
                    'rank': np.arange(1, top_n + 1),
                    'gene_id': gene_names,
                    'importance': gene_imp,
                })
                gene_df.to_csv(out_dir / 'top_predictive_genes.csv', index=False)
                display(gene_df.head(20))
                print("Saved ranked gene importance to Processed_Data/top_predictive_genes.csv")
            except Exception as e:
                print(f"Gene importance extraction skipped: {e}")

            # Expose integrated ML outputs for downstream cells.
            ml_results_table = ml_integrated_results_df.copy()
            globals()['ml_results_table'] = ml_results_table
            if 'fold_auc_mean' in ml_results_table.columns:
                ml_best_model = ml_results_table.sort_values('fold_auc_mean', ascending=False).head(1)
                globals()['ml_best_model'] = ml_best_model

            print("Saved benchmark table to Processed_Data/homology_aware_benchmark_results.csv")
            print("Saved integrated ML table to Processed_Data/ml_integrated_results.csv")
            print("Benchmark complete.")

Configured CPU threading: 4 threads
Response distribution: {'Non-Responder': 63074, 'Responder': 36993}
Patient_id coverage: 100067/100067
Subsampled to 12000 rows for lightweight benchmarking.
Homology-aware clustering created 12000 sequence-similarity groups (target identity threshold ~35%).
Class balance ratio: 0.586; class weights enabled: True
Wrote DeepTCR input files to Processed_Data/deeptcr_input
DeepTCR not installed. To install, run:
    pip install DeepTCR
Or follow the project's GitHub installation instructions. DeepTCR integration skipped.
Split strategy: StratifiedGroupKFold over homology clusters
Hyperparameter tuning: RandomizedSearchCV with n_iter=4, inner grouped CV on train-only homology groups.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 

,model,n_samples,n_homology_groups,accuracy,precision,recall,f1,auc,fold_auc_mean,fold_auc_std,fold_auc_ci95_low,fold_auc_ci95_high,outer_folds,inner_search,imbalance_handling,split_strategy,pipeline_section
0,LogReg_kmer,12000,12000,0.369667,0.369667,1.0,0.539791,0.500000,0.5,0.0,0.5,0.5,4,"RandomizedSearchCV(n_iter=4, cv<=3, scoring=ro...",class_weight enabled,StratifiedGroupKFold over homology clusters,homology_sequence_benchmark
1,LinearSVM_kmer,12000,12000,0.500000,0.369667,0.5,0.425067,0.499911,0.5,0.0,0.5,0.5,4,"RandomizedSearchCV(n_iter=4, cv<=3, scoring=ro...",class_weight enabled,StratifiedGroupKFold over homology clusters,homology_sequence_benchmark
2,RF_svd,12000,12000,0.630333,0.000000,0.0,0.000000,0.499911,0.5,0.0,0.5,0.5,4,"RandomizedSearchCV(n_iter=4, cv<=3, scoring=ro...",class_weight enabled,StratifiedGroupKFold over homology clusters,homology_sequence_benchmark
3,RBF_SVM_svd,12000,12000,0.630333,0.000000,0.0,0.000000,0.499732,0.5,0.0,0.5,0.5,4,"RandomizedSearchCV(n_iter=4, cv<=3, scoring=ro...",class_weight enabled,StratifiedGroupKFold over homology clusters,homology_sequence_benchmark


Wilcoxon top-2 fold AUC (LogReg_kmer vs LinearSVM_kmer): p=1


,rank,gene_id,importance
0,1,Human_week7,0.112325
1,2,Human_week1,0.092429
2,3,Human-Totalseq-C0251,0.084479
3,4,Human-Totalseq-C0252,0.082281
4,5,Human-Totalseq-C0253,0.073906
5,6,Human_week15,0.071592
6,7,JUND,0.030478
7,8,MT-ND3,0.016156
8,9,TAF10,0.014035
9,10,MT-ATP8,0.013794


Saved ranked gene importance to Processed_Data/top_predictive_genes.csv
Saved benchmark table to Processed_Data/homology_aware_benchmark_results.csv
Saved integrated ML table to Processed_Data/ml_integrated_results.csv
Benchmark complete.
CPU times: user 5min 48s, sys: 3.77 s, total: 5min 51s
Wall time: 8min 43s


## Publication-Quality Figures for Science Fair Presentation

The following cells generate 12 figures that tell a complete story of our analysis:

**Data Exploration (Figures 1–5):** Pipeline overview, UMAP clustering, quality control, PCA analysis, and patient cohort composition.

**Model Performance (Figures 6–8):** Comprehensive model comparison with deep learning models prominently featured, DL architecture deep-dive, and multi-metric radar charts.

**Feature Analysis (Figures 9–12):** SHAP-based feature importance from the best DL model, multi-model gene importance consensus, TCR sequence analysis, and cross-model agreement.

All figures use **actual computed results** from upstream cells. No placeholder data.

In [ ]:
# Helper: Load All Model Results & Configure Plotting
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'figure.titlesize': 16, 'figure.dpi': 150, 'figure.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
})

# Consistent color scheme
DL_COLORS = {'MLP': '#7B1FA2', '1D-CNN': '#C2185B', 'CNN': '#C2185B',
             'RNN': '#1565C0', 'BiLSTM': '#0277BD', 'Transformer': '#00695C'}
ML_COLORS = {'XGBoost': '#D84315', 'Random Forest': '#E65100', 'GradientBoosting': '#BF360C',
             'GradientBoost': '#BF360C', 'SVM': '#FF8F00', 'Logistic Regression': '#FFA000',
             'Logistic Reg.': '#FFA000', 'Decision Tree': '#FFB300',
             'RF_svd': '#E65100', 'LogReg_kmer': '#FFA000', 'LinearSVM_kmer': '#FF8F00',
             'RBF_SVM_svd': '#FF8F00', 'Extra Trees': '#E65100', 'ExtraTrees': '#E65100'}
ALL_COLORS = {**ML_COLORS, **DL_COLORS}

DL_MODELS = {'MLP', '1D-CNN', 'CNN', 'RNN', 'BiLSTM', 'Transformer'}
ENSEMBLE_MODELS = {'XGBoost', 'Random Forest', 'GradientBoosting', 'GradientBoost', 'RF_svd', 'Extra Trees'}

def get_model_color(name):
    return ALL_COLORS.get(name, '#6A1B9A' if name in DL_MODELS else '#E65100')

def get_model_type(name):
    if name in DL_MODELS: return 'Deep Learning'
    if name in ENSEMBLE_MODELS: return 'Ensemble ML'
    return 'Traditional ML'

def load_all_model_results():
    """Load ML/DL results from CSV files or in-memory variables."""
    frames = []
    csv_files = ['ml_integrated_results.csv', 'lopo_results.csv', 'dl_results.csv',
                 'homology_aware_benchmark_results.csv']
    for csv_name in csv_files:
        for base_dir in ['Processed_Data', '../Processed_Data', '.']:
            p = Path(base_dir) / csv_name
            if p.exists():
                try:
                    df = pd.read_csv(p)
                    frames.append(df)
                    print(f"  Loaded {len(df)} rows from {p}")
                except Exception:
                    pass
                break
    for vname in ['ml_integrated_results_df', 'results_df']:
        if vname in globals() and isinstance(globals()[vname], pd.DataFrame) and len(globals()[vname]) > 0:
            frames.append(globals()[vname].copy())
    for vname in ['lopo_summary_rows', 'dl_results_rows']:
        if vname in globals() and isinstance(globals()[vname], list) and len(globals()[vname]) > 0:
            frames.append(pd.DataFrame(globals()[vname]))
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True, sort=False)
    if 'architecture' in df.columns:
        df['model'] = df['architecture'].combine_first(df.get('model', pd.Series(dtype=str)))
    if 'fold_auc_mean' in df.columns:
        if 'auc' not in df.columns:
            df['auc'] = df['fold_auc_mean']
        else:
            df['auc'] = df['auc'].combine_first(df['fold_auc_mean'])
    if 'fold_auc_std' in df.columns:
        if 'auc_std' not in df.columns:
            df['auc_std'] = df['fold_auc_std']
        else:
            df['auc_std'] = df['auc_std'].combine_first(df['fold_auc_std'])
    dedup_cols = [c for c in ['model', 'feature_set', 'evaluation_level'] if c in df.columns]
    if dedup_cols:
        df = df.drop_duplicates(subset=dedup_cols, keep='last')
    return df

def get_best_results(df, level='cell'):
    """Get best result per model (highest AUC), optionally filtered by evaluation level."""
    if df is None or len(df) == 0:
        return None
    sub = df.copy()
    if 'evaluation_level' in sub.columns and level:
        mask = sub['evaluation_level'].astype(str).str.contains(level, case=False, na=True)
        if mask.sum() > 0:
            sub = sub[mask]
    if 'model' not in sub.columns or 'auc' not in sub.columns:
        return None
    return sub.sort_values('auc', ascending=False).drop_duplicates('model', keep='first').reset_index(drop=True)

print("Loading model results...")
all_results_df = load_all_model_results()
if all_results_df is not None:
    best_per_model = get_best_results(all_results_df)
    print(f"\nTotal: {len(all_results_df)} result rows")
    if best_per_model is not None:
        print(f"Unique models: {sorted(best_per_model['model'].unique())}")
        dl_mask = best_per_model['model'].isin(DL_MODELS)
        if dl_mask.any():
            best_dl = best_per_model[dl_mask].sort_values('auc', ascending=False).iloc[0]
            print(f"Best DL model: {best_dl['model']} (AUC={best_dl['auc']:.4f})")
else:
    best_per_model = None
    print("WARNING: No model results available. Run upstream cells first.")

Loading model results...
  Loaded 34 rows from Processed_Data/ml_integrated_results.csv
  Loaded 8 rows from Processed_Data/lopo_results.csv
  Loaded 4 rows from Processed_Data/homology_aware_benchmark_results.csv

Total: 42 result rows
Unique models: ['BiLSTM', 'CNN', 'Decision Tree', 'Logistic Regression', 'MLP', 'RNN', 'Random Forest', 'Transformer', 'XGBoost']
Best DL model: MLP (AUC=0.9735)


In [ ]:
# Figure 1: Analysis Pipeline Overview
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16); ax.set_ylim(0, 10); ax.axis('off')
fig.patch.set_facecolor('white')

colors = {'data': '#2E7D32', 'process': '#1565C0', 'feature': '#6A1B9A',
          'dl': '#C62828', 'ml': '#E65100', 'valid': '#37474F'}

def box(x, y, w, h, text, color, fs=9):
    b = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.12",
                       facecolor=color, edgecolor='#333', linewidth=1.5, alpha=0.88)
    ax.add_patch(b)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', fontsize=fs,
            fontweight='bold', color='white', multialignment='center')

def arrow(x1,y1,x2,y2):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color='#455A64', lw=2))

# Title
ax.text(8, 9.7, 'Figure 1: Analysis Pipeline', ha='center', fontsize=18, fontweight='bold')
ax.text(8, 9.3, 'HR+ Breast Cancer scRNA-seq + TCR-seq → Immunotherapy Response Prediction',
        ha='center', fontsize=11, color='#616161', style='italic')

# Row 1: Data Sources
box(0.5, 8, 3.5, 1, 'scRNA-seq (10x)\nGSE300475\n11 PBMC Samples', colors['data'])
box(4.5, 8, 3.5, 1, 'TCR-seq\nTRA/TRB CDR3\nContig Annotations', colors['data'])
box(9, 8, 3, 1, 'Clinical Metadata\n4 Patients × 3 Timepoints\nRCB Response Labels', colors['data'])
box(12.5, 8, 3, 1, 'Feature Reference\nGene IDs\nBarcode Mapping', colors['data'])

# Arrows down
for x in [2.25, 6.25, 10.5, 14]: arrow(x, 8, 8, 7.2)

# Row 2: Processing
box(1, 6, 4, 1, 'QC & Normalization\nmin_genes=200 | Log1p | HVG (n=1500)', colors['process'])
box(5.5, 6, 4.5, 1, 'TCR Integration\nBarcode Matching | CDR3 Extraction\nOne-Hot | K-mer (k=3) | Physicochemical', colors['process'])
box(10.5, 6, 5, 1, 'Feature Engineering\nGene PCA (50) | SVD (30) | UMAP\n4 Feature Configurations', colors['feature'])

for x in [3, 7.75, 13]: arrow(x, 6, 8, 5.2)

# Row 3: Models (DL prominent, larger boxes)
box(0.3, 3.5, 4, 1.2, '🧠 Deep Learning\nMLP | 1D-CNN | RNN\nBiLSTM | Transformer', colors['dl'], fs=10)
box(4.8, 3.5, 3.5, 1.2, 'Ensemble ML\nXGBoost | Random Forest\nGradient Boosting', colors['ml'])
box(8.8, 3.5, 3.5, 1.2, 'Traditional ML\nLogistic Reg. | SVM\nDecision Tree', colors['ml'])
box(12.8, 3.5, 2.8, 1.2, 'Unsupervised\nLeiden Clustering\nMulti-Resolution', colors['process'])

for x in [2.3, 6.55, 10.55, 14.2]: arrow(x, 3.5, 8, 2.8)

# Row 4: Validation
box(3, 1.5, 10, 1, 'LOPO Cross-Validation (Leakage-Free)\nGroupKFold Inner Tuning | Class-Weighted | Cell + Patient-Level Evaluation', colors['valid'], fs=11)

arrow(8, 1.5, 8, 0.8)

# Row 5: Output
box(2, 0, 12, 0.7, 'Response Prediction | SHAP Feature Importance | Gene Ranking | TCR Motif Analysis', colors['valid'], fs=10)

# Legend
legend_items = [mpatches.Patch(fc=c, ec='#333', label=l) for l,c in
    [('Data Sources',colors['data']),('Processing',colors['process']),
     ('Feature Eng.',colors['feature']),('Deep Learning',colors['dl']),
     ('Traditional ML',colors['ml']),('Validation',colors['valid'])]]
ax.legend(handles=legend_items, loc='lower left', fontsize=9, ncol=6, framealpha=0.9,
          bbox_to_anchor=(0.0, -0.06))

plt.tight_layout()
plt.savefig('Figure1_Pipeline_Overview.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("Figure 1 saved.")

Figure 1 saved.


In [ ]:
# Figure 4: UMAP Clusters With Cell Counts
import json
import os
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse as sp
from scipy.stats import spearmanr
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

SEED = 42
RNG = np.random.default_rng(SEED)
CELL_PREDICTION_GLOBS = [
    'lopo_cell_predictions_*.csv',
    'dl_cell_predictions_*.csv',
]
PATIENT_PREDICTION_GLOBS = [
    'lopo_patient_predictions_*.csv',
    'dl_patient_predictions_*.csv',
]
RESULT_TABLES = [
    'lopo_results.csv',
    'dl_results.csv',
    'ml_integrated_results.csv',
    'homology_aware_benchmark_results.csv',
]

def get_processed_data_dir() -> Path:
    is_kaggle = os.path.exists('/kaggle/input') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
    out_dir = Path('/kaggle/working/Processed_Data') if is_kaggle else Path('Processed_Data')
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'figures').mkdir(parents=True, exist_ok=True)
    return out_dir

def get_figures_dir() -> Path:
    return get_processed_data_dir() / 'figures'

def _to_builtin(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): _to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if pd.isna(value):
        return None
    return value

def _safe_slug(value: Any) -> str:
    text = ''.join(ch if str(ch).isalnum() or ch in ('_', '-') else '_' for ch in str(value))
    text = text.strip('_')
    return text or 'unknown'

def _standardize_prediction_df(df: pd.DataFrame, evaluation_level: str) -> pd.DataFrame:
    out = df.copy()
    if 'architecture' in out.columns and 'model' not in out.columns:
        out['model'] = out['architecture']
    elif 'architecture' in out.columns:
        out['model'] = out['model'].fillna(out['architecture'])
    if 'feature_set' not in out.columns:
        out['feature_set'] = 'unknown'
    if 'evaluation_level' not in out.columns:
        out['evaluation_level'] = evaluation_level
    if 'held_out_patient' not in out.columns and 'patient' in out.columns:
        out['held_out_patient'] = out['patient'].astype(str)
    if 'fold_id' in out.columns:
        out['fold_id'] = pd.to_numeric(out['fold_id'], errors='coerce')
    for col in ('y_true', 'y_pred', 'y_proba'):
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    return out

def load_prediction_details(evaluation_level: str | None = None) -> pd.DataFrame:
    out_dir = get_processed_data_dir()
    frames: list[pd.DataFrame] = []
    patterns = []
    if evaluation_level in (None, 'cell'):
        patterns.extend(CELL_PREDICTION_GLOBS)
    if evaluation_level in (None, 'patient'):
        patterns.extend(PATIENT_PREDICTION_GLOBS)

    for pattern in patterns:
        for path in sorted(out_dir.glob(pattern)):
            try:
                level = 'patient' if 'patient' in path.name else 'cell'
                df = pd.read_csv(path)
                if len(df) == 0:
                    continue
                frames.append(_standardize_prediction_df(df, evaluation_level=level))
            except Exception as exc:
                print(f'Could not load prediction detail file {path}: {exc}')

    if not frames:
        return pd.DataFrame()
    combined = pd.concat(frames, ignore_index=True, sort=False)
    desired_order = [
        'model', 'feature_set', 'evaluation_level', 'fold_id', 'held_out_patient',
        'patient', 'entity_id', 'y_true', 'y_pred', 'y_proba',
    ]
    cols = [c for c in desired_order if c in combined.columns] + [c for c in combined.columns if c not in desired_order]
    return combined[cols]

def load_result_tables() -> pd.DataFrame:
    out_dir = get_processed_data_dir()
    frames: list[pd.DataFrame] = []
    for name in RESULT_TABLES:
        path = out_dir / name
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path)
            if 'architecture' in df.columns and 'model' not in df.columns:
                df['model'] = df['architecture']
            elif 'architecture' in df.columns:
                df['model'] = df['model'].fillna(df['architecture'])
            frames.append(df)
        except Exception as exc:
            print(f'Could not load result table {path}: {exc}')
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, sort=False)

def _compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray) -> dict[str, float]:
    metrics: dict[str, float] = {}
    if len(y_true) == 0:
        return {k: float('nan') for k in ('accuracy', 'precision', 'recall', 'f1', 'specificity', 'auc')}
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
    metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
    try:
        metrics['auc'] = roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else float('nan')
    except Exception:
        metrics['auc'] = float('nan')
    try:
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    except Exception:
        metrics['specificity'] = float('nan')
    return metrics

def summarize_fold_metrics(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out_dir = get_processed_data_dir()
    if predictions.empty:
        empty_cols = ['model', 'feature_set', 'evaluation_level', 'fold_id', 'n_samples']
        empty_fold = pd.DataFrame(columns=empty_cols)
        empty_summary = pd.DataFrame(columns=['model', 'feature_set', 'evaluation_level'])
        empty_fold.to_csv(out_dir / 'loocv_fold_metrics.csv', index=False)
        empty_summary.to_csv(out_dir / 'loocv_metric_summary_mean_sd.csv', index=False)
        return empty_fold, empty_summary

    rows: list[dict[str, Any]] = []
    group_cols = ['model', 'feature_set', 'evaluation_level', 'fold_id']
    for keys, fold_df in predictions.groupby(group_cols, dropna=False):
        row = {col: val for col, val in zip(group_cols, keys)}
        y_true = fold_df['y_true'].to_numpy(dtype=float)
        y_pred = fold_df['y_pred'].to_numpy(dtype=float)
        y_proba = fold_df['y_proba'].to_numpy(dtype=float)
        row['n_samples'] = int(len(fold_df))
        row['n_patients'] = int(fold_df['patient'].nunique()) if 'patient' in fold_df.columns else np.nan
        row.update(_compute_binary_metrics(y_true, y_pred, y_proba))
        rows.append(row)

    fold_metrics = pd.DataFrame(rows).sort_values(['evaluation_level', 'model', 'feature_set', 'fold_id']).reset_index(drop=True)

    summary_rows: list[dict[str, Any]] = []
    metric_cols = ['accuracy', 'precision', 'recall', 'f1', 'specificity', 'auc']
    for keys, model_df in fold_metrics.groupby(['model', 'feature_set', 'evaluation_level'], dropna=False):
        row = {col: val for col, val in zip(['model', 'feature_set', 'evaluation_level'], keys)}
        row['n_folds'] = int(model_df['fold_id'].nunique()) if 'fold_id' in model_df.columns else int(len(model_df))
        row['n_samples_total'] = int(model_df['n_samples'].sum()) if 'n_samples' in model_df.columns else np.nan
        for metric in metric_cols:
            values = model_df[metric].dropna().to_numpy(dtype=float)
            row[f'{metric}_mean'] = float(np.mean(values)) if values.size else float('nan')
            row[f'{metric}_sd'] = float(np.std(values, ddof=1)) if values.size > 1 else (0.0 if values.size == 1 else float('nan'))
        summary_rows.append(row)

    summary = pd.DataFrame(summary_rows).sort_values(['evaluation_level', 'auc_mean', 'model'], ascending=[True, False, True]).reset_index(drop=True)
    fold_metrics.to_csv(out_dir / 'loocv_fold_metrics.csv', index=False)
    summary.to_csv(out_dir / 'loocv_metric_summary_mean_sd.csv', index=False)
    return fold_metrics, summary

def create_annotated_umap_figure(adata) -> dict[str, Any]:
    out_dir = get_processed_data_dir()
    figures_dir = out_dir / 'figures'

    if 'X_umap' not in adata.obsm:
        try:
            import scanpy as sc

            if 'X_pca' not in adata.obsm:
                sc.pp.pca(adata, n_comps=min(50, max(6, adata.n_obs - 1)))
            if 'neighbors' not in adata.uns:
                n_pcs = min(50, adata.obsm['X_pca'].shape[1])
                sc.pp.neighbors(adata, n_neighbors=min(15, max(2, adata.n_obs - 1)), n_pcs=n_pcs)
            sc.tl.umap(adata, random_state=SEED)
        except Exception as exc:
            raise RuntimeError(f'Could not compute UMAP coordinates: {exc}') from exc

    if 'leiden' not in adata.obs.columns:
        raise KeyError("`adata.obs['leiden']` is required for cluster annotations.")

    umap = np.asarray(adata.obsm['X_umap'])
    labels = adata.obs['leiden'].astype(str).to_numpy()
    unique_labels = sorted(set(labels), key=lambda val: int(val) if str(val).isdigit() else str(val))

    counts_rows: list[dict[str, Any]] = []
    fig, ax = plt.subplots(figsize=(10, 8))
    cmap = plt.cm.get_cmap('tab20', max(len(unique_labels), 2))
    color_lookup = {label: idx for idx, label in enumerate(unique_labels)}
    ax.scatter(
        umap[:, 0],
        umap[:, 1],
        c=[color_lookup[label] for label in labels],
        cmap=cmap,
        s=4,
        alpha=0.55,
        linewidths=0,
        rasterized=True,
    )

    for label in unique_labels:
        mask = labels == label
        cluster_coords = umap[mask]
        cell_count = int(mask.sum())
        centroid_x = float(cluster_coords[:, 0].mean())
        centroid_y = float(cluster_coords[:, 1].mean())
        counts_rows.append({
            'cluster_id': label,
            'cell_count': cell_count,
            'umap1_centroid': centroid_x,
            'umap2_centroid': centroid_y,
        })
        ax.text(
            centroid_x,
            centroid_y,
            f'{label} (n={cell_count:,})',
            fontsize=9,
            fontweight='bold',
            ha='center',
            va='center',
            bbox={'boxstyle': 'round,pad=0.2', 'fc': 'white', 'alpha': 0.9, 'ec': '#555'},
        )

    ax.set_title('Figure 4: UMAP Clusters With Cell Counts', fontsize=15, fontweight='bold')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_aspect('equal', adjustable='datalim')
    ax.grid(alpha=0.15)
    fig.tight_layout()

    counts_df = pd.DataFrame(counts_rows).sort_values('cluster_id')
    counts_path = out_dir / 'umap_cluster_counts.csv'
    fig_path = figures_dir / 'figure4_umap_clusters_annotated.png'
    counts_df.to_csv(counts_path, index=False)
    fig.savefig(fig_path, dpi=600, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)

    return {
        'cluster_counts': counts_df,
        'cluster_counts_path': counts_path,
        'figure_path': fig_path,
    }

def _get_pc6_scores(adata) -> np.ndarray:
    if 'X_pca' in adata.obsm and np.asarray(adata.obsm['X_pca']).shape[1] >= 6:
        return np.asarray(adata.obsm['X_pca'])[:, 5]

    if sp.issparse(adata.X):
        max_components = min(20, adata.n_vars, max(6, adata.n_obs - 1))
        if max_components < 6:
            raise ValueError('Not enough components available to compute PC6 from sparse matrix.')
        svd = TruncatedSVD(n_components=max_components, random_state=SEED)
        return svd.fit_transform(adata.X)[:, 5]

    matrix = np.asarray(adata.X, dtype=float)
    if matrix.ndim != 2 or matrix.shape[1] < 6:
        raise ValueError('Not enough features available to compute PC6.')
    if np.isnan(matrix).any():
        col_means = np.nanmean(matrix, axis=0)
        nan_rows, nan_cols = np.where(np.isnan(matrix))
        matrix[nan_rows, nan_cols] = col_means[nan_cols]
    max_components = min(20, matrix.shape[1], max(6, matrix.shape[0] - 1))
    if max_components < 6:
        raise ValueError('Not enough observations available to compute PC6.')
    pca = PCA(n_components=max_components, random_state=SEED)
    return pca.fit_transform(matrix)[:, 5]

def compute_pc6_mito_correlation(adata, n_permutations: int = 2000) -> dict[str, Any]:
    out_dir = get_processed_data_dir()
    figures_dir = out_dir / 'figures'

    if 'pct_counts_mt' not in adata.obs.columns:
        raise KeyError("`adata.obs['pct_counts_mt']` is required for the mitochondrial correlation analysis.")

    pc6 = _get_pc6_scores(adata)
    mito = pd.to_numeric(adata.obs['pct_counts_mt'], errors='coerce').to_numpy(dtype=float)
    mask = np.isfinite(pc6) & np.isfinite(mito)
    pc6 = pc6[mask]
    mito = mito[mask]
    if pc6.size < 3:
        raise ValueError('Not enough valid observations to compute the PC6 vs mitochondrial correlation.')

    rho, _ = spearmanr(pc6, mito)
    permuted = np.empty(n_permutations, dtype=float)
    for idx in range(n_permutations):
        permuted[idx] = spearmanr(pc6, RNG.permutation(mito))[0]
    p_value = float((np.sum(np.abs(permuted) >= abs(rho)) + 1) / (n_permutations + 1))

    corr_df = pd.DataFrame([{
        'method': 'spearman_permutation',
        'rho': float(rho),
        'permutation_p_value': p_value,
        'n_cells': int(pc6.size),
        'n_permutations': int(n_permutations),
    }])
    corr_path = out_dir / 'pc6_mito_correlation.csv'
    corr_df.to_csv(corr_path, index=False)

    fig, ax = plt.subplots(figsize=(8, 6))
    hb = ax.hexbin(pc6, mito, gridsize=55, cmap='viridis', mincnt=1)
    fig.colorbar(hb, ax=ax, label='Cell density')
    ax.set_title('PC6 vs Mitochondrial Percentage', fontsize=14, fontweight='bold')
    ax.set_xlabel('PC6 score')
    ax.set_ylabel('pct_counts_mt')
    ax.text(
        0.02,
        0.98,
        f'Spearman rho = {rho:.3f}Permutation p = {p_value:.4g}N = {pc6.size:,}',
        transform=ax.transAxes,
        va='top',
        ha='left',
        bbox={'boxstyle': 'round,pad=0.25', 'fc': 'white', 'alpha': 0.9, 'ec': '#555'},
    )
    fig.tight_layout()
    fig_path = figures_dir / 'pc6_vs_pct_counts_mt.png'
    fig.savefig(fig_path, dpi=400, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)

    return {
        'rho': float(rho),
        'p_value': p_value,
        'n_cells': int(pc6.size),
        'csv_path': corr_path,
        'figure_path': fig_path,
    }

def create_roc_figure_with_ci(predictions: pd.DataFrame | None = None, evaluation_level: str = 'cell', n_bootstrap: int = 1000) -> dict[str, Any]:
    out_dir = get_processed_data_dir()
    figures_dir = out_dir / 'figures'
    predictions = load_prediction_details() if predictions is None else predictions

    if predictions.empty:
        empty = pd.DataFrame()
        empty.to_csv(out_dir / 'figure6_roc_curve_data.csv', index=False)
        empty.to_csv(out_dir / 'figure6_roc_bootstrap_summary.csv', index=False)
        raise ValueError('No saved prediction detail files were found for ROC reconstruction.')

    pred = predictions[predictions['evaluation_level'].astype(str) == evaluation_level].copy()
    if pred.empty:
        raise ValueError(f'No prediction rows available for evaluation level `{evaluation_level}`.')

    pooled_rows: list[dict[str, Any]] = []
    for (model, feature_set), df in pred.groupby(['model', 'feature_set'], dropna=False):
        y_true = df['y_true'].to_numpy(dtype=float)
        y_proba = df['y_proba'].to_numpy(dtype=float)
        pooled_auc = roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else float('nan')
        pooled_rows.append({'model': model, 'feature_set': feature_set, 'pooled_auc': pooled_auc, 'n_samples': int(len(df))})
    pooled_df = pd.DataFrame(pooled_rows).sort_values(['pooled_auc', 'model'], ascending=[False, True])
    chosen = pooled_df.drop_duplicates(subset=['model'], keep='first').reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    fpr_grid = np.linspace(0.0, 1.0, 201)
    curve_rows: list[dict[str, Any]] = []
    summary_rows: list[dict[str, Any]] = []
    cmap = plt.cm.get_cmap('tab10', max(len(chosen), 3))

    for idx, row in chosen.iterrows():
        model = row['model']
        feature_set = row['feature_set']
        model_df = pred[(pred['model'] == model) & (pred['feature_set'] == feature_set)].copy()
        fold_curves: list[np.ndarray] = []
        fold_aucs: list[float] = []

        for fold_id, fold_df in model_df.groupby('fold_id', dropna=False):
            y_true = fold_df['y_true'].to_numpy(dtype=float)
            y_proba = fold_df['y_proba'].to_numpy(dtype=float)
            if len(np.unique(y_true)) < 2:
                continue
            fpr, tpr, _ = roc_curve(y_true, y_proba)
            interp_tpr = np.interp(fpr_grid, fpr, tpr)
            interp_tpr[0] = 0.0
            interp_tpr[-1] = 1.0
            fold_curves.append(interp_tpr)
            fold_auc = roc_auc_score(y_true, y_proba)
            fold_aucs.append(fold_auc)
            for fpr_value, tpr_value in zip(fpr_grid, interp_tpr):
                curve_rows.append({
                    'model': model,
                    'feature_set': feature_set,
                    'evaluation_level': evaluation_level,
                    'fold_id': fold_id,
                    'curve_type': 'fold',
                    'fpr': float(fpr_value),
                    'tpr': float(tpr_value),
                })

        if not fold_curves:
            continue

        fold_matrix = np.vstack(fold_curves)
        mean_curve = fold_matrix.mean(axis=0)
        mean_curve[0] = 0.0
        mean_curve[-1] = 1.0
        if len(fold_curves) == 1:
            ci_low_curve = mean_curve.copy()
            ci_high_curve = mean_curve.copy()
            auc_boot = np.asarray(fold_aucs, dtype=float)
        else:
            boot_curves = np.empty((n_bootstrap, len(fpr_grid)), dtype=float)
            auc_boot = np.empty(n_bootstrap, dtype=float)
            for boot_idx in range(n_bootstrap):
                sample_idx = RNG.integers(0, len(fold_curves), len(fold_curves))
                sample_curves = fold_matrix[sample_idx]
                boot_curves[boot_idx] = sample_curves.mean(axis=0)
                sampled_aucs = np.asarray(fold_aucs, dtype=float)[sample_idx]
                auc_boot[boot_idx] = float(np.nanmean(sampled_aucs))
            ci_low_curve = np.quantile(boot_curves, 0.025, axis=0)
            ci_high_curve = np.quantile(boot_curves, 0.975, axis=0)

        pooled_auc = float(row['pooled_auc'])
        auc_ci_low = float(np.quantile(auc_boot, 0.025)) if auc_boot.size else float('nan')
        auc_ci_high = float(np.quantile(auc_boot, 0.975)) if auc_boot.size else float('nan')
        color = cmap(idx)

        ax.plot(
            fpr_grid,
            mean_curve,
            linewidth=2.4,
            color=color,
            label=f'{model} ({feature_set}) AUC={pooled_auc:.3f} [{auc_ci_low:.3f}, {auc_ci_high:.3f}]',
        )
        ax.fill_between(fpr_grid, ci_low_curve, ci_high_curve, color=color, alpha=0.18)

        for curve_type, curve_values in (('mean', mean_curve), ('ci_low', ci_low_curve), ('ci_high', ci_high_curve)):
            for fpr_value, tpr_value in zip(fpr_grid, curve_values):
                curve_rows.append({
                    'model': model,
                    'feature_set': feature_set,
                    'evaluation_level': evaluation_level,
                    'fold_id': np.nan,
                    'curve_type': curve_type,
                    'fpr': float(fpr_value),
                    'tpr': float(tpr_value),
                })

        summary_rows.append({
            'model': model,
            'feature_set': feature_set,
            'evaluation_level': evaluation_level,
            'n_valid_folds': int(len(fold_curves)),
            'pooled_auc': pooled_auc,
            'fold_auc_mean': float(np.nanmean(fold_aucs)) if fold_aucs else float('nan'),
            'fold_auc_sd': float(np.nanstd(fold_aucs, ddof=1)) if len(fold_aucs) > 1 else (0.0 if fold_aucs else float('nan')),
            'auc_ci_low': auc_ci_low,
            'auc_ci_high': auc_ci_high,
        })

    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1.2, color='#666', label='Chance')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('Figure 6: ROC Comparison With 95% CI Across Held-Out Folds', fontsize=15, fontweight='bold')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8.5, loc='lower right', framealpha=0.95)
    fig.tight_layout()
    fig_path = figures_dir / 'figure6_roc_with_ci.png'
    fig.savefig(fig_path, dpi=500, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)

    curve_df = pd.DataFrame(curve_rows)
    summary_df = pd.DataFrame(summary_rows).sort_values('pooled_auc', ascending=False).reset_index(drop=True)
    curve_df.to_csv(out_dir / 'figure6_roc_curve_data.csv', index=False)
    summary_df.to_csv(out_dir / 'figure6_roc_bootstrap_summary.csv', index=False)

    return {'curve_data': curve_df, 'bootstrap_summary': summary_df, 'figure_path': fig_path}

def _format_metric_summary_lines(metric_summary: pd.DataFrame, evaluation_level: str = 'cell') -> list[str]:
    if metric_summary.empty:
        return ['No fold-level prediction detail files were available to summarize.']

    subset = metric_summary[metric_summary['evaluation_level'].astype(str) == evaluation_level].copy()
    if subset.empty:
        subset = metric_summary.copy()
    subset = subset.sort_values(['auc_mean', 'model'], ascending=[False, True])

    lines = []
    for _, row in subset.iterrows():
        auc_mean = row.get('auc_mean', np.nan)
        auc_sd = row.get('auc_sd', np.nan)
        acc_mean = row.get('accuracy_mean', np.nan)
        acc_sd = row.get('accuracy_sd', np.nan)
        lines.append(
            f"{row['model']} [{row['feature_set']}]: AUC {auc_mean:.3f} +/- {auc_sd:.3f}; Accuracy {acc_mean:.3f} +/- {acc_sd:.3f}; n_folds={int(row.get('n_folds', 0))}"
        )
    return lines

def run_end_analysis(adata, evaluation_level: str = 'cell') -> dict[str, Any]:
    out_dir = get_processed_data_dir()
    predictions = load_prediction_details()
    _, metric_summary = summarize_fold_metrics(predictions)
    result_tables = load_result_tables()

    roc_result = None
    roc_error = None
    try:
        roc_result = create_roc_figure_with_ci(predictions=predictions, evaluation_level=evaluation_level)
    except Exception as exc:
        roc_error = str(exc)
        print(f'ROC figure could not be created: {exc}')

    umap_result = None
    umap_error = None
    try:
        umap_result = create_annotated_umap_figure(adata)
    except Exception as exc:
        umap_error = str(exc)
        print(f'UMAP annotation could not be created: {exc}')

    pc6_result = None
    pc6_error = None
    try:
        pc6_result = compute_pc6_mito_correlation(adata)
    except Exception as exc:
        pc6_error = str(exc)
        print(f'PC6 mitochondrial analysis could not be created: {exc}')

    summary_lines = ['=' * 88, 'END ANALYSIS SUMMARY', '=' * 88]

    if not metric_summary.empty:
        best_row = metric_summary[metric_summary['evaluation_level'].astype(str) == evaluation_level].sort_values(['auc_mean', 'accuracy_mean'], ascending=[False, False])
        if best_row.empty:
            best_row = metric_summary.sort_values(['auc_mean', 'accuracy_mean'], ascending=[False, False])
        if not best_row.empty:
            top = best_row.iloc[0]
            summary_lines.append(f"Best {evaluation_level}-level model: {top['model']} [{top['feature_set']}] with AUC {top['auc_mean']:.3f} +/- {top['auc_sd']:.3f}")
    elif not result_tables.empty:
        result_tables = result_tables.copy()
        if 'model' in result_tables.columns and 'auc' in result_tables.columns:
            best_result = result_tables.sort_values('auc', ascending=False).iloc[0]
            summary_lines.append(f"Best available pooled result: {best_result['model']} ({best_result.get('feature_set', 'unknown')}) AUC={best_result['auc']:.3f}")

    summary_lines.append('')
    summary_lines.append('Fold-wise mean +/- SD metrics:')
    summary_lines.extend(_format_metric_summary_lines(metric_summary, evaluation_level=evaluation_level))

    summary_lines.append('')
    if roc_result is not None:
        summary_lines.append(f"ROC CI figure saved to {roc_result['figure_path']} with {len(roc_result['bootstrap_summary'])} model curves.")
    else:
        summary_lines.append(f'ROC CI figure unavailable: {roc_error}')

    if umap_result is not None:
        summary_lines.append('UMAP cluster counts:')
        for _, row in umap_result['cluster_counts'].iterrows():
            summary_lines.append(f"  Cluster {row['cluster_id']}: n={int(row['cell_count'])}")
    else:
        summary_lines.append(f'UMAP cluster annotation unavailable: {umap_error}')

    if pc6_result is not None:
        direction = 'positive' if pc6_result['rho'] >= 0 else 'negative'
        summary_lines.append(f"PC6 vs pct_counts_mt: Spearman rho={pc6_result['rho']:.3f}, permutation p={pc6_result['p_value']:.4g}, N={pc6_result['n_cells']:,} ({direction} association).")
    else:
        summary_lines.append(f'PC6 mitochondrial analysis unavailable: {pc6_error}')

    artifacts = {
        'processed_data_dir': out_dir,
        'metric_summary_csv': out_dir / 'loocv_metric_summary_mean_sd.csv',
        'fold_metrics_csv': out_dir / 'loocv_fold_metrics.csv',
        'roc_curve_data_csv': out_dir / 'figure6_roc_curve_data.csv',
        'roc_bootstrap_summary_csv': out_dir / 'figure6_roc_bootstrap_summary.csv',
        'roc_figure': None if roc_result is None else roc_result['figure_path'],
        'umap_cluster_counts_csv': None if umap_result is None else umap_result['cluster_counts_path'],
        'umap_figure': None if umap_result is None else umap_result['figure_path'],
        'pc6_mito_csv': None if pc6_result is None else pc6_result['csv_path'],
        'pc6_mito_figure': None if pc6_result is None else pc6_result['figure_path'],
    }
    summary_lines.append('')
    summary_lines.append('Saved artifacts:')
    for key, value in artifacts.items():
        if value is not None:
            summary_lines.append(f'  {key}: {value}')

    summary_text = ''.join(summary_lines)
    print(summary_text)

    text_path = out_dir / 'analysis_summary.txt'
    json_path = out_dir / 'analysis_summary.json'
    text_path.write_text(summary_text, encoding='utf-8')

    summary_payload = {
        'evaluation_level': evaluation_level,
        'metric_summary': metric_summary.to_dict(orient='records'),
        'roc_bootstrap_summary': [] if roc_result is None else roc_result['bootstrap_summary'].to_dict(orient='records'),
        'umap_cluster_counts': [] if umap_result is None else umap_result['cluster_counts'].to_dict(orient='records'),
        'pc6_mito_correlation': None if pc6_result is None else {
            'rho': pc6_result['rho'],
            'permutation_p_value': pc6_result['p_value'],
            'n_cells': pc6_result['n_cells'],
        },
        'artifacts': _to_builtin(artifacts),
        'errors': {'roc': roc_error, 'umap': umap_error, 'pc6_mito': pc6_error},
    }
    json_path.write_text(json.dumps(_to_builtin(summary_payload), indent=2), encoding='utf-8')

    return {
        'metric_summary': metric_summary,
        'roc_result': roc_result,
        'umap_result': umap_result,
        'pc6_result': pc6_result,
        'summary_text': summary_text,
        'summary_text_path': text_path,
        'summary_json_path': json_path,
    }

if 'adata' not in globals() or adata is None:
    print("ERROR: adata not available. Run upstream data loading cells first.")
else:
    try:
        umap_result = create_annotated_umap_figure(adata)
        print(f"Annotated UMAP figure saved to: {umap_result['figure_path']}")
        print(f"Cluster count table saved to: {umap_result['cluster_counts_path']}")
        display(umap_result['cluster_counts'])
    except Exception as e:
        print(f"Could not generate Figure 4 UMAP annotation: {e}")


In [ ]:
# Figure 3: Quality Control Metrics by Response Group
import matplotlib.pyplot as plt
import numpy as np

if 'adata' not in globals() or adata is None:
    print("ERROR: adata not available. Run upstream cells first.")
else:
    qc_cols = [c for c in ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'] if c in adata.obs.columns]
    has_resp = 'response' in adata.obs.columns
    titles = {'n_genes_by_counts': 'Genes Detected per Cell',
              'total_counts': 'Total UMI Counts', 'pct_counts_mt': 'Mitochondrial %'}
    resp_colors = {'Responder': '#2196F3', 'Non-Responder': '#F44336', 'Unknown': '#BDBDBD'}

    if len(qc_cols) >= 2:
        fig, axes = plt.subplots(1, len(qc_cols), figsize=(6*len(qc_cols), 5))
        if len(qc_cols) == 1: axes = [axes]

        for i, col in enumerate(qc_cols):
            ax = axes[i]
            if has_resp:
                groups = sorted([r for r in adata.obs['response'].unique()
                               if r in ['Responder', 'Non-Responder']])
                data_list, color_list = [], []
                for g in groups:
                    vals = adata.obs.loc[adata.obs['response'] == g, col].dropna().values
                    if len(vals) > 0:
                        data_list.append(vals)
                        color_list.append(resp_colors.get(g, '#999'))
                if data_list:
                    parts = ax.violinplot(data_list, showmeans=True, showmedians=True)
                    for j, pc in enumerate(parts['bodies']):
                        pc.set_facecolor(color_list[j]); pc.set_alpha(0.7)
                    for key in ['cmeans','cmedians','cbars','cmins','cmaxes']:
                        if key in parts: parts[key].set_color('#333')
                    ax.set_xticks(range(1, len(groups)+1))
                    ax.set_xticklabels(groups, fontsize=11)
                    # Add sample sizes
                    for j, g in enumerate(groups):
                        n = len(adata.obs[adata.obs['response'] == g])
                        ax.text(j+1, ax.get_ylim()[0], f'n={n:,}', ha='center', fontsize=9, color='#666')
            else:
                vals = adata.obs[col].dropna().values
                parts = ax.violinplot([vals], showmeans=True, showmedians=True)
                for pc in parts['bodies']: pc.set_facecolor('#2196F3'); pc.set_alpha(0.7)

            ax.set_title(f'({chr(65+i)}) {titles.get(col, col)}', fontsize=13, fontweight='bold')
            ax.set_ylabel(col.replace('_', ' ').title(), fontsize=11)
            ax.grid(axis='y', alpha=0.3)

        plt.suptitle('Figure 3: Quality Control Metrics', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('Figure3_QC_Metrics.png', dpi=600, bbox_inches='tight', facecolor='white')
        plt.show()
        print("Figure 3 saved.")
    else:
        print(f"Insufficient QC columns: {qc_cols}")

Figure 3 saved.


In [ ]:
# Figure 4: PCA Variance Explained
import matplotlib.pyplot as plt
import numpy as np

if 'adata' not in globals() or adata is None:
    print("ERROR: adata not available.")
else:
    var_ratio = None
    if 'pca' in adata.uns and 'variance_ratio' in adata.uns['pca']:
        var_ratio = adata.uns['pca']['variance_ratio']

    if var_ratio is not None and len(var_ratio) > 0:
        n_show = min(30, len(var_ratio))
        vr = var_ratio[:n_show]
        cum = np.cumsum(vr)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # Panel A: Individual PC variance
        ax1.bar(range(1, n_show+1), vr*100, color='#1565C0', alpha=0.85, edgecolor='#0D47A1')
        ax1.set_xlabel('Principal Component', fontsize=12)
        ax1.set_ylabel('Variance Explained (%)', fontsize=12)
        ax1.set_title('(A) Individual PC Variance', fontsize=14, fontweight='bold')
        ax1.grid(axis='y', alpha=0.3)

        # Panel B: Cumulative variance
        ax2.plot(range(1, n_show+1), cum*100, color='#C62828', lw=2.5, marker='o',
                 ms=5, mfc='white', mec='#C62828')
        ax2.fill_between(range(1, n_show+1), cum*100, alpha=0.12, color='#C62828')
        ax2.axhline(y=90, color='#666', ls='--', alpha=0.7, label='90% threshold')
        pc_90 = np.searchsorted(cum, 0.9) + 1
        if pc_90 <= n_show:
            ax2.axvline(x=pc_90, color='#2E7D32', ls='--', alpha=0.7)
            ax2.text(pc_90+0.5, 85, f'PC{pc_90}: {cum[pc_90-1]*100:.1f}%',
                     fontsize=10, color='#2E7D32', fontweight='bold')
        ax2.set_xlabel('Number of PCs', fontsize=12)
        ax2.set_ylabel('Cumulative Variance (%)', fontsize=12)
        ax2.set_title('(B) Cumulative Variance', fontsize=14, fontweight='bold')
        ax2.legend(fontsize=10)

        plt.suptitle('Figure 4: PCA Dimensionality Reduction', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('Figure4_PCA_Variance.png', dpi=600, bbox_inches='tight', facecolor='white')
        plt.show()
        print("Figure 4 saved.")
    else:
        print("PCA variance data not available in adata.uns.")

Figure 4 saved.


In [ ]:
# Figure 5: Patient Cohort & Sample Composition
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

if 'adata' not in globals() or adata is None:
    print("ERROR: adata not available.")
else:
    has_patient = 'patient_id' in adata.obs.columns
    has_resp = 'response' in adata.obs.columns
    has_tp = 'timepoint' in adata.obs.columns
    resp_colors = {'Responder': '#2196F3', 'Non-Responder': '#F44336', 'Unknown': '#BDBDBD'}

    n_panels = int(has_patient and has_resp) + int(has_patient and has_tp) + 1
    fig, axes = plt.subplots(1, min(3, n_panels), figsize=(6*min(3, n_panels), 5))
    if min(3, n_panels) == 1: axes = [axes]
    pi = 0

    # Panel A: Cells per patient by response
    if has_patient and has_resp:
        ct = adata.obs.groupby(['patient_id', 'response']).size().unstack(fill_value=0)
        resp_order = [r for r in ['Responder', 'Non-Responder', 'Unknown'] if r in ct.columns]
        ct = ct[resp_order]
        ct.plot(kind='bar', stacked=True, ax=axes[pi],
                color=[resp_colors.get(r, '#999') for r in resp_order],
                edgecolor='#333', linewidth=0.5)
        axes[pi].set_title('(A) Cells per Patient', fontsize=13, fontweight='bold')
        axes[pi].set_xlabel('Patient ID', fontsize=11)
        axes[pi].set_ylabel('Number of Cells', fontsize=11)
        axes[pi].legend(title='Response', fontsize=9)
        axes[pi].tick_params(axis='x', rotation=0)
        # Add total counts on bars
        for j, pid in enumerate(ct.index):
            total = ct.loc[pid].sum()
            axes[pi].text(j, total + total*0.02, f'{int(total):,}', ha='center', fontsize=8, fontweight='bold')
        pi += 1

    # Panel B: Cells per timepoint
    if has_patient and has_tp and pi < len(axes):
        tp_colors = {'Baseline': '#4CAF50', 'Post-Tx': '#FF9800', 'Recurrence': '#9C27B0',
                     'Pre': '#4CAF50', 'D21': '#FF9800', 'D42': '#9C27B0'}
        ct2 = adata.obs.groupby(['patient_id', 'timepoint']).size().unstack(fill_value=0)
        tp_order = [t for t in tp_colors.keys() if t in ct2.columns]
        if tp_order:
            ct2 = ct2[tp_order]
        ct2.plot(kind='bar', stacked=True, ax=axes[pi],
                color=[tp_colors.get(t, '#999') for t in ct2.columns],
                edgecolor='#333', linewidth=0.5)
        axes[pi].set_title('(B) Cells per Timepoint', fontsize=13, fontweight='bold')
        axes[pi].set_xlabel('Patient ID', fontsize=11)
        axes[pi].set_ylabel('Number of Cells', fontsize=11)
        axes[pi].legend(title='Timepoint', fontsize=9)
        axes[pi].tick_params(axis='x', rotation=0)
        pi += 1

    # Panel C: Overall response distribution
    if has_resp and pi < len(axes):
        resp_counts = adata.obs['response'].value_counts()
        r_order = [r for r in ['Responder', 'Non-Responder', 'Unknown'] if r in resp_counts.index]
        vals = [resp_counts[r] for r in r_order]
        cols = [resp_colors.get(r, '#999') for r in r_order]
        wedges, texts, autotexts = axes[pi].pie(vals, labels=r_order, colors=cols,
            autopct=lambda p: f'{p:.1f}%\n({int(p*sum(vals)/100):,})',
            startangle=90, textprops={'fontsize': 10})
        for at in autotexts: at.set_fontweight('bold')
        axes[pi].set_title('(C) Response Distribution', fontsize=13, fontweight='bold')

    plt.suptitle('Figure 5: Patient Cohort Composition', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('Figure5_Sample_Composition.png', dpi=600, bbox_inches='tight', facecolor='white')
    plt.show()
    print("Figure 5 saved.")

Figure 5 saved.


In [ ]:
# Figure 6: ROC Comparison With 95% CI Across Held-Out Folds
try:
    prediction_details_df = load_prediction_details(evaluation_level='cell')
    roc_result = create_roc_figure_with_ci(predictions=prediction_details_df, evaluation_level='cell')
    print(f"ROC figure saved to: {roc_result['figure_path']}")
    print("ROC curve data saved to: Processed_Data/figure6_roc_curve_data.csv")
    print("ROC bootstrap summary saved to: Processed_Data/figure6_roc_bootstrap_summary.csv")
    display(roc_result['bootstrap_summary'])
except Exception as e:
    print(f"Could not generate Figure 6 ROC comparison: {e}")


In [ ]:
# Figure 7: Deep Learning Architecture Comparison (Detailed)
# Dedicated DL figure: metrics across all architectures and feature sets.
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

results = load_all_model_results() if 'load_all_model_results' in dir() else all_results_df
if results is None:
    print("ERROR: No results available.")
else:
    dl_df = results[results['model'].isin(DL_MODELS)].copy() if 'model' in results.columns else pd.DataFrame()

    if len(dl_df) < 2:
        # Also try dl_results_rows directly
        if 'dl_results_rows' in globals() and isinstance(dl_results_rows, list) and len(dl_results_rows) > 0:
            dl_df = pd.DataFrame(dl_results_rows)
            if 'architecture' in dl_df.columns and 'model' not in dl_df.columns:
                dl_df['model'] = dl_df['architecture']

    if len(dl_df) < 2:
        print("ERROR: Insufficient DL results. Run deep learning training cells first.")
    else:
        metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
        avail_metrics = [m for m in metrics if m in dl_df.columns]

        # Filter to cell-level results
        if 'evaluation_level' in dl_df.columns:
            cell_dl = dl_df[dl_df['evaluation_level'].astype(str).str.contains('cell', case=False, na=True)]
            if len(cell_dl) > 0: dl_df = cell_dl

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Panel A: Grouped bar chart — metrics per DL architecture
        arch_summary = dl_df.groupby('model')[avail_metrics].mean().reset_index()
        arch_summary = arch_summary.sort_values('auc', ascending=False)
        archs = arch_summary['model'].values
        x = np.arange(len(avail_metrics))
        width = 0.8 / len(archs)

        for i, arch in enumerate(archs):
            row = arch_summary[arch_summary['model'] == arch].iloc[0]
            vals = [row[m] for m in avail_metrics]
            offset = (i - len(archs)/2 + 0.5) * width
            bars = axes[0].bar(x + offset, vals, width, label=arch,
                              color=DL_COLORS.get(arch, '#6A1B9A'), edgecolor='#333', linewidth=0.5)
            for bar, val in zip(bars, vals):
                if val > 0:
                    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                               f'{val:.2f}', ha='center', fontsize=7, fontweight='bold')

        axes[0].set_xticks(x)
        axes[0].set_xticklabels([m.upper() for m in avail_metrics], fontsize=11)
        axes[0].set_ylabel('Score', fontsize=12)
        axes[0].set_title('(A) DL Architecture Metrics Comparison', fontsize=14, fontweight='bold')
        axes[0].legend(fontsize=9, loc='lower left')
        axes[0].set_ylim(0, 1.15)
        axes[0].grid(axis='y', alpha=0.3)

        # Panel B: Performance by feature set (if available)
        if 'feature_set' in dl_df.columns and dl_df['feature_set'].nunique() > 1:
            pivot = dl_df.groupby(['model', 'feature_set'])['auc'].mean().unstack(fill_value=0)
            import seaborn as sns
            sns.heatmap(pivot, cmap='RdYlGn', annot=True, fmt='.3f', linewidths=0.5,
                       linecolor='gray', ax=axes[1], vmin=0.3, vmax=1.0,
                       cbar_kws={'label': 'AUC-ROC', 'shrink': 0.8})
            axes[1].set_title('(B) DL AUC by Feature Configuration', fontsize=14, fontweight='bold')
            axes[1].set_xlabel('Feature Set', fontsize=11)
            axes[1].set_ylabel('Architecture', fontsize=11)
            axes[1].tick_params(axis='x', rotation=30)
        else:
            # Fallback: show specificity + NPV if available
            extra_metrics = [m for m in ['specificity', 'npv'] if m in dl_df.columns]
            if extra_metrics:
                for arch in archs:
                    row = arch_summary[arch_summary['model'] == arch].iloc[0]
                    vals = [row.get(m, 0) for m in extra_metrics]
                    axes[1].barh(arch, vals[0] if vals else 0, color=DL_COLORS.get(arch, '#6A1B9A'),
                                edgecolor='#333', linewidth=0.5)
                axes[1].set_xlabel('Specificity', fontsize=11)
                axes[1].set_title('(B) Specificity per Architecture', fontsize=14, fontweight='bold')
            else:
                axes[1].text(0.5, 0.5, 'Single feature set\nused for DL',
                            ha='center', va='center', fontsize=12, transform=axes[1].transAxes)
                axes[1].set_title('(B) Feature Set Analysis', fontsize=14, fontweight='bold')

        plt.suptitle('Figure 7: Deep Learning Architecture Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('Figure7_DL_Architecture_Comparison.png', dpi=600, bbox_inches='tight', facecolor='white')
        plt.show()
        print("Figure 7 saved.")

  Loaded 34 rows from Processed_Data/ml_integrated_results.csv
  Loaded 8 rows from Processed_Data/lopo_results.csv
  Loaded 4 rows from Processed_Data/homology_aware_benchmark_results.csv
Figure 7 saved.


In [ ]:
# Figure 8: Performance Radar Chart — DL Models + Best Traditional ML
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

results = load_all_model_results() if 'load_all_model_results' in dir() else all_results_df
best = get_best_results(results) if results is not None else None

if best is None or len(best) < 3:
    print("ERROR: Insufficient results for radar chart.")
else:
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    avail = [m for m in metrics if m in best.columns]

    if len(avail) < 3:
        print(f"ERROR: Need at least 3 metrics, found: {avail}")
    else:
        # Get DL models and top 2-3 traditional ML models
        dl_models = best[best['model'].isin(DL_MODELS)].sort_values('auc', ascending=False)
        ml_models = best[~best['model'].isin(DL_MODELS)].sort_values('auc', ascending=False).head(3)
        show_models = pd.concat([dl_models, ml_models]).drop_duplicates('model')

        n_metrics = len(avail)
        angles = np.linspace(0, 2*np.pi, n_metrics, endpoint=False).tolist()
        angles += angles[:1]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), subplot_kw=dict(polar=True))

        # Panel A: DL Models
        for i, (_, row) in enumerate(dl_models.iterrows()):
            vals = [row[m] for m in avail] + [row[avail[0]]]
            color = DL_COLORS.get(row['model'], '#6A1B9A')
            ax1.plot(angles, vals, 'o-', lw=2.5, label=row['model'], color=color, ms=6)
            ax1.fill(angles, vals, alpha=0.1, color=color)

        ax1.set_xticks(angles[:-1])
        ax1.set_xticklabels([m.upper() for m in avail], fontsize=11, fontweight='bold')
        ax1.set_ylim(0, 1); ax1.set_yticks([0.2,0.4,0.6,0.8,1.0])
        ax1.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8, color='gray')
        ax1.set_title('(A) Deep Learning Models', fontsize=14, fontweight='bold', pad=20)
        ax1.legend(loc='lower right', bbox_to_anchor=(1.3, -0.1), fontsize=10)
        ax1.grid(True, alpha=0.3)

        # Panel B: Top Traditional/Ensemble ML
        ml_colors = ['#D84315', '#E65100', '#FF8F00', '#FFA000', '#FFB300']
        for i, (_, row) in enumerate(ml_models.iterrows()):
            vals = [row[m] for m in avail] + [row[avail[0]]]
            ax2.plot(angles, vals, 'o-', lw=2.5, label=row['model'],
                    color=ml_colors[i % len(ml_colors)], ms=6)
            ax2.fill(angles, vals, alpha=0.1, color=ml_colors[i % len(ml_colors)])

        ax2.set_xticks(angles[:-1])
        ax2.set_xticklabels([m.upper() for m in avail], fontsize=11, fontweight='bold')
        ax2.set_ylim(0, 1); ax2.set_yticks([0.2,0.4,0.6,0.8,1.0])
        ax2.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8, color='gray')
        ax2.set_title('(B) Top Traditional ML Models', fontsize=14, fontweight='bold', pad=20)
        ax2.legend(loc='lower right', bbox_to_anchor=(1.3, -0.1), fontsize=10)
        ax2.grid(True, alpha=0.3)

        plt.suptitle('Figure 8: Multi-Metric Performance Radar', fontsize=16, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig('Figure8_Radar_Chart.png', dpi=600, bbox_inches='tight', facecolor='white')
        plt.show()
        print("Figure 8 saved.")

  Loaded 34 rows from Processed_Data/ml_integrated_results.csv
  Loaded 8 rows from Processed_Data/lopo_results.csv
  Loaded 4 rows from Processed_Data/homology_aware_benchmark_results.csv
Figure 8 saved.


In [ ]:
# Figure 9: SHAP Feature Importance for Best Deep Learning Model
# Computes SHAP values using DeepExplainer/GradientExplainer on the best DL model,
# then maps PCA SHAP values back to original genes through PCA loadings.
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
import warnings
warnings.filterwarnings('ignore')

shap_gene_importance = None  # will hold DataFrame with gene, importance columns

if 'adata' not in globals() or adata is None:
    print("ERROR: adata not available.")
else:
    try:
        import shap
        HAS_SHAP = True
    except ImportError:
        print("Installing shap...")
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])
        import shap
        HAS_SHAP = True

    # Identify best DL model
    results = load_all_model_results() if 'load_all_model_results' in dir() else None
    best_dl_arch = None
    if results is not None and 'model' in results.columns:
        dl_res = results[results['model'].isin(DL_MODELS)].copy()
        if 'evaluation_level' in dl_res.columns:
            cell_dl = dl_res[dl_res['evaluation_level'].astype(str).str.contains('cell', case=False, na=True)]
            if len(cell_dl) > 0: dl_res = cell_dl
        if len(dl_res) > 0:
            best_dl_arch = dl_res.sort_values('auc', ascending=False).iloc[0]['model']
            print(f"Best DL architecture for SHAP: {best_dl_arch}")

    # Strategy 1: Load saved DL model and compute SHAP with DeepExplainer
    shap_values_computed = False
    if best_dl_arch and HAS_SHAP:
        from pathlib import Path
        import glob
        model_pattern = f'Output/Models/DL/dl_model_*_{best_dl_arch}_fold*.keras'
        model_files = sorted(glob.glob(model_pattern))
        if not model_files:
            # Try alternate patterns
            model_pattern = f'Output/Models/DL/dl_model_*_{best_dl_arch}*.keras'
            model_files = sorted(glob.glob(model_pattern))

        if model_files and 'supervised_mask' in globals():
            try:
                import tensorflow as tf
                model = tf.keras.models.load_model(model_files[0], compile=False)
                print(f"Loaded DL model: {model_files[0]}")

                # Get the gene PCA input data
                X_gene = adata.obsm['X_gene_pca'][supervised_mask]
                from sklearn.preprocessing import StandardScaler
                scaler = StandardScaler().fit(X_gene)
                X_scaled = scaler.transform(X_gene)

                # If model takes gene input (MLP), compute SHAP directly
                # For multi-input models, extract gene input branch
                input_shapes = [inp.shape for inp in model.inputs]
                if len(input_shapes) == 1 and input_shapes[0][-1] == X_scaled.shape[1]:
                    # MLP: direct SHAP on gene PCA
                    bg = X_scaled[np.random.choice(len(X_scaled), min(100, len(X_scaled)), replace=False)]
                    explainer = shap.GradientExplainer(model, bg)
                    test_sample = X_scaled[:min(200, len(X_scaled))]
                    sv = explainer.shap_values(test_sample)
                    if isinstance(sv, list): sv = sv[0]
                    mean_abs_shap = np.abs(sv).mean(axis=0).flatten()

                    # Map PCA SHAP values to gene space through PCA loadings
                    if 'pca' in adata.uns and 'PCs' in adata.uns['pca']:
                        pca_loadings = adata.uns['pca']['PCs']  # shape: (n_pcs, n_genes)
                        n_pcs_used = min(mean_abs_shap.shape[0], pca_loadings.shape[0])
                        gene_importance = np.abs(pca_loadings[:n_pcs_used].T) @ mean_abs_shap[:n_pcs_used]
                        top_idx = np.argsort(gene_importance)[::-1][:20]
                        shap_gene_importance = pd.DataFrame({
                            'gene': np.array(adata.var_names)[top_idx],
                            'importance': gene_importance[top_idx]
                        })
                        shap_values_computed = True
                        print(f"SHAP computed via GradientExplainer + PCA loading projection")
                    else:
                        # Use PCA component importance directly
                        top_pcs = np.argsort(mean_abs_shap)[::-1][:20]
                        shap_gene_importance = pd.DataFrame({
                            'gene': [f'PC{i+1}' for i in top_pcs],
                            'importance': mean_abs_shap[top_pcs]
                        })
                        shap_values_computed = True
                else:
                    # Multi-input model: use gene input branch
                    print(f"Multi-input model (shapes: {input_shapes}). Using gene branch for SHAP.")
                    # Build a sub-model from the gene input to the output
                    gene_input_idx = None
                    for idx, inp in enumerate(model.inputs):
                        if inp.shape[-1] == X_scaled.shape[1]:
                            gene_input_idx = idx
                            break
                    if gene_input_idx is not None:
                        bg = X_scaled[np.random.choice(len(X_scaled), min(100, len(X_scaled)), replace=False)]
                        # Create a wrapper that fixes the sequence input
                        tra_seq, trb_seq, seq_len = None, None, None
                        if 'X_tcr_tra_onehot' in adata.obsm and 'X_tcr_trb_onehot' in adata.obsm:
                            tra_flat = np.asarray(adata.obsm['X_tcr_tra_onehot'][supervised_mask])
                            trb_flat = np.asarray(adata.obsm['X_tcr_trb_onehot'][supervised_mask])
                            if hasattr(tra_flat, 'toarray'): tra_flat = tra_flat.toarray()
                            if hasattr(trb_flat, 'toarray'): trb_flat = trb_flat.toarray()
                            n_ch = 20
                            if tra_flat.shape[1] % n_ch == 0:
                                sl = tra_flat.shape[1] // n_ch
                                tra_seq = tra_flat.reshape(-1, sl, n_ch)
                                trb_seq = trb_flat.reshape(-1, sl, n_ch)
                                X_seq = np.concatenate([tra_seq, trb_seq], axis=2)
                                mean_seq = X_seq[:min(100, len(X_seq))].mean(axis=0, keepdims=True)
                                mean_seq = np.broadcast_to(mean_seq, (len(bg), *mean_seq.shape[1:]))

                                # Use KernelExplainer with fixed sequence input
                                def model_predict_gene_only(gene_data):
                                    seq_data = np.broadcast_to(X_seq[:1], (len(gene_data), *X_seq.shape[1:]))
                                    return model.predict([seq_data, gene_data], verbose=0).flatten()

                                try:
                                    explainer = shap.KernelExplainer(model_predict_gene_only, bg[:50])
                                    sv = explainer.shap_values(X_scaled[:min(100, len(X_scaled))])
                                    if isinstance(sv, list): sv = sv[0]
                                    mean_abs_shap = np.abs(sv).mean(axis=0).flatten()

                                    if 'pca' in adata.uns and 'PCs' in adata.uns['pca']:
                                        pca_loadings = adata.uns['pca']['PCs']
                                        n_pcs_used = min(mean_abs_shap.shape[0], pca_loadings.shape[0])
                                        gene_importance = np.abs(pca_loadings[:n_pcs_used].T) @ mean_abs_shap[:n_pcs_used]
                                        top_idx = np.argsort(gene_importance)[::-1][:20]
                                        shap_gene_importance = pd.DataFrame({
                                            'gene': np.array(adata.var_names)[top_idx],
                                            'importance': gene_importance[top_idx]
                                        })
                                        shap_values_computed = True
                                        print(f"SHAP computed via KernelExplainer for multi-input model")
                                except Exception as e:
                                    print(f"KernelExplainer failed: {e}")
            except Exception as e:
                print(f"DL model SHAP computation failed: {e}")

    # Strategy 2: Use ExtraTrees as surrogate + SHAP TreeExplainer
    if not shap_values_computed and HAS_SHAP and 'response' in adata.obs.columns:
        try:
            from sklearn.ensemble import ExtraTreesClassifier
            from sklearn.preprocessing import LabelEncoder
            import scipy.sparse as sp

            mask = adata.obs['response'].isin(['Responder', 'Non-Responder']).values
            if mask.sum() > 10:
                y = LabelEncoder().fit_transform(adata.obs['response'][mask])
                X = adata.X[mask]
                if sp.issparse(X):
                    var = np.asarray(X.multiply(X).mean(axis=0)).ravel() - np.asarray(X.mean(axis=0)).ravel()**2
                else:
                    var = np.var(X, axis=0)
                top_idx = np.argsort(var)[-500:]
                if sp.issparse(X):
                    X_top = X[:, top_idx].toarray()
                else:
                    X_top = X[:, top_idx]
                gene_names = np.asarray(adata.var_names)[top_idx]

                et = ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
                et.fit(X_top, y)

                explainer = shap.TreeExplainer(et)
                sv = explainer.shap_values(X_top[:min(300, len(X_top))])
                if isinstance(sv, list): sv = sv[1]  # class 1 (Responder)
                mean_abs_shap = np.abs(sv).mean(axis=0)

                order = np.argsort(mean_abs_shap)[::-1][:20]
                shap_gene_importance = pd.DataFrame({
                    'gene': gene_names[order],
                    'importance': mean_abs_shap[order]
                })
                shap_values_computed = True
                best_dl_arch = best_dl_arch or 'Surrogate ExtraTrees'
                print(f"SHAP computed via TreeExplainer on ExtraTrees surrogate")
        except Exception as e:
            print(f"Surrogate SHAP also failed: {e}")

    # Strategy 3: Use upstream gene_df if available
    if not shap_values_computed and 'gene_df' in globals() and isinstance(gene_df, pd.DataFrame) and len(gene_df) > 0:
        shap_gene_importance = gene_df.head(20).copy()
        shap_gene_importance.columns = [c.lower() for c in shap_gene_importance.columns]
        if 'gene_id' in shap_gene_importance.columns:
            shap_gene_importance = shap_gene_importance.rename(columns={'gene_id': 'gene'})
        shap_values_computed = True
        best_dl_arch = best_dl_arch or 'ExtraTrees'
        print("Using upstream gene importance rankings")

    # Plot
    if shap_gene_importance is not None and len(shap_gene_importance) > 0:
        shap_sorted = shap_gene_importance.sort_values('importance', ascending=True).tail(20)

        fig, ax = plt.subplots(figsize=(10, 8))

        def gene_color(g):
            gu = str(g).upper()
            if gu in ['GZMB','GZMA','GZMK','PRF1','NKG7','GNLY']: return '#E53935'
            if any(gu.startswith(p) for p in ['IFIT','ISG','MX','IFN']): return '#1E88E5'
            if gu in ['PDCD1','TIGIT','LAG3','HAVCR2','CTLA4','CD274']: return '#43A047'
            if any(gu.startswith(p) for p in ['CD','CCL','CXC']): return '#FB8C00'
            return '#7E57C2'

        colors = [gene_color(g) for g in shap_sorted['gene']]
        bars = ax.barh(range(len(shap_sorted)), shap_sorted['importance'].values,
                      color=colors, edgecolor='#333', linewidth=0.5, height=0.7)
        ax.set_yticks(range(len(shap_sorted)))
        ax.set_yticklabels(shap_sorted['gene'].values, fontsize=11)
        ax.set_xlabel('Mean |SHAP Value|', fontsize=13)
        ax.set_title(f'Figure 9: SHAP Feature Importance — {best_dl_arch or "Best DL Model"}',
                     fontsize=15, fontweight='bold')

        for i, (val, bar) in enumerate(zip(shap_sorted['importance'].values, bars)):
            ax.text(val + max(shap_sorted['importance'].values)*0.01, i, f'{val:.4f}',
                   va='center', fontsize=9)

        legend_els = [
            mpatches.Patch(color='#E53935', label='Cytotoxic (GZMB, PRF1)'),
            mpatches.Patch(color='#1E88E5', label='IFN-Stimulated (IFIT, ISG)'),
            mpatches.Patch(color='#43A047', label='Checkpoint (PD-1, CTLA4)'),
            mpatches.Patch(color='#FB8C00', label='Chemokines/Markers'),
            mpatches.Patch(color='#7E57C2', label='Other'),
        ]
        ax.legend(handles=legend_els, loc='lower right', fontsize=9, framealpha=0.9)
        ax.grid(axis='x', alpha=0.3)

        plt.tight_layout()
        plt.savefig('Figure9_SHAP_DL_Feature_Importance.png', dpi=600, bbox_inches='tight', facecolor='white')
        plt.show()
        print("Figure 9 saved.")
    else:
        print("Could not compute SHAP values. Try running upstream cells first.")

  Loaded 34 rows from Processed_Data/ml_integrated_results.csv
  Loaded 8 rows from Processed_Data/lopo_results.csv
  Loaded 4 rows from Processed_Data/homology_aware_benchmark_results.csv
Best DL architecture for SHAP: MLP
Loaded DL model: Output/Models/DL/dl_model_comprehensive_MLP_fold0.keras
Figure 9 saved.


In [ ]:
# Figure 10: Multi-Model Feature Importance Consensus
# Shows which genes are consistently ranked as important across multiple ML
# methods (ExtraTrees, Random Forest, Gradient Boosting) — not hardcoded.
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np, pandas as pd
import warnings
warnings.filterwarnings('ignore')

if 'adata' not in globals() or adata is None:
    print("ERROR: adata not available.")
else:
    from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
    from sklearn.preprocessing import LabelEncoder
    import scipy.sparse as sp

    mask = adata.obs['response'].isin(['Responder', 'Non-Responder']).values
    y = LabelEncoder().fit_transform(adata.obs['response'][mask])
    X = adata.X[mask]

    # Use top 500 most variable genes
    if sp.issparse(X):
        var = np.asarray(X.multiply(X).mean(axis=0)).ravel() - np.asarray(X.mean(axis=0)).ravel()**2
        X_dense = X.toarray()
    else:
        var = np.var(X, axis=0)
        X_dense = X
    top_idx = np.argsort(var)[-500:]
    X_top = X_dense[:, top_idx]
    gene_names = np.asarray(adata.var_names)[top_idx]

    # Fit 3 tree-based models
    models = {
        'ExtraTrees': ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced'),
        'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced'),
        'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5),
    }
    importances = {}
    for name, clf in models.items():
        clf.fit(X_top, y)
        importances[name] = clf.feature_importances_
        print(f"  Fit {name} — top gene: {gene_names[np.argmax(clf.feature_importances_)]}")

    # Aggregate: average rank across methods
    ranks = {}
    for name, imp in importances.items():
        ranks[name] = np.argsort(np.argsort(-imp))  # rank 0 = most important
    avg_rank = np.mean(list(ranks.values()), axis=0)
    consensus_idx = np.argsort(avg_rank)[:20]

    fig, axes = plt.subplots(1, 3, figsize=(18, 8), sharey=True)
    model_colors = {'ExtraTrees': '#7B1FA2', 'RandomForest': '#2E7D32', 'GradientBoosting': '#E65100'}

    for ax, (name, imp) in zip(axes, importances.items()):
        vals = imp[consensus_idx]
        sorted_idx = np.argsort(vals)
        ax.barh(range(20), vals[sorted_idx], color=model_colors[name],
               edgecolor='#333', linewidth=0.5, height=0.7, alpha=0.85)
        ax.set_yticks(range(20))
        ax.set_yticklabels(gene_names[consensus_idx][sorted_idx], fontsize=10)
        ax.set_xlabel('Feature Importance', fontsize=11)
        ax.set_title(name, fontsize=13, fontweight='bold', color=model_colors[name])
        ax.grid(axis='x', alpha=0.3)

    fig.suptitle('Figure 10: Multi-Model Feature Importance Consensus\n(Top 20 Genes by Average Rank)',
                fontsize=15, fontweight='bold', y=1.02)

    # Add SHAP comparison if available from Figure 9
    if shap_gene_importance is not None and len(shap_gene_importance) > 0:
        shap_genes = set(shap_gene_importance['gene'].values[:10])
        consensus_genes = set(gene_names[consensus_idx[:10]])
        overlap = shap_genes & consensus_genes
        fig.text(0.5, -0.02,
                f'Overlap with SHAP top-10: {len(overlap)}/10 genes ({", ".join(sorted(overlap)) if overlap else "none"})',
                ha='center', fontsize=11, style='italic')

    plt.tight_layout()
    plt.savefig('Figure10_MultiModel_Feature_Importance.png', dpi=600, bbox_inches='tight', facecolor='white')
    plt.show()
    print("Figure 10 saved.")

  Fit ExtraTrees — top gene: Human_week7
  Fit RandomForest — top gene: Human_week15
  Fit GradientBoosting — top gene: Human_week7
Figure 10 saved.


In [ ]:
# Figure 11: TCR CDR3 Sequence Analysis & Feature Set Impact
# Panel A: CDR3 length distributions by response (from actual adata)
# Panel B: Feature set performance comparison — does TCR data help? (from actual results)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np, pandas as pd
import warnings
warnings.filterwarnings('ignore')

if 'adata' not in globals() or adata is None:
    print("ERROR: adata not available.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Panel A: CDR3 Length Distribution by Response
    ax = axes[0]
    has_cdr3_data = False
    for col_pattern in ['cdr3_length', 'CDR3_length', 'TCR_cdr3_len', 'cdr3_aa_length']:
        cdr3_cols = [c for c in adata.obs.columns if col_pattern.lower() in c.lower()]
        if cdr3_cols:
            cdr3_col = cdr3_cols[0]
            has_cdr3_data = True
            break

    if not has_cdr3_data:
        # Try to compute from TCR one-hot data (length from non-padding)
        for key in ['X_tcr_tra_onehot', 'X_tcr_trb_onehot']:
            if key in adata.obsm:
                onehot = np.asarray(adata.obsm[key])
                if hasattr(onehot, 'toarray'): onehot = onehot.toarray()
                n_ch = 20
                if onehot.shape[1] % n_ch == 0:
                    seq_len = onehot.shape[1] // n_ch
                    reshaped = onehot.reshape(-1, seq_len, n_ch)
                    lengths = (reshaped.sum(axis=-1) > 0).sum(axis=-1)
                    chain = 'TRA' if 'tra' in key else 'TRB'
                    adata.obs[f'{chain}_cdr3_length'] = lengths
                    has_cdr3_data = True

    if has_cdr3_data:
        length_cols = [c for c in adata.obs.columns if 'cdr3_length' in c.lower() or 'cdr3_len' in c.lower()]
        if len(length_cols) == 0:
            length_cols = [c for c in adata.obs.columns if 'cdr3' in c.lower() and 'length' in c.lower()]

        resp_mask = adata.obs['response'].isin(['Responder', 'Non-Responder']).values
        colors_resp = {'Responder': '#2196F3', 'Non-Responder': '#F44336'}

        if length_cols:
            col = length_cols[0]
            for resp, color in colors_resp.items():
                vals = adata.obs.loc[resp_mask & (adata.obs['response'] == resp), col].dropna().values
                if len(vals) > 0:
                    vals = vals.astype(float)
                    ax.hist(vals, bins=30, alpha=0.6, color=color, label=f'{resp} (n={len(vals)})',
                           edgecolor='white', density=True)
            ax.set_xlabel(f'{col.replace("_", " ").title()}', fontsize=12)
            ax.set_ylabel('Density', fontsize=12)
            ax.set_title('A) CDR3 Length Distribution by Response', fontsize=13, fontweight='bold')
            ax.legend(fontsize=10)
            ax.grid(alpha=0.3)

            # Add KS test
            from scipy import stats
            r_vals = adata.obs.loc[resp_mask & (adata.obs['response'] == 'Responder'), col].dropna().astype(float)
            nr_vals = adata.obs.loc[resp_mask & (adata.obs['response'] == 'Non-Responder'), col].dropna().astype(float)
            if len(r_vals) > 5 and len(nr_vals) > 5:
                ks_stat, ks_p = stats.ks_2samp(r_vals, nr_vals)
                ax.text(0.95, 0.95, f'KS stat={ks_stat:.3f}\np={ks_p:.2e}',
                       transform=ax.transAxes, ha='right', va='top', fontsize=10,
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.8))
        else:
            ax.text(0.5, 0.5, 'No CDR3 length columns found', ha='center', va='center', transform=ax.transAxes)
    else:
        # Fallback: show TCR feature availability
        tcr_keys = [k for k in adata.obsm.keys() if 'tcr' in k.lower()]
        tcr_obs_cols = [c for c in adata.obs.columns if 'tcr' in c.lower() or 'CDR3' in c]
        ax.text(0.5, 0.6, f'TCR obsm matrices: {len(tcr_keys)}', ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.text(0.5, 0.4, f'TCR obs columns: {len(tcr_obs_cols)}', ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.set_title('A) TCR Data Summary', fontsize=13, fontweight='bold')

    # Panel B: Feature Set Impact on Model Performance
    ax2 = axes[1]
    results = load_all_model_results() if 'load_all_model_results' in dir() else None

    if results is not None and 'feature_set' in results.columns:
        # Compare feature sets across all models (cell-level AUC)
        if 'evaluation_level' in results.columns:
            cell_res = results[results['evaluation_level'].astype(str).str.contains('cell', case=False, na=True)]
            if len(cell_res) > 0:
                results = cell_res

        fs_auc = results.groupby('feature_set')['auc'].agg(['mean', 'std', 'count']).reset_index()
        fs_auc = fs_auc.sort_values('mean', ascending=True)

        fs_colors = {'basic': '#90CAF9', 'gene_enhanced': '#42A5F5',
                    'tcr_enhanced': '#EF5350', 'comprehensive': '#7E57C2',
                    'sequence_structure': '#FF7043'}
        bar_colors = [fs_colors.get(fs, '#999') for fs in fs_auc['feature_set']]

        bars = ax2.barh(range(len(fs_auc)), fs_auc['mean'].values,
                       xerr=fs_auc['std'].values, color=bar_colors,
                       edgecolor='#333', linewidth=0.5, height=0.6, capsize=3)
        ax2.set_yticks(range(len(fs_auc)))
        ax2.set_yticklabels([fs.replace('_', '\n') for fs in fs_auc['feature_set']], fontsize=10)
        ax2.set_xlabel('Mean AUC', fontsize=12)
        ax2.set_title('B) Feature Set Impact on Performance', fontsize=13, fontweight='bold')
        ax2.grid(axis='x', alpha=0.3)

        for i, (val, n) in enumerate(zip(fs_auc['mean'].values, fs_auc['count'].values)):
            ax2.text(val + 0.005, i, f'{val:.3f} (n={n})', va='center', fontsize=9)

        # Highlight TCR contribution
        basic_auc = fs_auc[fs_auc['feature_set'].str.contains('basic', case=False)]['mean'].values
        tcr_auc = fs_auc[fs_auc['feature_set'].str.contains('tcr', case=False)]['mean'].values
        if len(basic_auc) > 0 and len(tcr_auc) > 0:
            delta = tcr_auc[0] - basic_auc[0]
            ax2.text(0.95, 0.05, f'TCR ΔAUCvs basic: {delta:+.3f}',
                    transform=ax2.transAxes, ha='right', va='bottom', fontsize=10,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.9))
    else:
        ax2.text(0.5, 0.5, 'No results with feature_set info.\nRun model training cells first.',
                ha='center', va='center', transform=ax2.transAxes, fontsize=11, style='italic')
        ax2.set_title('B) Feature Set Impact', fontsize=13, fontweight='bold')

    fig.suptitle('Figure 11: TCR CDR3 Sequence Analysis & Feature Set Impact',
                fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('Figure11_TCR_Analysis.png', dpi=600, bbox_inches='tight', facecolor='white')
    plt.show()
    print("Figure 11 saved.")

  Loaded 34 rows from Processed_Data/ml_integrated_results.csv
  Loaded 8 rows from Processed_Data/lopo_results.csv
  Loaded 4 rows from Processed_Data/homology_aware_benchmark_results.csv
Figure 11 saved.


In [ ]:
# Figure 12: Cross-Model Agreement & Patient-Level Validation
# Panel A: Heatmap of patient-level AUC per model (from actual results)
# Panel B: Model-model performance correlation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np, pandas as pd
import warnings
warnings.filterwarnings('ignore')

results = load_all_model_results() if 'load_all_model_results' in dir() else None

if results is None or 'model' not in results.columns:
    print("No model results available. Run upstream training cells first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Panel A: Patient-Level Heatmap
    ax = axes[0]
    patient_res = None
    if 'evaluation_level' in results.columns:
        patient_res = results[results['evaluation_level'].astype(str).str.contains('patient', case=False)]

    if patient_res is not None and len(patient_res) > 0:
        # Pivot: rows = models, columns = feature_sets, values = AUC
        if 'feature_set' in patient_res.columns:
            pivot = patient_res.pivot_table(index='model', columns='feature_set', values='auc', aggfunc='mean')
        else:
            # If no feature_set, create a simple model x metric view
            metrics = ['auc', 'accuracy', 'f1']
            avail = [m for m in metrics if m in patient_res.columns]
            pivot = patient_res.groupby('model')[avail].mean()

        if pivot.shape[0] > 0 and pivot.shape[1] > 0:
            # Sort by mean AUC
            pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

            im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0.3, vmax=1.0)
            ax.set_xticks(range(pivot.shape[1]))
            ax.set_xticklabels([c.replace('_', '\n') for c in pivot.columns], fontsize=9, rotation=45, ha='right')
            ax.set_yticks(range(pivot.shape[0]))
            labels = []
            for m in pivot.index:
                cat = get_model_type(m) if 'get_model_type' in dir() else 'ML'
                marker = '★' if cat == 'DL' else ''
                labels.append(f'{marker}{m}')
            ax.set_yticklabels(labels, fontsize=10)

            # Annotate cells
            for i in range(pivot.shape[0]):
                for j in range(pivot.shape[1]):
                    val = pivot.values[i, j]
                    if not np.isnan(val):
                        color = 'white' if val < 0.5 or val > 0.85 else 'black'
                        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)

            plt.colorbar(im, ax=ax, label='AUC', shrink=0.8)
            ax.set_title('A) Patient-Level AUC by Model & Feature Set', fontsize=13, fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'Insufficient patient-level data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title('A) Patient-Level Results', fontsize=13, fontweight='bold')
    else:
        # Fallback: cell-level AUC heatmap
        if 'feature_set' in results.columns:
            pivot = results.pivot_table(index='model', columns='feature_set', values='auc', aggfunc='mean')
        else:
            metrics = ['auc', 'accuracy', 'f1', 'precision', 'recall']
            avail = [m for m in metrics if m in results.columns]
            pivot = results.groupby('model')[avail].mean()

        if pivot.shape[0] > 0:
            pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]
            im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0.3, vmax=1.0)
            ax.set_xticks(range(pivot.shape[1]))
            ax.set_xticklabels([c.replace('_', '\n') for c in pivot.columns], fontsize=9, rotation=45, ha='right')
            ax.set_yticks(range(pivot.shape[0]))
            labels = []
            for m in pivot.index:
                cat = get_model_type(m) if 'get_model_type' in dir() else 'ML'
                marker = '★ ' if cat == 'DL' else ''
                labels.append(f'{marker}{m}')
            ax.set_yticklabels(labels, fontsize=10)
            for i in range(pivot.shape[0]):
                for j in range(pivot.shape[1]):
                    val = pivot.values[i, j]
                    if not np.isnan(val):
                        color = 'white' if val < 0.5 or val > 0.85 else 'black'
                        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)
            plt.colorbar(im, ax=ax, label='AUC', shrink=0.8)
            ax.set_title('A) Cell-Level AUC by Model & Feature Set', fontsize=13, fontweight='bold')

    # Panel B: Model-Model Performance Correlation
    ax2 = axes[1]
    if 'feature_set' in results.columns and len(results['feature_set'].unique()) > 1:
        # Build model x feature_set AUC matrix
        auc_pivot = results.pivot_table(index='feature_set', columns='model', values='auc', aggfunc='mean')
        if auc_pivot.shape[0] > 1 and auc_pivot.shape[1] > 1:
            corr = auc_pivot.corr()
            im2 = ax2.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
            ax2.set_xticks(range(len(corr.columns)))
            ax2.set_xticklabels(corr.columns, fontsize=9, rotation=45, ha='right')
            ax2.set_yticks(range(len(corr.index)))
            ax2.set_yticklabels(corr.index, fontsize=9)
            for i in range(corr.shape[0]):
                for j in range(corr.shape[1]):
                    val = corr.values[i, j]
                    if not np.isnan(val):
                        ax2.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                                color='white' if abs(val) > 0.6 else 'black')
            plt.colorbar(im2, ax=ax2, label='Correlation', shrink=0.8)
            ax2.set_title('B) Model-Model AUC Correlation', fontsize=13, fontweight='bold')
        else:
            ax2.text(0.5, 0.5, 'Not enough data\nfor correlation', ha='center', va='center',
                    transform=ax2.transAxes, fontsize=12)
            ax2.set_title('B) Model Correlation', fontsize=13, fontweight='bold')
    else:
        # Fallback: bar chart of all model AUCs sorted
        model_auc = results.groupby('model')['auc'].mean().sort_values(ascending=True)
        colors = [get_model_color(m) if 'get_model_color' in dir() else '#666' for m in model_auc.index]
        ax2.barh(range(len(model_auc)), model_auc.values, color=colors,
                edgecolor='#333', linewidth=0.5, height=0.6)
        ax2.set_yticks(range(len(model_auc)))
        ax2.set_yticklabels(model_auc.index, fontsize=10)
        ax2.set_xlabel('Mean AUC', fontsize=12)
        ax2.set_title('B) All Models Ranked by AUC', fontsize=13, fontweight='bold')
        ax2.grid(axis='x', alpha=0.3)
        for i, v in enumerate(model_auc.values):
            ax2.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

    fig.suptitle('Figure 12: Cross-Model Agreement & Validation',
                fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('Figure12_CrossModel_Agreement.png', dpi=600, bbox_inches='tight', facecolor='white')
    plt.show()
    print("Figure 12 saved.")

  Loaded 34 rows from Processed_Data/ml_integrated_results.csv
  Loaded 8 rows from Processed_Data/lopo_results.csv
  Loaded 4 rows from Processed_Data/homology_aware_benchmark_results.csv
Figure 12 saved.


In [ ]:
if 'adata' not in globals() or adata is None:
    print("ERROR: adata not available. Run upstream cells first.")
else:
    analysis_result = run_end_analysis(adata, evaluation_level='cell')
    if not analysis_result['metric_summary'].empty:
        display(analysis_result['metric_summary'])

    results = load_result_tables()
    if not results.empty and 'model' in results.columns and 'auc' in results.columns:
        best_per_model = results.sort_values('auc', ascending=False).drop_duplicates('model', keep='first')
        perf_path = get_processed_data_dir() / 'performance_summary.csv'
        best_per_model.to_csv(perf_path, index=False)
        print(f"Saved pooled performance summary to: {perf_path}")


In [ ]:
# Experiment Conclusion: Key Findings Summary
import pandas as pd
import numpy as np

results = load_all_model_results() if 'load_all_model_results' in dir() else None

print("=" * 70)
print("  HR+ BREAST CANCER IMMUNOTHERAPY RESPONSE PREDICTION — KEY FINDINGS")
print("=" * 70)

if results is not None and 'model' in results.columns:
    dl_models_set = set(DL_MODELS) if 'DL_MODELS' in dir() else {'MLP','1D-CNN','CNN','RNN','BiLSTM','Transformer'}

    # Overall best
    best = results.sort_values('auc', ascending=False).iloc[0]
    print(f"\n  Best Overall Model: {best['model']}")
    print(f"     AUC = {best['auc']:.4f}", end='')
    if 'feature_set' in best.index and pd.notna(best.get('feature_set')):
        print(f"  |  Feature Set: {best['feature_set']}", end='')
    if 'evaluation_level' in best.index and pd.notna(best.get('evaluation_level')):
        print(f"  |  Level: {best['evaluation_level']}", end='')
    print()

    # Best DL
    dl_res = results[results['model'].isin(dl_models_set)]
    if len(dl_res) > 0:
        best_dl = dl_res.sort_values('auc', ascending=False).iloc[0]
        print(f"\n  Best Deep Learning: {best_dl['model']}")
        print(f"     AUC = {best_dl['auc']:.4f}", end='')
        if 'feature_set' in best_dl.index: print(f"  |  Feature Set: {best_dl.get('feature_set', '?')}", end='')
        print()

    # Best Traditional ML
    ml_res = results[~results['model'].isin(dl_models_set)]
    if len(ml_res) > 0:
        best_ml = ml_res.sort_values('auc', ascending=False).iloc[0]
        print(f"\n  Best Traditional ML: {best_ml['model']}")
        print(f"     AUC = {best_ml['auc']:.4f}", end='')
        if 'feature_set' in best_ml.index: print(f"  |  Feature Set: {best_ml.get('feature_set', '?')}", end='')
        print()

    # Feature set impact
    if 'feature_set' in results.columns:
        fs_perf = results.groupby('feature_set')['auc'].mean().sort_values(ascending=False)
        print(f"\n  Feature Set Rankings (mean AUC):")
        for fs, auc in fs_perf.items():
            print(f"     {fs:25s} → {auc:.4f}")

    # Model count
    print(f"\n  Models evaluated: {len(results['model'].unique())} "
          f"({len(dl_res['model'].unique()) if len(dl_res) > 0 else 0} DL + "
          f"{len(ml_res['model'].unique()) if len(ml_res) > 0 else 0} ML)")
    print(f"  Total experiments: {len(results)}")
else:
    print("\n  No results available. Run upstream training cells first.")

print(f"\n{'=' * 70}")

  Loaded 34 rows from Processed_Data/ml_integrated_results.csv
  Loaded 8 rows from Processed_Data/lopo_results.csv
  Loaded 4 rows from Processed_Data/homology_aware_benchmark_results.csv
  HR+ BREAST CANCER IMMUNOTHERAPY RESPONSE PREDICTION — KEY FINDINGS

  🏆 Best Overall Model: Decision Tree
     AUC = 1.0000  |  Feature Set: comprehensive  |  Level: patient

  🧠 Best Deep Learning: MLP
     AUC = 1.0000  |  Feature Set: sequence_structure

  📊 Best Traditional ML: Random Forest
     AUC = 1.0000  |  Feature Set: comprehensive

  📋 Feature Set Rankings (mean AUC):
     sequence_structure        → 0.9851
     comprehensive             → 0.9735
     tcr_enhanced              → 0.0000

  Models evaluated: 13 (5 DL + 8 ML)
  Total experiments: 42



In [ ]:
# Summary: List all generated figure files
import os

figure_files = sorted([
    f for f in os.listdir('.')
    if f.endswith('.png') and (f.startswith('Figure') or f.startswith('Table'))
])

print("=" * 65)
print("  GENERATED FIGURES — HR+ Breast Cancer Response Prediction")
print("=" * 65)
print()
expected = [
    ("Figure1_Pipeline_Overview.png",      "Pipeline & Experimental Design"),
    ("Figure2_UMAP_Visualization.png",     "UMAP: Clusters, Response, Patients"),
    ("Figure3_QC_Metrics.png",             "Quality Control Metrics"),
    ("Figure4_PCA_Variance.png",           "PCA Variance Explained"),
    ("Figure5_Sample_Composition.png",     "Patient & Timepoint Composition"),
    ("Figure6_All_Models_AUC.png",         "All Models AUC Comparison (DL ★)"),
    ("Figure7_DL_Architecture_DeepDive.png","DL Architecture Deep Dive"),
    ("Figure8_Radar_Chart.png",            "Multi-Metric Radar Comparison"),
    ("Figure9_SHAP_DL_Feature_Importance.png","SHAP Feature Importance (Best DL)"),
    ("Figure10_MultiModel_Feature_Importance.png","Multi-Model Gene Importance"),
    ("Figure11_TCR_Analysis.png",          "TCR CDR3 & Feature Set Impact"),
    ("Figure12_CrossModel_Agreement.png",  "Cross-Model Agreement & Validation"),
]

for i, (fname, desc) in enumerate(expected, 1):
    exists = os.path.exists(fname)
    size_kb = os.path.getsize(fname) / 1024 if exists else 0
    status = f"{size_kb:.0f}KB" if exists else "not yet generated"
    print(f"  {i:2d}. {desc:45s} {status}")
    if exists:
        print(f"      → {fname}")

print(f"\n{'=' * 65}")
found = sum(1 for f, _ in expected if os.path.exists(f))
print(f"  Generated: {found}/{len(expected)} figures")
print(f"  All figures saved at 300 DPI for presentation quality.")
print(f"  Run all cells sequentially to regenerate from actual data.")
print(f"{'=' * 65}")

  GENERATED FIGURES — HR+ Breast Cancer Response Prediction

   1. Pipeline & Experimental Design                ✅ 1279KB
      → Figure1_Pipeline_Overview.png
   2. UMAP: Clusters, Response, Patients            ✅ 5180KB
      → Figure2_UMAP_Visualization.png
   3. Quality Control Metrics                       ✅ 548KB
      → Figure3_QC_Metrics.png
   4. PCA Variance Explained                        ✅ 493KB
      → Figure4_PCA_Variance.png
   5. Patient & Timepoint Composition               ✅ 425KB
      → Figure5_Sample_Composition.png
   6. All Models AUC Comparison (DL ★)              ⬜ not yet generated
   7. DL Architecture Deep Dive                     ⬜ not yet generated
   8. Multi-Metric Radar Comparison                 ✅ 1675KB
      → Figure8_Radar_Chart.png
   9. SHAP Feature Importance (Best DL)             ✅ 556KB
      → Figure9_SHAP_DL_Feature_Importance.png
  10. Multi-Model Gene Importance                   ✅ 649KB
      → Figure10_MultiModel_Feature_Importance.png
  